# Explorador de motivos
### Una lectura en red de obras narrativas · prototipo · Taller de Humanidades Digitales 1

**Qué es.** Una herramienta que toma el PDF de una obra que **ya leíste** y te devuelve una *lectura en red*: una página interactiva donde ves qué motivos (imágenes, objetos, espacios, ideas, personajes) se repiten, con cuáles se unen, cómo cambian esas uniones a lo largo de la obra y **en qué frases** ocurre, con su página.

**Para quién.** Para quien prepara una tesis o un trabajo sobre una obra narrativa, y para cualquier lector curioso que quiera mirarla de nuevo. No reemplaza la lectura: los diagramas condensan lo que leíste y te devuelven al texto para confirmar o descartar cada hallazgo.

**Cómo funciona.** Cinco pasos. Ejecuta cada celda en orden; el código está oculto y solo verás formularios.

| Paso | Qué hace | Qué haces tú |
|---|---|---|
| 1 · Subir y limpiar | Lee el PDF, quita lo que no es la obra y la divide en capítulos, frases y páginas | Revisar el informe de limpieza |
| 2 · Punto de partida | Sugiere motivos (o usa los de un perfil, o los pide a la IA) | Elegir cómo empezar |
| 3 · Personajes | Propone el reparto, arma la red de personajes y el árbol de familia con sus citas | Revisar el reparto y la familia; corregirlos en 3b |
| 4 · La lectura en red | Genera la página interactiva y la descarga | **Explorarla**: es el producto |
| 5 · IA (opcional) | Con tu clave de OpenAI, propone lecturas de una conexión ancladas en frases citadas | Juzgar si se sostienen |

**La lectura en red** tiene seis pestañas: **Hallazgos** (puntos de entrada), **Red** (la red de motivos, que puedes recorrer tramo a tramo), **Personajes** (red, árbol de familia y una ficha por personaje), **Motivos** (una ficha por motivo), **Taller** (cambias los motivos y todo se recalcula) y **Cuaderno** (tus notas y citas con página, descargables).

> Probado con *El principito* (ILCE), *La metamorfosis* (UNAM), *El mundo es ancho y ajeno* (Losada), *Hamlet* (Elejandría) y *Cien años de soledad* (eBooket). Funciona con narrativa y, en segundo plano, con teatro. Solo PDF con **texto seleccionable**.

## 0 · Prepara la herramienta
Ejecuta estas dos celdas una sola vez (▶). Instalan lo necesario y cargan el motor y el diseño de la lectura. No necesitas abrirlas; si tienes curiosidad, haz doble clic en el título para ver el código.

In [ ]:
#@title ⚙️ Motor de la herramienta (ejecútala una vez; no necesitas abrirla)
import subprocess, sys, os, re, json, html, unicodedata, collections, math, glob, random, zipfile

def instalar(*paquetes):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *paquetes], check=False)
try:
    import pdfplumber
except ImportError:
    instalar("pdfplumber"); import pdfplumber
try:
    from wordfreq import zipf_frequency
except ImportError:
    if os.environ.get("SIN_WORDFREQ") != "1": instalar("wordfreq")
    try: from wordfreq import zipf_frequency
    except Exception: zipf_frequency = None
import numpy as np, pandas as pd
try:
    from IPython.display import display, HTML
except ImportError:
    display, HTML = print, (lambda s: f"[HTML de {len(s):,} caracteres]")
try:
    from google.colab import files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# ======================= PERFILES DE LAS OBRAS PROBADAS =======================
# Un perfil guarda lo que sabemos de una edición: cómo agruparla en tramos y un diccionario inicial de motivos
# (formas separadas por comas; el asterisco = cualquier terminación). Es un punto de partida editable, no la respuesta.
PERFILES = {
  "El principito": {"claves": ["el principito"], "autor": "Antoine de Saint-Exupéry",
    "tramos": {"Los mayores y el piloto": [1], "El encuentro": [2, 3, 4], "Baobabs y la flor": [5, 6, 7, 8, 9],
               "Los seis planetas": [10, 11, 12, 13, 14, 15], "La Tierra: el vacío": [16, 17, 18, 19, 20],
               "El zorro": [21], "Prisa y sed": [22, 23], "El pozo": [24, 25], "La despedida": [26, 27]},
    "erratas": {"qué contesto estoy": "qué contento estoy", "con gustó a este": "con gusto a este", "dijeran las rosas": "dijeron las rosas"},
    "personajes": """el principito: principito
la flor: flor, rosa
el zorro: zorro
la serpiente: serpiente
el cordero: cordero
el rey: rey
el vanidoso: vanidoso
el bebedor: bebedor
el hombre de negocios: hombre de negocios
el farolero: farolero
el geógrafo: geografo
el comerciante: comerciante
las personas mayores: personas mayores""",
    "motivos": """flor: flor, flores
rosa: rosa, rosas, rosal*
estrellas: estrella, estrellas
mirar y ver: ojo, ojos, mir*, ver, ves, veo, vista
corazón: corazon, corazones
agua y pozo: agua, sed, pozo*, fuente*, beber, bebe
desierto: desierto*, arena, duna*
personas mayores: mayores, mayor
números: cifra*, numero*, millon*, cuenta, cuentas, sumar
serio: serio, seria, serios, serias
domesticar: domestic*, vinculo*, rito*
invisible y esencial: invisible*, esencial*
risa: reir, rie, rio, risa*, cascabel*
cordero: cordero*, bozal
serpiente: serpiente*, veneno, morder, mordisco
baobabs: baobab*
zorro: zorro, zorros
planeta: planeta, planetas
rey: rey, reyes"""},
  "La metamorfosis": {"claves": ["la metamorfosis", "gregorio samsa"], "autor": "Franz Kafka",
    "personajes": """Gregorio: gregorio samsa, gregorio
Grete (la hermana): greta, grete, hermana
el padre: padre, señor samsa
la madre: madre, señora samsa
el apoderado: apoderado
el jefe: jefe
la asistenta: asistenta
los huéspedes: huesped*""",
    "familia": """Gregorio: hijo del padre y de la madre; hermano de Grete (la hermana)
Grete (la hermana): hija del padre y de la madre
el padre: esposo de la madre""",
    "motivos": """insecto: insecto*, bicho*, sabandija*, monstruo*, animal*
cuerpo: cuerpo*, patas, pata, patitas, caparazon, espalda, vientre, antenas, mandibula*
habitación: habitacion*, cuarto, pared*, techo, muebles, mueble, cama, sofa, canape
puerta: puerta*, llave*, cerradura*
ventana: ventana*
padre: padre, padres
madre: madre
hermana: hermana, grete
trabajo y dinero: trabajo*, oficina, apoderado, jefe, viajante, negocio*, deuda*, dinero, empleo*
comida: comida*, comer, leche, pan, manzana*, alimento*
música: violin, musica, tocar, tocaba
miedo y asco: miedo*, asco, repugnan*, horror*, espanto*
silencio y ruido: silencio*, ruido*, grito*, chillido*
huéspedes: huesped*
herida y dolor: sangr*, herida*, dolor*
Gregorio: gregorio"""},
  "El mundo es ancho y ajeno": {"claves": ["el mundo es ancho y ajeno"], "autor": "Ciro Alegría",
    "tramos": {"Rumi: la comunidad": [1, 2, 3, 4, 5, 6], "Litigio y despojo": [7, 8], "Yanañahui y la cárcel": [9, 10, 11, 12],
               "Los dispersos I": [13, 14, 15], "Muerte de Rosendo": [16], "Los dispersos II": [17, 18, 19, 20],
               "Benito Castro y la resistencia": [21, 22, 23, 24]},
    "stop_extra": ["pa", "aura", "tamien", "ques", "pue", "onde", "ande", "nomas", "dao", "taba", "asina"],
    "motivos": """tierra: tierra, tierras
comunidad: comunidad, comunidades, comuner*
indio: indio, indios, india, indias, indiada, indigena*
hacendado y patrón: hacend*, patron, patrones, gamonal*, hacienda*
ley y papeles: ley, leyes, juez, jueces, juicio*, papeles, titulos, abogado*, tinterillo
sangre y muerte: sangre, muert*, morir, muere, murio, cadaver*
trigo y maíz: trigo*, maiz*, cosecha*, siembra*, sembr*
cerros y Andes: cerro, cerros, andes, puna, punas, cumbre*
coca: coca
camino: camino*, sendero*
lucha: lucha*, rebel*, sublev*, fusil*, balazo*
cárcel: carcel*, preso, presos
selva y caucho: selva*, caucho*, cauchero*
éxodo: exodo, destierro, ajeno*, emigr*
Rosendo: rosendo
Benito: benito
Amenábar: amenabar
Rumi: rumi"""},
  "Hamlet": {"claves": ["hamlet"], "autor": "William Shakespeare (trad. Moratín)",
    "personajes": """Hamlet: hamlet, principe
Claudio (el rey): claudio, rey
Gertrudis (la reina): gertrudis, reina
la Sombra (el padre): la sombra, sombra, fantasma
Polonio: polonio
Ofelia: ofelia
Laertes: laertes
Horacio: horacio
Marcelo: marcelo
Bernardo: bernardo
Francisco: francisco
Ricardo: ricardo
Guillermo: guillermo
Enrique: enrique
Reynaldo: reynaldo
Fortimbrás: fortimbras
Voltiman: voltiman
los cómicos: comico, comicos
los sepultureros: sepulturero, sepultureros
Yorick: yorick""",
    "familia": """Hamlet: hijo de la Sombra (el padre) y de Gertrudis (la reina); sobrino de Claudio (el rey)
la Sombra (el padre): esposo de Gertrudis (la reina); hermano de Claudio (el rey)
Claudio (el rey): esposo de Gertrudis (la reina)
Ofelia: hija de Polonio; hermana de Laertes
Laertes: hijo de Polonio""",
    "motivos": """padre: padre, padres
madre: madre
espectro y sombra: espectro, sombra*, fantasma, aparecido, aparicion
muerte: muerte*, muert*, morir, muero, muere, murio
venganza: veng*
locura: loco, loca, locura*, demencia, delirio
veneno: veneno*, emponzoñ*
sepultura y calavera: sepultur*, calavera*, craneo*, tumba*, sepulcro*
cielo: cielo, cielos
infierno: infierno*, infernal*
amor: amor, amores, amar, amo, ama
noche: noche, noches
teatro y apariencia: comico*, comedia, representa*, fingi*, apariencia*, mascara
alma y conciencia: alma, almas, conciencia
corona y trono: corona, trono, reino, cetro
Hamlet: hamlet
Ofelia: ofelia"""},
}

PERFILES["Cien años de soledad"] = {"claves": ["cien años de soledad", "cien anos de soledad"], "autor": "Gabriel García Márquez",
    # Los nombres se repiten entre generaciones: los rangos [a-b] (capítulos) dicen a quién se refiere un nombre suelto en cada parte.
    "personajes": """José Arcadio Buendía: jose arcadio buendia
Úrsula Iguarán: ursula iguaran, ursula
José Arcadio (hijo): jose arcadio [1-10]
Coronel Aureliano Buendía: coronel aureliano buendia, coronel aureliano, aureliano buendia, coronel, aureliano [1-14]
Amaranta: amaranta buendia, amaranta
Rebeca: rebeca
Remedios Moscote: remedios moscote, remedios [1-6]
Apolinar Moscote: don apolinar moscote, apolinar moscote, apolinar
Pilar Ternera: pilar ternera, pilar
Arcadio: arcadio
Aureliano José: aureliano jose
Santa Sofía de la Piedad: santa sofia de la piedad, santa sofia
Remedios, la bella: remedios la bella, remedios [7-20]
Aureliano Segundo: aureliano segundo
José Arcadio Segundo: jose arcadio segundo
Fernanda del Carpio: fernanda del carpio, fernanda
Petra Cotes: petra cotes, petra
Meme (Renata Remedios): renata remedios, meme
José Arcadio (seminarista): jose arcadio [11-20]
Amaranta Úrsula: amaranta ursula
Mauricio Babilonia: mauricio babilonia, mauricio
Aureliano Babilonia: aureliano babilonia, aureliano [15-20]
Gastón: gaston
Melquíades: melquiades
Prudencio Aguilar: prudencio aguilar, prudencio
Pietro Crespi: pietro crespi, pietro
Gerineldo Márquez: coronel gerineldo marquez, gerineldo marquez, gerineldo
los diecisiete Aurelianos: aureliano triste, aureliano centeno, aureliano amador, aureliano serrador, aureliano arcaya, diecisiete aurelianos
Padre Nicanor: padre nicanor reyna, padre nicanor, nicanor
General Moncada: general jose raquel moncada, general moncada, coronel moncada, moncada
Nigromanta: nigromanta
Gabriel: gabriel""",
    "familia": """José Arcadio (hijo): hijo de José Arcadio Buendía y Úrsula Iguarán; esposo de Rebeca
Coronel Aureliano Buendía: hijo de José Arcadio Buendía y Úrsula Iguarán; esposo de Remedios Moscote
Amaranta: hija de José Arcadio Buendía y Úrsula Iguarán
Rebeca: hija adoptiva de José Arcadio Buendía y Úrsula Iguarán
Remedios Moscote: hija de Apolinar Moscote
José Arcadio Buendía: esposo de Úrsula Iguarán
Arcadio: hijo de José Arcadio (hijo) y Pilar Ternera; pareja de Santa Sofía de la Piedad
Aureliano José: hijo de Coronel Aureliano Buendía y Pilar Ternera
los diecisiete Aurelianos: hijos de Coronel Aureliano Buendía
Remedios, la bella: hija de Arcadio y Santa Sofía de la Piedad
Aureliano Segundo: hijo de Arcadio y Santa Sofía de la Piedad; esposo de Fernanda del Carpio; amante de Petra Cotes
José Arcadio Segundo: hijo de Arcadio y Santa Sofía de la Piedad
José Arcadio (seminarista): hijo de Aureliano Segundo y Fernanda del Carpio
Meme (Renata Remedios): hija de Aureliano Segundo y Fernanda del Carpio; amante de Mauricio Babilonia
Amaranta Úrsula: hija de Aureliano Segundo y Fernanda del Carpio; esposa de Gastón; amante de Aureliano Babilonia
Aureliano Babilonia: hijo de Mauricio Babilonia y Meme (Renata Remedios); sobrino de Amaranta Úrsula
Pietro Crespi: novio de Rebeca; pretendiente de Amaranta
Gerineldo Márquez: pretendiente de Amaranta""",
    "motivos": """soledad: soledad, solitari*
memoria y olvido: memoria, recuerd*, olvid*
insomnio y peste: insomnio, peste
lluvia y diluvio: lluvia*, llov*, diluvio
oro y pescaditos: oro, pescadito*
guerra: guerra*
hielo: hielo
gitanos e inventos: gitano*, invento*, iman*, lupa
pergaminos: pergamino*, manuscrito*
casa: casa
cola de cerdo: cola, cerdo*
mariposas amarillas: mariposa*, amarill*
muerte: muerte, muert*, morir, murio
tiempo: tiempo, siglo*
laboratorio: laboratorio, alquimia, platero*
compañía bananera: banano*, bananera*, compañia, gringo*
tren: tren
fusilamiento: peloton, fusil*
olor: olor*
Macondo: macondo
Melquíades: melquiades"""}

# ======================= UTILIDADES =======================
def sin_tildes(t):
    t = t.replace("ñ", "§").replace("Ñ", "¶")
    t = "".join(c for c in unicodedata.normalize("NFKD", t) if not unicodedata.combining(c))
    return t.replace("§", "ñ").replace("¶", "Ñ")
def palabras(t): return len(re.findall(r"\w+", t))
RX_ROMANO = re.compile(r"M{0,3}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})")
def romano_a_int(s):
    if not s or not RX_ROMANO.fullmatch(s): return None
    v = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}; tot = 0
    for a, b in zip(s, s[1:] + " "): tot += -v[a] if b != " " and v[b] > v[a] else v[a]
    return tot
def a_romano(n):
    out = ""
    for v, r in [(100, "C"), (90, "XC"), (50, "L"), (40, "XL"), (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")]:
        while n >= v: out += r; n -= v
    return out
ORDINALES = {w: i for i, ws in enumerate([[], ["primero", "primera", "uno", "una"], ["segundo", "segunda", "dos"],
    ["tercero", "tercera", "tres"], ["cuarto", "cuarta", "cuatro"], ["quinto", "quinta", "cinco"], ["sexto", "sexta", "seis"],
    ["septimo", "setimo", "septima", "siete"], ["octavo", "octava", "ocho"], ["noveno", "novena", "nueve"],
    ["decimo", "decima", "diez"], ["undecimo", "once"], ["duodecimo", "doce"]]) for w in ws}
def a_numero(s):
    s = s.strip(" .:")
    if s.isdigit(): return int(s)
    if s.upper() == s or len(s) <= 4:
        r = romano_a_int(s.upper())
        if r: return r
    return ORDINALES.get(sin_tildes(s.lower()))

# ======================= 1. LECTURA DEL PDF =======================
LIGADURAS = {"ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl", "ﬅ": "st", "ﬆ": "st", "\u00ad": "-", "¬": "-"}
def normalizar_linea(l):
    for a, b in LIGADURAS.items(): l = l.replace(a, b)
    return l.strip()
def leer_pdf(ruta):
    paginas = []
    with pdfplumber.open(ruta) as pdf:
        n = len(pdf.pages)
        for i, p in enumerate(pdf.pages):
            paginas.append([normalizar_linea(l) for l in (p.extract_text() or "").split("\n")])
            if (i + 1) % 50 == 0: print(f"  leídas {i + 1}/{n} páginas…")
    return paginas

# ======================= 2. LIMPIEZA =======================
RUIDO = re.compile(r"https?://|www\.|©|derechos reservados|todos los derechos|libro descargado|esperamos que lo disfrut|"
                   r"gracias por leer|descubre nuestra colecci|material autorizado|edici[oó]n digital|\bisbn\b", re.I)
RUIDO_FINAL = re.compile(r"se termin[oó] de imprimir|impreso en|la edici[oó]n consta|\btiraje\b|primera edici[oó]n|"
                         r"departamento editorial|dise[ñn]o y formaci[oó]n|^rector$|derechos patrimoniales|prohibida la reproducci|"
                         r"^colof[oó]n$", re.I)
NUM_PAG = re.compile(r"[\-–—~·•\s]*(?:p[aá]g(?:ina)?\.?\s*)?(\d{1,4})[\-–—~·•\s]*", re.I)
_PAL = r"([0-9]+|[A-Za-zÁÉÍÓÚáéíóú]+)"
PAT_TIT = {"capitulo": re.compile(rf"^\W{{0,2}}C?AP[IÍ]TULO\s+{_PAL}\b\.?\s*[:.\-–—]?\s*(.*)$", re.I),
           "parte": re.compile(rf"^(?:PARTE|CANTO|LIBRO|JORNADA|TRATADO|CUENTO)\s+{_PAL}\b\.?\s*[:.\-–—]?\s*(.*)$", re.I),
           "romano": re.compile(r"^([IVXLC]{1,7})\.?()$"), "numero": re.compile(r"^(\d{1,3})\.?()$")}
PAT_ACTO, PAT_ESCENA = re.compile(rf"^ACTO\s+{_PAL}\.?\s*$", re.I), re.compile(rf"^ESCENA\s+{_PAL}\.?\s*$", re.I)
RAYA_INICIO = re.compile(r"^[-–—]\s?[¿¡«\"“A-ZÁÉÍÓÚÑ]")
CORTE_RAYA = re.compile(r"(?:(?<=\s)|^)[-–—]+\s*|\s*[-–—]+(?=[\s,.;:!?…»)]|$)")
FIN_ORACION = re.compile(r"[.!?…:»”\"]$")
TAG_TEATRO = re.compile(r"^([A-ZÁÉÍÓÚÑÜ][A-ZÁÉÍÓÚÑÜ0-9º\.\sl]*?)\s*(?:\.\s*[-–—]|:)\s*(.*)$")
MARCA = re.compile("(\\d+)")
def marca(p): return f"{p}"
def es_titulo(l): return any(p.match(l) for p in list(PAT_TIT.values()) + [PAT_ACTO, PAT_ESCENA])

def coincidencias(textos, patron, desde=0, hasta=None):
    out = []
    for i in range(desde, len(textos) if hasta is None else hasta):
        m = patron.match(textos[i])
        if m:
            n = a_numero(m.group(1))
            if n: out.append((i, n, (m.group(2) if m.lastindex and m.lastindex >= 2 else "").strip()))
    return out

def cadena(ms):
    """La secuencia 1, 2, 3… más larga (tolera que falten hasta 2 títulos seguidos)."""
    cadenas, actual, esp, faltan = [], [], 1, []
    for m in ms:
        if m[1] == esp: actual.append(m); esp += 1
        elif actual and esp < m[1] <= esp + 2: faltan += list(range(esp, m[1])); actual.append(m); esp = m[1] + 1
        elif m[1] == 1: cadenas.append((actual, faltan)); actual, esp, faltan = [m], 2, []
    cadenas.append((actual, faltan))
    return max(cadenas, key=lambda c: (len(c[0]), c[0][-1][0] if c[0] else 0))

def titulos_sin_numero(textos, inicios):
    """Líneas cortas en MAYÚSCULAS al comienzo de una página, separadas por bastante texto: posibles títulos de capítulo."""
    cand = []
    for i, t in enumerate(textos):
        letras = re.sub(r"[^A-Za-zÁÉÍÓÚÑáéíóúñ]", "", t)
        if (inicios[i] and 3 <= len(letras) and sum(c.isupper() for c in letras) / len(letras) > 0.85 and len(t.split()) <= 8
                and not FIN_ORACION.search(t.rstrip("!?¡¿")) and not RUIDO.search(t)):
            cand.append(i)
    if len(cand) < 3: return []
    tramos = [b - a for a, b in zip(cand, cand[1:] + [len(textos)])]
    if np.median(tramos) < 25: return []                       # demasiado juntos: no son capítulos
    return [(i, k + 1, textos[i], None) for k, i in enumerate(cand)]

def detectar_estructura(textos, conf, inicios=None):
    forz, cand = conf.get("estructura") or None, {}
    if conf.get("patron_titulo"): cand["propio"] = cadena(coincidencias(textos, re.compile(conf["patron_titulo"])))
    for k, p in PAT_TIT.items(): cand[k] = cadena(coincidencias(textos, p))
    actos, f_actos = cadena(coincidencias(textos, PAT_ACTO))
    escenas, f_esc = [], []
    for j, a in enumerate(actos):
        fin = actos[j + 1][0] if j + 1 < len(actos) else len(textos)
        es, fa = cadena(coincidencias(textos, PAT_ESCENA, a[0], fin))
        escenas += [(i, n, t, a[1]) for i, n, t in es]; f_esc += [f"Acto {a_romano(a[1])}, escena {a_romano(x)}" for x in fa]
    largos = {k: len(c[0]) for k, c in cand.items()}
    if forz == "teatro" or (forz is None and len(actos) >= 2 and len(escenas) >= len(actos)):
        return "teatro", escenas, [f"Acto {a_romano(x)}" for x in f_actos] + f_esc
    if forz == "bloques": return "bloques", [], []
    if forz == "titulos": return "titulos", titulos_sin_numero(textos, inicios), []
    if forz: clave = forz
    elif "propio" in cand and largos["propio"] >= 2: clave = "propio"
    else:
        orden = ["capitulo", "parte", "romano", "numero"]
        clave = max(orden, key=lambda k: (largos[k], -orden.index(k)))
        if largos[clave] < 2:
            tit = titulos_sin_numero(textos, inicios) if inicios else []
            return ("titulos", tit, []) if tit else ("bloques", [], [])
    heads, faltan = cand[clave]
    return clave, [(i, n, t, None) for i, n, t in heads], faltan

def norm_hablante(s):
    s = re.sub(r"\bl(?=\s*\.?\s*º)", "1", s); s = re.sub(r"\.?\s*º", "", s).strip(" .")
    return re.sub(r"\s+", " ", s)
def es_tag(l):
    m = TAG_TEATRO.match(l)
    if not m: return None
    nombre = m.group(1)
    if len(re.sub(r"[^A-ZÁÉÍÓÚÑÜ]", "", nombre)) < 2 or romano_a_int(nombre.strip(" .")): return None
    return norm_hablante(nombre), m.group(2)

def piezas_narrativa(p):
    """Divide un párrafo en tramos (voz, texto): diálogo (raya o «») y narración."""
    if RAYA_INICIO.match(MARCA.sub("", p)):
        partes = CORTE_RAYA.split(p)
        out = [("diálogo" if i % 2 == 1 else "narración", x.strip(" ,")) for i, x in enumerate(partes) if i > 0]
    elif re.match(r"^[“\"]", MARCA.sub("", p).strip()):
        m = re.match(r"^(\s*(?:\ue000\d+\ue001)?\s*[“\"])([^”\"]{1,1500})([”\"]?)(.*)$", p, re.S)
        out = [("diálogo", m.group(2))] + ([("narración", m.group(4))] if m.group(4).strip() else []) if m else [("narración", p)]
    else:
        out, pos = [], 0
        for m in re.finditer(r"«[^»]{1,800}»", p):
            if m.start() > pos: out.append(("narración", p[pos:m.start()]))
            out.append(("diálogo", m.group(0)[1:-1])); pos = m.end()
        out.append(("narración", p[pos:]))
    return [(v, re.sub(r"\s+", " ", t).strip()) for v, t in out if re.search(r"\w", MARCA.sub("", t))]

def _unir(pars, linea, pag):
    p = pars[-1]
    p[1] += " " + (marca(pag) if pag != p[2] else "") + linea; p[2] = pag

def parrafos_narrativa(lineas, ancho):
    pars, prev = [], ""
    for t, pg, *_ in lineas:
        if not pars or RAYA_INICIO.match(t) or (FIN_ORACION.search(prev) and len(prev) < 0.8 * ancho):
            pars.append([pg, t, pg])
        else:
            _unir(pars, t, pg)
        prev = t
    return [(p[0], piezas_narrativa(p[1])) for p in pars]

def parrafos_teatro(lineas):
    pars = []   # [pag_inicio, texto, pag_actual, voz]
    for t, pg, *_ in lineas:
        tag = es_tag(t)
        if tag: pars.append([pg, tag[1], pg, tag[0]])
        elif not pars: pars.append([pg, t, pg, "ACOTACIÓN"])
        else: _unir(pars, t, pg)
    return [(p[0], [(p[3], re.sub(r"\s+", " ", p[1]).strip())]) for p in pars if p[1].strip()]

MENORES = {"y", "e", "o", "u", "de", "del", "la", "las", "el", "los", "en", "a", "al", "con", "por", "para", "un", "una", "unos", "otros", "sus"}
def titular(t):
    if not t.isupper(): return t
    return " ".join(w if (k and w in MENORES) else w[:1].upper() + w[1:] for k, w in enumerate(t.lower().split())).replace("¿a", "¿A")

def limpiar(paginas, conf):
    inf = {"quitadas": collections.Counter(), "ejemplos": collections.defaultdict(list), "avisos": []}
    def quitar(motivo, l):
        inf["quitadas"][motivo] += 1
        if len(inf["ejemplos"][motivo]) < 3 and l[:70] not in inf["ejemplos"][motivo]: inf["ejemplos"][motivo].append(l[:70])
    n_pag = len(paginas)
    clave = lambda l: re.sub(r"\d+", "#", l.lower())
    pag_con = collections.Counter(k for p in paginas for k in {clave(l) for l in p if l})
    repetidas = {k for k, c in pag_con.items() if c >= max(4, 0.2 * n_pag) and len(k) <= 120}
    es_num = lambda l: NUM_PAG.fullmatch(l) is not None
    # Un número suelto es «de página» si sigue la paginación (valor ≈ a·hoja + b, con 1 o 2 páginas del libro por hoja);
    # los demás pueden ser títulos de capítulo («1», «2»…), así que se conservan.
    nums = [(ip, int(NUM_PAG.fullmatch(l).group(1))) for ip, p in enumerate(paginas) for l in p if l and es_num(l)]
    paginacion = set()
    if len(nums) >= 0.5 * n_pag:
        mejor = (0, None)
        for a in (1, 2):
            r = np.array([v - a * ip for ip, v in nums]); ips = np.array([ip for ip, _ in nums])
            # referencia local: mediana de los números de las hojas vecinas (tolera páginas en blanco que desplazan la cuenta)
            ok = {nums[i] for i in range(len(nums))
                  if abs(r[i] - np.median(r[np.abs(ips - ips[i]) <= 8])) <= a + 1}
            if len(ok) > mejor[0]: mejor = (len(ok), ok)
        paginacion = mejor[1] if mejor[0] >= 0.5 * n_pag else set()
    quitar_numeros = bool(paginacion)
    reglas = [re.compile(q) for q in conf.get("quitar", [])]
    lineas, impresas = [], []
    for ip, p in enumerate(paginas):
        visibles, nums = [], []
        for l in p:
            if not l: continue
            if es_num(l):
                v = int(NUM_PAG.fullmatch(l).group(1))
                if (ip, v) in paginacion:
                    quitar("número de página", l); nums.append((len(visibles), v)); continue
            elif clave(l) in repetidas and not es_titulo(l):
                quitar("cabecera o pie repetido", l); continue
            if re.search(r"\.{4,}\s*\d+\s*$|(?:\s\.){4,}", l): quitar("índice", l); continue
            if len(re.findall(r"[A-Za-zÁÉÍÓÚÑáéíóúñ0-9]", l)) <= 1 and not es_titulo(l): quitar("adorno o letra suelta", l); continue
            if any(r.search(l) for r in reglas): quitar("regla de AJUSTES['quitar']", l); continue
            visibles.append([l, ip])
        while len(visibles) > 3 and re.match(r"^(\d{1,3}|\*{1,3})\s?[A-ZÁÉÍÓÚ«“\"(]", visibles[-1][0]) and not es_titulo(visibles[-1][0]):
            quitar("nota al pie", visibles[-1][0]); visibles.pop()
        if len(nums) == 1 and nums[0][0] in (0, len(visibles)): impresas.append((ip, nums[0][1]))
        if visibles: visibles[0].append(True)          # marca: primera línea de la página
        lineas += visibles
    lineas = [l if len(l) == 3 else l + [False] for l in lineas]
    # ¿se puede citar por página impresa?
    offs = collections.Counter(n - ip for ip, n in impresas)
    if impresas and len(impresas) >= max(5, 0.3 * n_pag) and offs.most_common(1)[0][1] >= 0.6 * len(impresas):
        off = offs.most_common(1)[0][0]; inf["cita"] = "impresa"
        for l in lineas: l[1] = l[1] + off
    else:
        inf["cita"] = "pdf"
        for l in lineas: l[1] = l[1] + 1
    # final: frase indicada, o nota editorial / colofón en el último 8 %
    textos = [l[0] for l in lineas]
    if conf.get("final"):
        corte = next((i for i, t in enumerate(textos) if t.startswith(conf["final"])), None)
        if corte is None: inf["avisos"].append(f"No encontré la frase final «{conf['final']}»")
    else:
        desde = int(len(textos) * 0.92)
        corte = next((i for i in range(desde, len(textos)) if RUIDO.search(textos[i]) or RUIDO_FINAL.search(textos[i])), None)
        while corte and len(textos[corte - 1].split()) <= 6 and not FIN_ORACION.search(textos[corte - 1]) and corte > desde:
            corte -= 1
    if corte is not None:
        for t in textos[corte:]: quitar("nota editorial o colofón al final", t)
        inf["final_cortado"] = textos[corte:corte + 3]; lineas = lineas[:corte]
    limpias = []
    for l in lineas:
        if RUIDO.search(l[0]): quitar("ruido editorial (URL, ©, ISBN…)", l[0])
        else: limpias.append(l)
    lineas = []
    for l in limpias:   # palabras partidas al final de línea (no une si el guion cierra un inciso)
        if (lineas and re.search(r"[a-záéíóúñü]-$", lineas[-1][0]) and re.match(r"[a-záéíóúñü]", l[0])
                and lineas[-1][0][:-1].count("-") % 2 == 0):
            inf["quitadas"]["palabra partida unida"] += 1; lineas[-1][0] = lineas[-1][0][:-1] + l[0]
        else:
            lineas.append(l)
    if conf.get("inicio"):
        i0 = next((i for i, l in enumerate(lineas) if l[0].startswith(conf["inicio"])), None)
        if i0 is None: inf["avisos"].append(f"No encontré la frase de inicio «{conf['inicio']}»")
        else:
            while i0 > 0 and es_titulo(lineas[i0 - 1][0]): i0 -= 1
            inf["antes"] = [l[0] for l in lineas[:i0]]; lineas = lineas[i0:]
    textos = [l[0] for l in lineas]
    modo, heads, faltan = detectar_estructura(textos, conf, [l[2] for l in lineas])
    inf.update(modo=modo, faltan=faltan)
    teatro, segs = modo == "teatro", []
    if modo == "bloques":
        n_pal = int(conf.get("palabras_por_bloque") or 2500)
        ancho = np.percentile([len(t) for t in textos], 90)
        bloque, cuenta = [], 0
        for p in parrafos_narrativa(lineas, ancho):
            bloque.append(p); cuenta += sum(palabras(MARCA.sub("", t)) for _, t in p[1])
            if cuenta >= n_pal: segs.append(bloque); bloque, cuenta = [], 0
        if bloque: segs.append(bloque)
        segs = [{"num": k + 1, "etiqueta": str(k + 1), "titulo": "", "acto": None, "parrafos": b} for k, b in enumerate(segs)]
        inf["avisos"].append(f"No encontré títulos de capítulo: dividí la obra en bloques de ~{n_pal} palabras.")
        inf.setdefault("antes", [])
    else:
        inicio = heads[0][0]
        if teatro:
            inicio = min(inicio, max([i for i, t in enumerate(textos[:inicio + 1]) if PAT_ACTO.match(t)] or [inicio]))
        inf["antes"] = inf.get("antes", []) + textos[:inicio]
        inf["irregulares"] = [textos[i] for i, *_ in heads if modo == "capitulo" and not re.match(r"^CAP[IÍ]TULO \S+$", textos[i], re.I)]
        ancho = np.percentile([len(t) for t in textos[inicio:]], 90)
        for k, (i, n, titulo, acto) in enumerate(heads):
            fin = heads[k + 1][0] if k + 1 < len(heads) else len(lineas)
            cuerpo = [l for l in lineas[i + 1:fin] if not (teatro and PAT_ACTO.match(l[0]))]
            if modo == "titulos": titulo = textos[i]
            if not titulo and cuerpo and len(cuerpo[0][0]) < 90 and not teatro:
                letras = re.sub(r"[^A-Za-zÁÉÍÓÚÑáéíóúñ]", "", cuerpo[0][0])
                if letras and sum(c.isupper() for c in letras) / len(letras) > 0.8:
                    titulo, cuerpo = cuerpo[0][0], cuerpo[1:]
            segs.append({"num": k + 1, "etiqueta": f"{a_romano(acto)}.{n}" if teatro else (a_romano(n) if modo == "romano" else str(n)),
                         "titulo": titular(titulo), "acto": acto,
                         "parrafos": parrafos_teatro(cuerpo) if teatro else parrafos_narrativa(cuerpo, ancho)})
    for s in segs:
        for pg, piezas in s["parrafos"]:
            for j, (v, t) in enumerate(piezas):
                for mal, bien in conf.get("erratas", {}).items(): t = t.replace(mal, bien)
                piezas[j] = (v, t)
    return {"modo": modo, "teatro": teatro, "segmentos": segs, "cita": inf["cita"]}, inf

# ======================= 3. ORACIONES Y TRAMOS =======================
PAT_TOK = re.compile(r"\d+(?:[.,]\d+)*|[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+")
RX_ORACION = re.compile(r"(?<=[.!?…])\s+")
def tokenizar(t):
    orig = PAT_TOK.findall(t); return orig, [sin_tildes(x.lower()) for x in orig]

def armar_oraciones(obra, incluir_acotaciones=False):
    O, n_par = [], 0
    for s in obra["segmentos"]:
        for pg0, piezas in s["parrafos"]:
            actual, hubo = pg0, False
            for v, t in piezas:
                if v == "ACOTACIÓN" and not incluir_acotaciones: continue
                for o in RX_ORACION.split(t):
                    m0 = MARCA.match(o.strip())
                    pag = int(m0.group(1)) if m0 else actual
                    ms = MARCA.findall(o)
                    if ms: actual = int(ms[-1])
                    limpio = re.sub(r"\s+", " ", MARCA.sub("", o)).strip()
                    orig, norm = tokenizar(limpio)
                    if norm:
                        O.append({"seg": s["num"], "par": n_par, "pag": pag, "voz": v, "texto": limpio, "toks": norm, "orig": orig}); hubo = True
            n_par += hubo
    return O

def armar_tramos(obra, conf, O):
    segs = obra["segmentos"]
    if conf.get("tramos"): return dict(conf["tramos"])
    if obra["teatro"]:
        tr = {}
        for s in segs: tr.setdefault(f"Acto {a_romano(s['acto'])}", []).append(s["num"])
        return tr
    if len(segs) <= 12:
        return {(f"{s['etiqueta']}. {s['titulo']}" if s["titulo"] else f"{ETQ_SEG[obra['modo']]} {s['etiqueta']}"): [s["num"]] for s in segs}
    largo = collections.Counter(o["seg"] for o in O for _ in o["toks"])
    total, tr, actual, acum = sum(largo.values()), {}, [], 0
    for s in segs:
        actual.append(s); acum += largo[s["num"]]
        if acum >= total / 6 * (len(tr) + 1) or s is segs[-1]:
            tr[f"{ETQ_SEG[obra['modo']]} {actual[0]['etiqueta']}–{actual[-1]['etiqueta']}"] = [x["num"] for x in actual]; actual = []
    return tr
ETQ_SEG = {"capitulo": "Cap.", "romano": "Cap.", "numero": "Cap.", "parte": "Parte", "propio": "Sección", "teatro": "Escena", "bloques": "Bloque", "titulos": "Cap."}

# ======================= 4. CANDIDATOS A MOTIVOS =======================
STOP = set("""a al algo alguien algun alguna algunas alguno algunos ante antes aqui ahi alli alla aca asi aun aunque bien cada casi
como con contra cual cuales cualquier cuando cuanto cuanta cuantos cuantas de del desde donde dos durante e el ella ellas ello ellos
en entre esa esas ese eso esos esta estas este esto estos hasta hacia la las le les lo los luego mas me mi mis mientras mio mia mios
mias mucho mucha muchos muchas muy nada nadie ni no nos nosotros nosotras nuestra nuestro nuestras nuestros o os otra otro otros otras
para pero poco poca pocos pocas por porque pues que quien quienes se si sin sino sobre su sus tal tan tanto tanta tantos tantas te ti
todo toda todos todas tras tu tus tuyo tuya un una uno unos unas usted ustedes vos vosotros vosotras vuestro vuestra vuestros vuestras
y ya yo ahora entonces despues jamas nunca siempre tambien tampoco solo sola solamente aquel aquella aquellos aquellas aquello mismo
misma mismos mismas suyo suya suyos suyas cuyo cuya cuyos cuyas consigo conmigo contigo menos apenas acaso quiza quizas demasiado
bastante pronto tarde temprano todavia ademas mediante segun hoy ayer manana etc oh ah eh ay adonde vete cosa cosas vez veces modo
manera parte partes lado momento rato forma caso gente demas cierto cierta propio propia primer primero primera segundo segunda ultimo
ultima bueno buena buen buenos buenas gran grande grandes nuevo nueva señor señora señores don doña mejor peor medio media lugar fin
tipo verdad sr ud uno dos tres cuatro cinco seis siete ocho nueve diez""".split())
def conjugar(inf):
    r, t = inf[:-2], inf[-2:]
    term = {"ar": "o as a amos ais an e aste o amos asteis aron aba abas abamos abais aban are aras ara aremos areis aran aria arias "
                  "ariamos ariais arian e es emos eis en ara aras aramos arais aran ase ases asen ando ado ada ados adas ar ad",
            "er": "o es e emos eis en i iste io imos isteis ieron ia ias iamos iais ian ere eras era eremos ereis eran eria erias eriamos "
                  "eriais erian a as amos ais an iera ieras ieramos ierais ieran iese iesen iendo ido ida idos idas er ed",
            "ir": "o es e imos is en i iste io isteis ieron ia ias iamos iais ian ire iras ira iremos ireis iran iria irias iriamos iriais "
                  "irian a as amos ais an iera ieras ieramos ierais ieran iese iesen iendo ido ida idos idas ir id"}[t]
    return {r + x for x in term.split()}
REGULARES = ("llegar pasar deber quedar hablar llevar dejar tomar acabar entrar responder contestar preguntar exclamar añadir replicar agregar "
             "continuar murmurar explicar tratar esperar ocurrir suceder resultar existir permitir lograr usar necesitar comprender recibir "
             "partir subir bajar terminar intentar insistir").split()
IRREGULARES = """ser soy eres es somos sois son fui fuiste fue fuimos fuisteis fueron era eras eramos erais eran sere seras sera seremos seran
seria serias seriamos serian sea seas seamos sean fuera fueras fueramos fueran fuese fuesen siendo sido sed estar estoy estas esta estamos
estais estan estuve estuviste estuvo estuvimos estuvieron estaba estabas estabamos estaban estare estara estaran estaria estarian este estes
esten estuviera estuvieran estuviese estando estado haber he has ha hemos habeis han hube hubo hubieron habia habias habiamos habian habre
habra habran habria habrian haya hayas hayamos hayan hubiera hubieras hubieramos hubieran hubiese hubiesen habiendo habido hay tener tengo
tienes tiene tenemos teneis tienen tuve tuviste tuvo tuvimos tuvieron tenia tenias teniamos tenian tendre tendra tendran tendria tendrian
tenga tengas tengamos tengan tuviera tuvieran tuviese teniendo tenido ten hacer hago haces hace hacemos haceis hacen hice hiciste hizo
hicimos hicieron hacia hacias haciamos hacian hare haras hara haran haria harian haga hagas hagamos hagan hiciera hicieran hiciese haciendo
hecho hecha hechos haz poder puedo puedes puede podemos podeis pueden pude pudo pudieron podia podias podiamos podian podre podra podran
podria podrias podrian pueda puedas puedan pudiera pudieran pudiese pudiendo podido decir digo dices dice decimos decis dicen dije dijiste
dijo dijimos dijeron decia decias deciamos decian dire diras dira diran diria dirian diga digas digamos digan dijera dijeran dijese diciendo
dicho dicha di decidme dime ir voy vas va vamos vais van iba ibas ibamos iban ire iras ira iran iria irian vaya vayas vayamos vayan yendo
ido dar doy das da damos dais dan di diste dio dimos dieron daba daban dare dara daria de des den diera dieran diese dando dado dada saber
se sabes sabe sabemos sabeis saben supe supo supieron sabia sabias sabian sabre sabra sabria sepa sepas sepan supiera supieran sabiendo
sabido querer quiero quieres quiere queremos quereis quieren quise quiso quisieron queria querias querian querre querra querria quiera
quieras quieran quisiera quisieran queriendo venir vengo vienes viene venimos vienen vine vinieron venia venian vendre vendra vendria venga
vengan viniera vinieran viniendo venido poner pongo pones pone ponemos ponen puse puso pusieron ponia ponian pondre pondra pondria ponga
pongan pusiera pusieran poniendo puesto puesta salir salgo sales sale salimos salen sali salio salieron salia salian saldre saldra saldria
salga salgan saliera salieran saliendo salido seguir sigo sigues sigue seguimos siguen segui siguio siguieron seguia seguian seguire
seguira seguiria siga sigan siguiera siguiendo seguido volver vuelvo vuelves vuelve volvemos vuelven volvi volvio volvieron volvia volvian
volvere volvera volveria vuelva vuelvan volviera volviendo vuelto pensar pienso piensas piensa pensamos piensan pense penso pensaron
pensaba pensaban pensare pensara pensaria piense piensen pensando pensado encontrar encuentro encuentra encuentran encontre encontro
encontraron encontraba encontrado encontrando creer creo crees cree creemos creen crei creyo creyeron creia creian creere creera creeria
crea crean creyera creyendo creido conocer conozco conoces conoce conocemos conocen conoci conocio conocieron conocia conocian conocera
conozca conociendo conocido parecer parezco pareces parece parecen parecio parecia parecian pareceria parezca parecido pareciendo empezar
empiezo empieza empiezan empece empezo empezaron empezaba empezando empezado""".split()
VERBOS_COMUNES = set(IRREGULARES) | {f for v in REGULARES for f in conjugar(sin_tildes(v))}
NO_PLURAL = {"dios", "pais", "mes", "tos", "lunes", "martes", "jueves", "viernes", "crisis", "atlas", "virus", "jesus", "venus", "seis", "tres"}

def candidatos(O, conf, por_lista=15):
    stop = STOP | VERBOS_COMUNES | set(conf.get("stop_extra", []))
    toks = [t for o in O for t in o["toks"]]; N = len(toks)
    frec = collections.Counter(toks)
    def clave_plural(w):
        if w in NO_PLURAL: return w
        op = []
        if w.endswith("ces") and len(w) > 4: op.append(w[:-3] + "z")
        if w.endswith("es") and len(w) > 4 and w[-3] in "rlndjzy": op.append(w[:-2])
        if w.endswith("s") and len(w) > 3 and w[-2] in "aeiou": op.append(w[:-1])
        return next((x for x in op if x in frec), w)
    orig_de, may, no_ini = collections.defaultdict(collections.Counter), collections.Counter(), collections.Counter()
    pos, seg_de = collections.defaultdict(list), []
    i = 0
    for o in O:
        for j, (a, n) in enumerate(zip(o["orig"], o["toks"])):
            orig_de[n][a.lower()] += 1; pos[n].append(i); seg_de.append(o["seg"]); i += 1
            if j > 0: no_ini[n] += 1; may[n] += a[0].isupper()
    seg_de = np.array(seg_de); largo_seg = collections.Counter(seg_de.tolist())
    grupos = collections.defaultdict(list)
    for w in frec:
        if w not in stop and len(w) >= 3 and not w.isdigit(): grupos[clave_plural(w)].append(w)
    min_frec, decil, filas = max(4, round(N / 15000)), np.arange(N) * 10 // N, []
    for k, formas in grupos.items():
        p = np.array(sorted(x for f in formas for x in pos[f])); n = len(p)
        if n < min_frec or k in stop: continue
        dec = np.bincount(decil[p], minlength=10)
        sc = collections.Counter(seg_de[p].tolist())
        pico = max(sc, key=lambda s: sc[s] / largo_seg[s] if sc[s] >= 2 else 0)
        princ = max(formas, key=lambda f: frec[f]); vista = orig_de[princ].most_common(1)[0][0]
        nombre = no_ini[princ] >= 2 and may[princ] / no_ini[princ] >= 0.85
        car = None
        if zipf_frequency and not nombre:
            z = max(zipf_frequency(orig_de[f].most_common(1)[0][0], "es") for f in formas)
            car = math.log10(n / N * 1e9) - z if z > 0 else None
        filas.append({"motivo": vista.capitalize() if nombre else vista, "formas": sorted(formas, key=lambda f: -frec[f]), "apariciones": n,
                      "recorrido": int((dec > 0).sum()), "concentracion": round(dec.max() / n, 2), "pico": pico, "nombre": nombre,
                      "caracteristico": car})
    C = pd.DataFrame(filas)
    dup = C.motivo.duplicated(keep=False)
    C.loc[dup, "motivo"] = C.loc[dup, "motivo"] + " (" + C.loc[dup, "formas"].map(lambda f: f[0]) + ")"
    C = C.set_index("motivo")
    peso = C.caracteristico.fillna(0).clip(lower=0.2) if zipf_frequency else 1
    L = {"recurrente": C[(~C.nombre) & (C.recorrido >= 8)].assign(_o=C.apariciones * peso).sort_values("_o", ascending=False),
         "concentrado": C[(~C.nombre) & (C.concentracion >= 0.45)].assign(_o=C.apariciones * C.concentracion).sort_values("_o", ascending=False),
         "característico": C[(~C.nombre) & C.caracteristico.notna()].sort_values("caracteristico", ascending=False) if zipf_frequency else C.iloc[0:0],
         "nombre": C[C.nombre].sort_values("apariciones", ascending=False)}
    return C, {k: v.head(por_lista).drop(columns="_o", errors="ignore") for k, v in L.items()}, frec

# ======================= 5. DICCIONARIO DE MOTIVOS =======================
def leer_diccionario(texto, tipo="propio"):
    out = []
    for linea in texto.strip().splitlines():
        if ":" not in linea: continue
        nombre, formas = linea.split(":", 1)
        fs = [sin_tildes(f.strip().lower()) for f in formas.split(",") if f.strip()]
        if fs: out.append({"nombre": nombre.strip(), "formas": fs, "tipo": tipo})
    return out
def coincide(w, formas): return any(w.startswith(f[:-1]) if f.endswith("*") else w == f for f in formas)
def diccionario_automatico(LISTAS, cupos={"recurrente": 5, "concentrado": 6, "característico": 5, "nombre": 3}):
    elegidos = list(dict.fromkeys(m for l, k in cupos.items() for m in LISTAS[l].index[:k]))
    tipo = lambda m: next(l for l, df in LISTAS.items() if m in df.index)
    return [{"nombre": m, "formas": list(next(df.loc[m, "formas"] for df in LISTAS.values() if m in df.index)), "tipo": tipo(m)} for m in elegidos]
def contar_motivo(m, frec): return sum(c for w, c in frec.items() if coincide(w, m["formas"]))

# ======================= 6. INFORME DE LIMPIEZA =======================
def informe_html(obra, inf, O, tramos, perfil):
    segs, esc = obra["segmentos"], html.escape
    pal = collections.Counter(); [pal.update({o["seg"]: len(o["toks"])}) for o in O]
    ok = lambda t: f'<li><span style="color:#2E8B6E">✔</span> {t}</li>'
    av = lambda t: f'<li><span style="color:#C0392B">⚠</span> {t}</li>'
    items = [ok(f"<b>{sum(pal.values()):,}</b> palabras en <b>{len(segs)}</b> segmentos "
                f"({ {'teatro': 'actos y escenas', 'bloques': 'bloques de longitud fija'}.get(obra['modo'], 'capítulos') }) "
                f"agrupados en <b>{len(tramos)}</b> tramos" + (f" · perfil: <i>{esc(perfil)}</i>" if perfil else " · sin perfil (detección automática)")),
             ok("Citas por <b>página impresa</b> del libro" if obra["cita"] == "impresa" else "Citas por <b>página del PDF</b> (no se detectó la numeración impresa)")]
    q = ", ".join(f"{k} ({v})" for k, v in inf["quitadas"].most_common())
    items.append(ok(f"Se quitó: {esc(q) or 'nada'}"))
    if inf.get("faltan"): items.append(av(f"Faltan títulos de segmento: {inf['faltan']}. Ese texto quedó unido al segmento anterior."))
    if inf.get("irregulares"): items.append(ok(f"Títulos dañados que se reconocieron igual: {esc(', '.join(inf['irregulares']))}"))
    for a in inf["avisos"]: items.append(av(esc(a)))
    antes = inf.get("antes", [])
    items.append(ok(f"Descartado antes del inicio: {palabras(' '.join(antes))} palabras"
                    + (f" (<i>{esc(' / '.join(antes[:2])[:90])}</i> … <i>{esc(' / '.join(antes[-2:])[:90])}</i>)" if antes else "")))
    if inf.get("final_cortado"): items.append(ok(f"Cortado al final desde: <i>{esc(' / '.join(inf['final_cortado'])[:120])}</i>"))
    filas = "".join(f"<tr><td>{esc(s['etiqueta'])}</td><td>{esc(s['titulo'][:38])}</td><td style='text-align:right'>{pal[s['num']]:,}</td>"
                    f"<td>{esc(next((o['texto'] for o in O if o['seg'] == s['num']), '')[:80])}</td></tr>" for s in segs)
    return (f'<div style="font-family:system-ui,sans-serif;font-size:14px;line-height:1.5;max-width:900px">'
            f'<h3 style="margin:4px 0">Informe de limpieza</h3><ul style="padding-left:18px;margin:6px 0">{"".join(items)}</ul>'
            f'<details><summary style="cursor:pointer">Ver cómo empieza cada segmento (revisa que coincida con el libro)</summary>'
            f'<table style="font-size:12.5px;border-collapse:collapse;margin-top:6px"><tr style="text-align:left"><th>seg.</th><th>título</th>'
            f'<th>palabras</th><th>empieza</th></tr>{filas}</table></details></div>')

# ======================= 7. LA LECTURA (HTML) =======================
def datos_lectura(obra, O, tramos, motivos, LISTAS, titulo, autor):
    tr_idx = {n: k for k, ns in enumerate(tramos.values()) for n in ns}
    voces = list(dict.fromkeys(o["voz"] for o in O))
    vi = {v: k for k, v in enumerate(voces)}
    return {"titulo": titulo, "autor": autor or "", "teatro": obra["teatro"], "cita": obra["cita"],
            "unidad": "parrafo" if obra["teatro"] else "oracion", "tipo_seg": ETQ_SEG[obra["modo"]],
            "segmentos": [{"e": s["etiqueta"], "t": s["titulo"], "tr": tr_idx[s["num"]]} for s in obra["segmentos"]],
            "tramos": list(tramos), "voces": [v.title() if obra["teatro"] and v not in ("diálogo", "narración") else v for v in voces],
            "O": [[o["seg"] - 1, o["par"], o["pag"], vi[o["voz"]], o["texto"]] for o in O],
            "candidatos": {l: [{"n": m, "f": list(df.loc[m, "formas"])[:6], "c": int(df.loc[m, "apariciones"])} for m in df.index]
                           for l, df in LISTAS.items()},
            "motivos": [{"n": m["nombre"], "f": m["formas"], "t": m["tipo"]} for m in motivos]}

def generar_lectura(datos, plantilla):
    js = json.dumps(datos, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")
    return plantilla.replace("__DATOS__", js)

def documento(fragmento, titulo):
    return (f'<!doctype html><html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">'
            f'<title>Lectura en red · {html.escape(titulo)}</title><style>html,body{{margin:0;background:#f7f4ee}}'
            f'@media (prefers-color-scheme:dark){{html,body{{background:#16181d}}}}</style></head><body>{fragmento}</body></html>')

# ======================= 8. LECTURA ASISTIDA CON IA (opcional) =======================
CLIENTE, MODELO_IA, IA_REGISTRO = None, "gpt-6-luna", []
def conectar_ia(modelo):
    global CLIENTE, MODELO_IA
    MODELO_IA = modelo
    clave = os.environ.get("OPENAI_API_KEY")
    if not clave and EN_COLAB:
        try:
            from google.colab import userdata; clave = userdata.get("OPENAI_API_KEY")
        except Exception: clave = None
    if not clave and (EN_COLAB or "ipykernel" in sys.modules):
        from getpass import getpass
        clave = getpass("Pega tu clave de OpenAI (no se mostrará ni se guardará; Enter para saltar): ").strip() or None
    if not clave: print("Sin clave de OpenAI: la IA queda desactivada."); return False
    try: from openai import OpenAI
    except ImportError: instalar("openai"); from openai import OpenAI
    CLIENTE = OpenAI(api_key=clave); print(f"IA conectada (modelo {modelo})."); return True

def preguntar(sistema, pedido, intentos=2):
    for k in range(intentos):
        extra = "" if k == 0 else "\n\nResponde SOLO con un objeto JSON válido."
        try:
            r = CLIENTE.responses.create(model=MODELO_IA, instructions=sistema, input=pedido + extra); texto = r.output_text
        except AttributeError:
            r = CLIENTE.chat.completions.create(model=MODELO_IA, messages=[{"role": "system", "content": sistema},
                                                                            {"role": "user", "content": pedido + extra}])
            texto = r.choices[0].message.content
        try:
            texto = re.sub(r"^```(?:json)?|```$", "", texto.strip(), flags=re.M)
            return json.loads(texto[texto.find("{"):texto.rfind("}") + 1])
        except Exception:
            continue
    raise ValueError("El modelo no devolvió JSON válido; vuelve a intentarlo o cambia de modelo.")

def sistema_ia(titulo):
    return (f"Eres un asistente de análisis literario para un taller universitario. Trabajas sobre «{titulo}». Basa tus afirmaciones "
            f"SOLO en los fragmentos numerados que recibes y cítalos por número. Lo que añadas por conocimiento previo de la obra va aparte, "
            f"en el campo indicado. Si los fragmentos no sostienen algo, dilo. Español claro y breve. Responde SOLO con JSON válido.")

def lugar(o): return f"{ETQ_SEG_ACTUAL} {SEG_ETQ[o['seg']]}, p. {o['pag']}"

def ia_sugerir_motivos(LISTAS, O, frec, titulo, n=45):
    cands = pd.concat([df.assign(lista=l) for l, df in LISTAS.items() if len(df)])
    cands = cands[~cands.index.duplicated()].head(n); lineas = []
    for m, f in cands.iterrows():
        ej = next((o for o in O if f["formas"][0] in o["toks"] and 6 <= len(o["toks"]) <= 40), None)
        lineas.append(f"- {m} [{f['lista']}; {f['apariciones']} veces; formas: {', '.join(f['formas'][:5])}]" + (f" Ejemplo: «{ej['texto'][:180]}»" if ej else ""))
    r = preguntar(sistema_ia(titulo), "Palabras candidatas a motivos (imágenes, objetos, espacios, ideas, personajes que se repiten):\n\n" + "\n".join(lineas) +
        "\n\nPropón entre 12 y 16 motivos. Agrupa palabras de la lista cuando nombren lo mismo y añade variantes con asterisco (mir*). Descarta las genéricas. "
        'JSON: {"motivos": [{"nombre": str, "formas": [str], "por_que": str}]}')
    out = []
    for m in r.get("motivos", []):
        mot = {"nombre": m.get("nombre", "?"), "formas": [sin_tildes(x.strip().lower()) for x in m.get("formas", []) if x.strip()], "tipo": "IA"}
        if contar_motivo(mot, frec): out.append(mot)
    IA_REGISTRO.append({"tarea": "sugerir_motivos", "modelo": MODELO_IA, "respuesta": r})
    return out

def _frases_con(O, motivos_ids, k, semilla=1):
    idx = [i for i, o in enumerate(O) if all(any(coincide(t, m["formas"]) for t in o["toks"]) for m in motivos_ids)]
    if len(idx) > k: idx = [idx[int(round(j))] for j in np.linspace(0, len(idx) - 1, k)]
    return idx

def ia_leer_conexion(O, motivos, a, b, titulo, k=10):
    ma, mb = next((m for m in motivos if m["nombre"] == a), None), next((m for m in motivos if m["nombre"] == b), None)
    if not ma or not mb: print("Escribe dos motivos de tu diccionario:", ", ".join(m["nombre"] for m in motivos)); return
    idx = _frases_con(O, [ma, mb], k)
    if not idx: print(f"«{a}» y «{b}» nunca aparecen en la misma oración."); return
    frags = "\n".join(f"[{j + 1}] ({lugar(O[i])}) {O[i]['texto'][:320]}" for j, i in enumerate(idx))
    r = preguntar(sistema_ia(titulo), f"Motivo A: «{a}» ({', '.join(ma['formas'])}). Motivo B: «{b}» ({', '.join(mb['formas'])}).\n"
        f"Fragmentos donde aparecen juntos, repartidos por la obra:\n{frags}\n\n"
        'JSON: {"relacion": str (2-4 frases), "citas": [int], "cambio": str (si el vínculo cambia a lo largo de la obra), '
        '"matiz": str (qué fragmento matiza la lectura o "ninguno"), "fuera_de_los_fragmentos": str (conocimiento previo o "nada"), "que_leer": str}')
    citas = [c for c in r.get("citas", []) if isinstance(c, int) and 1 <= c <= len(idx)]
    malas = [c for c in r.get("citas", []) if c not in citas]
    e = lambda x: html.escape(str(x))
    citado = "".join(f"<li><b>[{c}]</b> <small>{e(lugar(O[idx[c - 1]]))}</small> — {e(O[idx[c - 1]]['texto'])}</li>" for c in citas)
    display(HTML(f'<div style="font-family:system-ui;font-size:14px;line-height:1.5;max-width:880px"><h3>{e(a)} — {e(b)}</h3>'
        f'<p>{e(r.get("relacion", ""))}</p><p><b>¿Cambia?</b> {e(r.get("cambio", ""))}</p><p><b>Lo matiza:</b> {e(r.get("matiz", ""))}</p>'
        f'<p style="background:#fff3d6;padding:6px 10px;border-radius:6px"><b>Fuera de los fragmentos (no verificado):</b> {e(r.get("fuera_de_los_fragmentos", ""))}</p>'
        f'<p><b>Qué leer:</b> {e(r.get("que_leer", ""))}</p>' + (f'<p style="color:#b03a2e">⚠ Citó fragmentos inexistentes: {malas}</p>' if malas else "")
        + f'<b>Frases que cita</b><ul>{citado or "<li><i>Ninguna: la lectura no está anclada.</i></li>"}</ul></div>'))
    IA_REGISTRO.append({"tarea": "leer_conexion", "a": a, "b": b, "fragmentos": [O[i]["texto"] for i in idx], "respuesta": r})

def ia_precision(O, motivos, nombre, sentido, titulo, k=20):
    m = next((x for x in motivos if x["nombre"] == nombre), None)
    if not m: print("Escribe un motivo de tu diccionario:", ", ".join(x["nombre"] for x in motivos)); return
    idx = [i for i, o in enumerate(O) if any(coincide(t, m["formas"]) for t in o["toks"])]
    random.Random(1).shuffle(idx); idx = idx[:k]
    frags = "\n".join(f"[{j + 1}] {O[i]['texto'][:220]}" for j, i in enumerate(idx))
    r = preguntar(sistema_ia(titulo), f"Motivo «{nombre}», en este sentido: {sentido}. Palabras: {', '.join(m['formas'])}.\n{frags}\n\n"
        'Para cada fragmento, ¿la palabra se usa en ese sentido? JSON: {"juicios": [{"n": int, "veredicto": "sí|no|dudoso", "razon": str}]}')
    jz = {j.get("n"): j for j in r.get("juicios", []) if isinstance(j, dict)}
    tabla = pd.DataFrame([{"veredicto": jz.get(j + 1, {}).get("veredicto", "—"), "razón": jz.get(j + 1, {}).get("razon", ""),
                           "frase": O[i]["texto"][:150]} for j, i in enumerate(idx)])
    si, no = (tabla.veredicto == "sí").sum(), (tabla.veredicto == "no").sum()
    print(f"«{nombre}» ({sentido}): precisión estimada ≈ {si / max(si + no, 1):.0%} ({si} sí, {no} no). Lee tú los «no» y «dudoso».")
    display(tabla.sort_values("veredicto"))
    IA_REGISTRO.append({"tarea": "precision", "motivo": nombre, "sentido": sentido, "tabla": tabla.to_dict("records")})

print("Motor listo.", "wordfreq disponible." if zipf_frequency else "Sin wordfreq: se omite la lista de palabras características.")

# ======================= 9. PERSONAJES =======================
# Quién aparece, cuánto, cuándo y con quién. Se cuentan las MENCIONES EXPLÍCITAS (nombres y papeles como «el padre»);
# los pronombres («él», «ella») no se resuelven. En teatro también cuentan los parlamentos de cada hablante.
NO_PERSONA = set("""dios dioses cristo jesucristo señor señora virgen santisima espiritu santo diablo demonio usted ustedes vuestra vuestro
merced excelencia majestad alteza señoria lunes martes miercoles jueves viernes sabado domingo enero febrero marzo abril mayo junio
julio agosto setiembre septiembre octubre noviembre diciembre navidad pascua oh ah eh ay bah hola adios gracias capitulo parte acto escena
fin nota prologo epilogo indice""".split())
HONORIFICOS = set("don doña señor señora señorita niño niña mama papa tio tia taita compadre comadre fray sor doctor coronel general capitan "
                  "sargento teniente comandante cabo padre madre hermano hermana maestro".split())
VERBOS_HABLA = set("dijo dice decia pregunto preguntaba respondio respondia contesto contestaba grito gritaba exclamo murmuro replico "
                   "explico añadio agrego insistio susurro ordeno llamo conto dijeron".split())
LOCATIVOS = {"a", "de", "del", "en", "hacia", "desde", "hasta"}
PARENTESCO = {"hijo": "hijo", "hija": "hija", "padre": "padre", "madre": "madre", "hermano": "hermano", "hermana": "hermana",
              "esposa": "esposa", "mujer": "esposa", "marido": "esposo", "esposo": "esposo", "abuelo": "abuelo", "abuela": "abuela",
              "nieto": "nieto", "nieta": "nieta", "tio": "tío", "tia": "tía", "sobrino": "sobrino", "sobrina": "sobrina",
              "primo": "primo", "prima": "prima", "hijastro": "hijastro", "hijastra": "hijastra", "suegro": "suegro", "suegra": "suegra",
              "cuñado": "cuñado", "cuñada": "cuñada", "novia": "novia", "novio": "novio", "amante": "amante", "viuda": "viuda"}
PAPELES = ("padre madre hermana hermano hijo hija esposa marido abuela abuelo tio tia criada criado sirvienta asistenta señora señor "
           "jefe patron patrona rey reina principe princesa capitan coronel cura medico doctor maestro maestra apoderado huespedes "
           "principito zorro serpiente piloto aviador viejo vieja niño niña muchacho muchacha").split()

COLECTIVOS = set("los las todos todas ambos varios voces unos unas otros coro".split())
FEMENINOS = set("madre hermana hija esposa abuela tia criada sirvienta asistenta señora patrona reina princesa maestra vieja niña muchacha serpiente flor rosa".split())
def articulo(p): return ("las" if p in FEMENINOS else "los") if p.endswith("s") and p not in ("dios",) else ("la" if p in FEMENINOS or p.endswith("a") else "el")
SUJETO = set("se le les lo la los las me te nos no ya tambien también nunca siempre había estaba tenía era fue dijo estuvo tuvo volvió hizo quedó sintió".split())
SUFIJOS_NOMBRE = {"segundo", "segunda", "tercero", "tercera", "primero", "primera", "chico", "chica", "grande", "menor", "mayor", "hijo", "viejo"}
def _spans(texto): return [(m.start(), m.end()) for m in PAT_TOK.finditer(texto)]

def tokens_nombre(O, stop_extra=()):
    """Palabras que casi siempre van con mayúscula cuando no empiezan la oración: nombres propios."""
    may, no_ini = collections.Counter(), collections.Counter()
    for o in O:
        for j, (a, n) in enumerate(zip(o["orig"], o["toks"])):
            if j > 0: no_ini[n] += 1; may[n] += a[0].isupper()
    malas = STOP | NO_PERSONA | set(stop_extra)          # no se excluyen verbos: «Tomás» ≠ «tomas»
    return {n for n in no_ini if no_ini[n] >= 2 and may[n] / no_ini[n] >= 0.85 and n not in malas and len(n) >= 2 and not n.isdigit()}

CONECTORES = {"de", "del", "la"}
def secuencias_nombre(O, NOM):
    """Tramos de nombres seguidos (sin puntuación en medio): «Rosendo Maqui», «José Arcadio Buendía», «Fernanda del Carpio»."""
    for i, o in enumerate(O):
        sp, tx, orig, toks, run, j = _spans(o["texto"]), o["texto"], o["orig"], o["toks"], [], 0
        es_nom = lambda q: q < len(toks) and orig[q][0].isupper() and toks[q] in NOM
        pegado = lambda a, b: tx[sp[a][1]:sp[b][0]].strip() == ""
        while j < len(toks):
            a, n = orig[j], toks[j]
            if run and len(run) < 6 and pegado(run[-1], j):
                if a[0].isupper() and (n in NOM or n in SUFIJOS_NOMBRE): run.append(j); j += 1; continue
                # «de», «del», «de la» dentro del nombre, solo si sigue otro nombre
                k = j
                while k < len(toks) and toks[k] in CONECTORES and not orig[k][0].isupper() and k - j < 2 and pegado(k - 1 if k > j else run[-1], k): k += 1
                if k > j and es_nom(k) and pegado(k - 1, k): run += list(range(j, k + 1)); j = k + 1; continue
            if run: yield i, run
            run = [j] if (a[0].isupper() and (n in NOM or (n in ("santa", "san") and es_nom(j + 1)))) else []
            j += 1
        if run: yield i, run

def proponer_reparto(O, obra, conf, max_personajes=25):
    """Propuesta automática del reparto. Devuelve (texto del diccionario, informe de cómo se armó)."""
    N = sum(len(o["toks"]) for o in O)
    NOM = tokens_nombre(O, conf.get("stop_extra", []))
    cuenta, previa, habla, sujeto = collections.Counter(), collections.defaultdict(collections.Counter), collections.Counter(), collections.Counter()
    vista = collections.defaultdict(collections.Counter)
    en_oracion = collections.defaultdict(set)
    for i, run in secuencias_nombre(O, NOM):
        o = O[i]; ts = tuple(o["toks"][j] for j in run)
        while len(ts) > 1 and ts[0] in HONORIFICOS: ts, run = ts[1:], run[1:]
        if len(ts) == 2 and ts[0] in ("san", "santa"): ts, run = ts[1:], run[1:]
        cuenta[ts] += 1; vista[ts][" ".join(o["orig"][j] for j in run)] += 1; en_oracion[ts].add(i)
        previa[ts][o["toks"][run[0] - 1] if run[0] > 0 else "^"] += 1
        cerca = o["toks"][max(0, run[0] - 3):run[0]] + o["toks"][run[-1] + 1:run[-1] + 4]
        habla[ts] += any(w in VERBOS_HABLA for w in cerca)
        sig = o["orig"][run[-1] + 1] if run[-1] + 1 < len(o["orig"]) else ""
        sujeto[ts] += bool(sig) and (sig.lower() in SUJETO or re.search(r"(ó|aba|ía|ió|aron|ieron|aban|ían)$", sig) is not None)
    minc = max(3, round(N / 25000))
    cands = {s for s, c in cuenta.items() if c >= minc}
    # en teatro, los hablantes son personajes seguros
    hablantes = {}
    if obra["teatro"]:
        for v, c in collections.Counter(o["voz"] for o in O).items():
            ts = tuple(t for t in tokenizar(v)[1] if not t.isdigit())
            if v in ("narración", "diálogo", "ACOTACIÓN") or c < 2 or not ts or ts[0] in COLECTIVOS: continue
            if ts in hablantes: hablantes[ts] = (hablantes[ts][0], hablantes[ts][1] + c)
            else: hablantes[ts] = (re.sub(r"\s*\d+$", "", v).title(), c)
    largos = sorted(cands, key=len, reverse=True)
    def contiene(L, s): return any(L[k:k + len(s)] == s for k in range(len(L) - len(s) + 1))
    alias, compartidos, familias, distintos = collections.defaultdict(list), {}, {}, []
    principales = []
    for s in sorted(cands, key=lambda x: (-len(x), -cuenta[x])):
        cont = [L for L in largos if len(L) > len(s) and L in principales and contiene(L, s)]
        # si la forma corta y la larga se nombran en la misma oración, son personas distintas («su hijo Ignacio Ventura … Ignacio Ventura Olmedo»)
        juntos = [L for L in cont if (lambda c: c >= 2 or c >= 0.05 * min(cuenta[s], cuenta[L]))(len(en_oracion[s] & en_oracion[L]))]
        if juntos: distintos += [(s, L) for L in juntos]; principales.append(s); continue    # «Aureliano» ≠ «Aureliano Segundo»: es otro
        if not cont: principales.append(s); continue
        if len(cont) == 1: alias[cont[0]].append(s); continue
        los = previa[s]["los"] + previa[s]["las"]
        if los >= 0.3 * cuenta[s]: familias[s] = cont; continue        # «los Buendía»: apellido de familia
        compartidos[s] = cont                                           # «Aureliano», «Maqui»: se decide por contexto
    for s, cont in compartidos.items():
        for L in cont: alias[L].append(s)
    total = lambda L: cuenta[L] + sum(cuenta[a] for a in alias[L])
    lugares = []
    filas = []
    for L in principales:
        menc = total(L)
        loc = sum(previa[x][w] for x in [L] + alias[L] for w in LOCATIVOS) / max(1, menc)
        ini = sum(previa[x]["^"] for x in [L] + alias[L]) / max(1, menc)
        hab = sum(habla[x] for x in [L] + alias[L]) / max(1, menc)
        suj = sum(sujeto[x] for x in [L] + alias[L]) / max(1, menc)
        lugar = loc >= 0.45 and ini < 0.15 and hab < 0.05 and suj < 0.3 if len(L) == 1 else loc >= 0.7 and hab < 0.02
        if L not in hablantes and lugar: lugares.append((L, menc)); continue
        filas.append((L, menc))
    filas = [(L, c + 2 * hablantes[L][1]) if L in hablantes else (L, c) for L, c in filas]
    for ts, (v, c) in hablantes.items():
        if not any(ts == L or contiene(L, ts) for L, _ in filas): filas.append((ts, cuenta.get(ts, 0) + 2 * c))
    filas.sort(key=lambda x: (x[0] not in hablantes, -x[1]))
    filas = filas[:max_personajes]
    nombre_de = lambda L: hablantes[L][0] if L in hablantes else vista[L].most_common(1)[0][0]
    lineas = []
    for L, _ in filas:
        fs = [L] + sorted(alias[L], key=lambda a: -cuenta[a])
        lineas.append(f"{nombre_de(L)}: " + ", ".join(" ".join(f) for f in fs))
    # papeles sin nombre (el padre, la hermana…) si hay pocos nombres
    papeles = []
    for p in PAPELES:
        c = sum(1 for o in O for j, t in enumerate(o["toks"]) if t == p and j > 0 and o["toks"][j - 1] in ("el", "la", "los", "las", "su"))
        if c >= max(8, round(N / 8000)): papeles.append((p, c))
    papeles.sort(key=lambda x: -x[1])
    if len(filas) < 5:
        for p, _ in papeles[:max(0, 8 - len(filas))]: lineas.append(f"{articulo(p)} {p}: {p}")
    elegidos = {L for L, _ in filas}
    info = {"distintos": [(nombre_de(a), nombre_de(b)) for a, b in distintos if a in elegidos and b in elegidos],
            "compartidos": {" ".join(s): [nombre_de(L) for L in c if L in elegidos] for s, c in compartidos.items()},
            "familias": [vista[s].most_common(1)[0][0] for s in familias],
            "lugares": [vista[L].most_common(1)[0][0] for L, _ in sorted(lugares, key=lambda x: -x[1])[:12]],
            "papeles": [p for p, _ in papeles], "teatro_papeles": obra["teatro"] and any(p in ("rey", "reina", "principe", "princesa") for p, _ in papeles)}
    return "\n".join(lineas), info

def leer_reparto(texto, NOM=frozenset()):
    """«Nombre: forma, otra forma, forma*, forma [12-20]». Una forma puede tener varias palabras; [a-b] limita a esos segmentos."""
    R = []
    for linea in texto.strip().splitlines():
        if ":" not in linea or linea.strip().startswith("#"): continue
        nombre, resto = linea.split(":", 1)
        formas = []
        for f in resto.split(","):
            f = f.strip(); rango = None
            m = re.search(r"\[\s*(\d+)\s*[-–]\s*(\d+)\s*\]\s*$", f)
            if m: rango, f = (int(m.group(1)), int(m.group(2))), f[:m.start()].strip()
            ts = [sin_tildes(t.lower()) for t in f.split()]
            if ts: formas.append({"t": ts, "r": rango})
        if formas: R.append({"nombre": nombre.strip(), "formas": formas})
    return R

def _coincide_tok(t, f): return t.startswith(f[:-1]) if f.endswith("*") else t == f

def buscar_menciones(O, reparto, NOM):
    """Cada mención: (oración, inicio, fin, [personajes posibles]). Se toma la forma más larga; si dos personajes comparten
    la forma (p. ej. «Aureliano»), la mención queda ambigua y se resuelve después por contexto."""
    por_inicio = collections.defaultdict(list)
    for k, p in enumerate(reparto):
        for f in p["formas"]: por_inicio[f["t"][0].rstrip("*")[:3]].append((f["t"], f["r"], k))
    M = []
    for i, o in enumerate(O):
        toks, orig, sp, j = o["toks"], o["orig"], None, 0
        while j < len(toks):
            mejor, quien = 0, set()
            for ft, rango, k in por_inicio.get(toks[j][:3], ()):
                n = len(ft)
                if j + n > len(toks) or n < mejor: continue
                if rango and not (rango[0] <= o["seg"] <= rango[1]): continue
                if not all(_coincide_tok(toks[j + q], ft[q]) for q in range(n)): continue
                if any(ft[q] in NOM and not orig[j + q][0].isupper() for q in range(n)): continue   # «rosa» no es «Rosa»
                if n > mejor: mejor, quien = n, {k}
                else: quien.add(k)
            if mejor:
                sp = sp or _spans(o["texto"])
                M.append({"o": i, "a": sp[j][0], "b": sp[j + mejor - 1][1], "pos": j, "n": mejor, "p": sorted(quien), "pj": None})
                j += mejor
            else: j += 1
    return M

def resolver_ambiguas(M, O, compartidos="resolver por contexto"):
    """Una mención compartida se asigna al candidato nombrado sin ambigüedad más cerca, dentro del mismo segmento."""
    tok_ini, acc = [], 0
    for o in O: tok_ini.append(acc); acc += len(o["toks"])
    for m in M:
        m["g"] = tok_ini[m["o"]] + m["pos"]
        if len(m["p"]) == 1: m["pj"] = m["p"][0]
    if compartidos != "resolver por contexto": return {"resueltas": 0, "sin_resolver": sum(m["pj"] is None for m in M)}
    por_seg = collections.defaultdict(list)
    for m in M: por_seg[O[m["o"]]["seg"]].append(m)
    res = sin = 0
    for ms in por_seg.values():
        claras = [m for m in ms if m["pj"] is not None]
        for m in ms:
            if m["pj"] is not None: continue
            mejor = None
            for c in claras:
                if c["pj"] not in m["p"]: continue
                d = m["g"] - c["g"]; d = d if d >= 0 else -d * 1.5          # se prefiere a quien se nombró antes
                if mejor is None or d < mejor[0]: mejor = (d, c["pj"])
            if mejor: m["pj"], m["amb"] = mejor[1], True; res += 1
            else: sin += 1
    return {"resueltas": res, "sin_resolver": sin}

def pasajes(O, obra, palabras_min=120):
    """Unidad de relación: párrafos (o parlamentos) seguidos hasta sumar ~120 palabras, sin cruzar de segmento."""
    P, ini, acc, par = [], 0, 0, None
    for i, o in enumerate(O):
        if i > ini and (o["seg"] != O[i - 1]["seg"] or (o["par"] != par and acc >= palabras_min)):
            P.append((ini, i - 1)); ini, acc = i, 0
        par = o["par"]; acc += len(o["toks"])
    if O: P.append((ini, len(O) - 1))
    return P

def _louvain(n, edges):
    w = [dict() for _ in range(n)]; deg = [0.0] * n
    for a, b, f in edges: w[a][b] = w[b][a] = f; deg[a] += f; deg[b] += f
    m2 = sum(deg) or 1; com = list(range(n)); tot = deg[:]
    for _ in range(20):
        mov = False
        for i in range(n):
            if not deg[i]: continue
            ci = com[i]; tot[ci] -= deg[i]; enl = collections.Counter()
            for j, f in w[i].items(): enl[com[j]] += f
            mejor, gan = ci, enl.get(ci, 0) - tot[ci] * deg[i] / m2
            for c, v in enl.items():
                g = v - tot[c] * deg[i] / m2
                if g > gan + 1e-9: gan, mejor = g, c
            com[i] = mejor; tot[mejor] += deg[i]; mov |= mejor != ci
        if not mov: break
    tam = collections.Counter(com[i] for i in range(n) if deg[i])
    orden = [c for c, _ in tam.most_common()]
    return [orden.index(com[i]) if deg[i] else -1 for i in range(n)]

def _intermediacion(n, edges):
    import heapq
    adj = [[] for _ in range(n)]
    for a, b, f in edges: adj[a].append((b, 1 / f)); adj[b].append((a, 1 / f))
    cb = [0.0] * n
    for s in range(n):
        if not adj[s]: continue
        dist, sigma, pred, orden = [math.inf] * n, [0] * n, [[] for _ in range(n)], []
        dist[s], sigma[s], h, hecho = 0, 1, [(0, s)], [False] * n
        while h:
            d, u = heapq.heappop(h)
            if hecho[u]: continue
            hecho[u] = True; orden.append(u)
            for v, ww in adj[u]:
                nd = d + ww
                if nd < dist[v] - 1e-12: dist[v], sigma[v], pred[v] = nd, sigma[u], [u]; heapq.heappush(h, (nd, v))
                elif abs(nd - dist[v]) < 1e-12: sigma[v] += sigma[u]; pred[v].append(u)
        delta = [0.0] * n
        for w_ in reversed(orden):
            for v in pred[w_]: delta[v] += sigma[v] / sigma[w_] * (1 + delta[w_])
            if w_ != s: cb[w_] += delta[w_]
    k = sum(1 for x in adj if x); norma = (k - 1) * (k - 2) if k > 2 else 1
    return [x / norma for x in cb]

def _disponer(n, edges, activos, semilla=7):
    """Resortes (Fruchterman–Reingold) para el grupo principal; los sueltos en una fila abajo."""
    par = list(range(n))
    def raiz(x):
        while par[x] != x: par[x] = par[par[x]]; x = par[x]
        return x
    for a, b, _ in edges: par[raiz(a)] = raiz(b)
    comps = collections.defaultdict(list)
    for i in activos: comps[raiz(i)].append(i)
    lista = sorted(comps.values(), key=len, reverse=True); pos = [None] * n
    def resortes(nodos):
        m = len(nodos)
        if m == 1: return np.array([[0.5, 0.5]])
        idx = {x: i for i, x in enumerate(nodos)}
        es = [(idx[a], idx[b], f) for a, b, f in edges if a in idx and b in idx]
        fmax = max([f for *_, f in es] + [1e-6])
        rng = np.random.default_rng(semilla)
        p = np.c_[0.5 + 0.4 * np.cos(2 * np.pi * np.arange(m) / m), 0.5 + 0.4 * np.sin(2 * np.pi * np.arange(m) / m)] + rng.normal(0, .01, (m, 2))
        kk = math.sqrt(1 / m) * 0.9
        for it in range(500):
            t = 0.08 * (1 - it / 500) + 0.002
            dx = p[:, None, :] - p[None, :, :]; dist = np.linalg.norm(dx, axis=2) + np.eye(m); dist[dist < 1e-3] = 1e-3
            d = (dx / dist[..., None] * (kk * kk / dist)[..., None]).sum(1)
            for a, b, f in es:
                v = p[a] - p[b]; dd = max(np.linalg.norm(v), 1e-3); fz = dd * dd / kk * (0.4 + f / fmax)
                d[a] -= v / dd * fz; d[b] += v / dd * fz
            ln = np.linalg.norm(d, axis=1, keepdims=True); ln[ln == 0] = 1
            p += d / ln * np.minimum(ln, t)
        p = (p - p.min(0)) / np.where(np.ptp(p, 0) == 0, 1, np.ptp(p, 0))
        d0 = min(0.085, 0.5 / math.sqrt(m))                      # que los nodos no se monten unos sobre otros (en proporción de pantalla)
        esc = np.array([1.0, 0.52]); p = p * esc
        for _ in range(300):
            dx = p[:, None, :] - p[None, :, :]; dist = np.linalg.norm(dx, axis=2) + np.eye(m) * 9
            cerca = dist < d0
            if not cerca.any(): break
            emp = (dx / np.maximum(dist, 1e-4)[..., None] * ((d0 - dist) * cerca)[..., None]).sum(1) * 0.5
            p = p + emp
        p = p / esc
        p = (p - p.min(0)) / np.where(np.ptp(p, 0) == 0, 1, np.ptp(p, 0))
        return p
    if lista:
        for x, q in zip(lista[0], resortes(lista[0])): pos[x] = [float(q[0]), float(q[1]) * 0.8]
    resto = lista[1:]; ancho = sum(len(c) for c in resto); x0 = 0
    for c in resto:
        w = len(c) / ancho
        for x, q in zip(c, resortes(c)): pos[x] = [x0 + w * (0.15 + 0.7 * float(q[0])), 0.93 + 0.07 * float(q[1])]
        x0 += w
    return pos

def pistas_parentesco(O, M, reparto):
    """«X, hijo de Y» · «Y y su hija X» · «X, su esposa»: frases donde el texto nombra un parentesco entre dos personajes."""
    por_o = collections.defaultdict(list)
    for m in M:
        if m["pj"] is not None: por_o[m["o"]].append(m)
    pistas = collections.defaultdict(list)
    for i, ms in por_o.items():
        tx, toks = O[i]["texto"], O[i]["toks"]; sp = _spans(tx); ms = sorted(ms, key=lambda m: m["pos"])
        fin = lambda m: m["pos"] + len(PAT_TOK.findall(tx[m["a"]:m["b"]]))
        entre = lambda c1, t: tx[c1:sp[t][0]] if t < len(sp) else ""            # texto entre un carácter y el token t
        for x, A in enumerate(ms):
            j = fin(A)
            if j >= len(toks): continue
            k = j + (toks[j] in ("el", "la"))
            # A, (el|la) K de B          «Arcadio, el hijo de José Arcadio»
            if k + 1 < len(toks) and toks[k] in PARENTESCO and toks[k + 1] in ("de", "del") and entre(A["b"], j).strip() == ",":
                B = next((b for b in ms[x + 1:] if b["pos"] in (k + 2, k + 3)), None)
                if B and B["pj"] != A["pj"]:
                    pistas[(A["pj"], PARENTESCO[toks[k]], B["pj"])].append(i)
                    fb = fin(B)                                              # «… hijo de B y C»
                    C = next((c for c in ms if c["pos"] in (fb + 1, fb + 2, fb + 3) and fb < len(toks) and toks[fb] in ("y", "e")), None)
                    if C and C["pj"] not in (A["pj"], B["pj"]) and toks[k] in ("hijo", "hija"): pistas[(A["pj"], PARENTESCO[toks[k]], C["pj"])].append(i)
            # B … A, su K               «habló con Juanacha, su hija»
            if j + 1 < len(toks) and toks[j] == "su" and toks[j + 1] in PARENTESCO and entre(A["b"], j).strip() == "," and x > 0:
                B = ms[x - 1]
                if B["pj"] != A["pj"] and A["pos"] - fin(B) <= 8: pistas[(A["pj"], PARENTESCO[toks[j + 1]], B["pj"])].append(i)
        for x, A in enumerate(ms):
            # B y su K A                «Úrsula y su hijo Aureliano»
            q = A["pos"]
            if q >= 2 and x > 0 and toks[q - 2] == "su" and toks[q - 1] in PARENTESCO and tx[sp[q - 1][1]:A["a"]].strip() == "":
                B = ms[x - 1]
                if B["pj"] != A["pj"] and q - fin(B) <= 6 and ";" not in tx[B["b"]:A["a"]] and "." not in tx[B["b"]:A["a"]]:
                    pistas[(A["pj"], PARENTESCO[toks[q - 1]], B["pj"])].append(i)
    return sorted(([a, k, b, sorted(set(os))] for (a, k, b), os in pistas.items()), key=lambda x: -len(x[3]))


# ----------------------- familia y árbol -----------------------
# Una línea por personaje:   Nombre: hijo de A y B; esposa de C; amante de D
# Tipos para el árbol: hijo/hija (de), padre/madre (de), esposo/esposa/marido/mujer, pareja/amante. Los demás (hermano, abuelo,
# tío, novio, pretendiente…) se muestran en la ficha, no en el árbol.
REL_FAM = {"hijo": ("hijo", "hijo"), "hija": ("hijo", "hija"), "hijos": ("hijo", "hijo"), "hijas": ("hijo", "hija"),
           "hijo adoptivo": ("adoptivo", "hijo adoptivo"), "hija adoptiva": ("adoptivo", "hija adoptiva"),
           "padre": ("padre", "padre"), "madre": ("padre", "madre"),
           "esposo": ("pareja", "esposo"), "esposa": ("pareja", "esposa"), "marido": ("pareja", "esposo"), "mujer": ("pareja", "esposa"),
           "pareja": ("amante", "pareja"), "amante": ("amante", "amante"), "concubina": ("amante", "pareja")}
REL_OTRA = set("hermano hermana medio hermano hermanastro hermanastra abuelo abuela nieto nieta bisabuelo bisabuela bisnieto bisnieta tatarabuelo "
               "tatarabuela tataranieto tataranieta tio tia tío tía sobrino sobrina primo prima cuñado cuñada suegro suegra yerno nuera padrino "
               "madrina ahijado ahijada novio novia prometido prometida pretendiente amigo amiga compadre comadre rival".split())
CLAVES_REL = {"hijo": "hijo hija hijos hijas padre madre papa mama parir pario nacio nacer nacimiento engendr* embaraz* criatura bautiz* nieto nieta".split(),
              "adoptivo": "hijo hija adopt* recog* crianza criar crio padres".split(),
              "pareja": "esposa esposo marido mujer casar* caso casaron boda matrimonio novia novio viuda viudo".split(),
              "amante": "amante amantes amor* concubin* querida cama novia novio mujer".split()}

def _buscar_pj(nombre, R):
    q = sin_tildes(nombre.strip().lower()); q = re.sub(r"\s+", " ", q)
    nombres = [sin_tildes(p["nombre"].lower()) for p in R]
    if q in nombres: return nombres.index(q)
    sin_par = [re.sub(r"\s*\(.*?\)", "", x).strip() for x in nombres]
    if sin_par.count(q) == 1: return sin_par.index(q)
    cand = [k for k, x in enumerate(nombres) if re.search(rf"(?<![\wñ]){re.escape(q)}(?![\wñ])", x)]
    if len(cand) == 1: return cand[0]
    cand = [k for k, p in enumerate(R) if any(" ".join(f["t"]) == q for f in p["formas"])]
    return cand[0] if len(cand) == 1 else None

def leer_familia(texto, R):
    F, sin = [], []
    for linea in texto.strip().splitlines():
        if ":" not in linea or linea.strip().startswith("#"): continue
        x, resto = linea.split(":", 1); a = _buscar_pj(x, R)
        if a is None: sin.append(x.strip()); continue
        for trozo in re.split(r"[;\n]", resto):
            m = re.match(r"\s*([a-záéíóúñ ]+?)\s+(?:de|del)\s+(.+)$", trozo.strip(), re.I)
            if not m: continue
            pal, nombres = sin_tildes(m.group(1).lower().strip()), m.group(2)
            pal = re.sub(r"^(el|la|los|las|su|sus)\s+", "", pal)
            tipo, etq = REL_FAM.get(pal, (None, pal)) if pal in REL_FAM else ("otro" if pal in REL_OTRA or pal.split()[-1] in REL_OTRA else "otro", pal)
            for nb in re.split(r"\s*,\s*|\s+y\s+|\s+e\s+", nombres.strip().rstrip(".")):
                nb = re.sub(r"^(?:de|del)\s+", "", nb.strip(), flags=re.I)
                if not nb.strip(): continue
                b = _buscar_pj(nb, R)
                if b is None: sin.append(nb.strip()); continue
                if tipo == "padre": F.append({"a": b, "b": a, "t": "hijo", "e": "hijo/a"})            # «madre de B» = B es hijo de A
                else: F.append({"a": a, "b": b, "t": tipo, "e": etq})
    vistos, out = set(), []
    for r in F:
        clave = (r["a"], r["b"], r["t"]) if r["t"] not in ("pareja", "amante") else (min(r["a"], r["b"]), max(r["a"], r["b"]), r["t"])
        if clave not in vistos and r["a"] != r["b"]: vistos.add(clave); out.append(r)
    return out, sorted(set(sin))

def familia_desde_pistas(pistas):
    """Las frases «X, hijo de Y» se convierten en vínculos propuestos (con su cita)."""
    F = []
    for a, k, b, os_ in pistas:
        k0 = sin_tildes(k)
        if k0 in ("hijo", "hija", "hijastro", "hijastra"): F.append({"a": a, "b": b, "t": "hijo", "e": k, "os": os_})
        elif k0 in ("padre", "madre"): F.append({"a": b, "b": a, "t": "hijo", "e": "hijo/a", "os": os_})
        elif k0 in ("esposa", "esposo"): F.append({"a": a, "b": b, "t": "pareja", "e": k, "os": os_})
        elif k0 in ("amante", "novia", "novio"): F.append({"a": a, "b": b, "t": "amante" if k0 == "amante" else "otro", "e": k, "os": os_})
        else: F.append({"a": a, "b": b, "t": "otro", "e": k, "os": os_})
    return F

def evidencias_familia(F, O, M, P):
    """Para cada vínculo busca frases que lo digan: los dos personajes nombrados en la misma oración junto a una palabra de parentesco."""
    por_o = collections.defaultdict(lambda: collections.defaultdict(list)); dentro = collections.defaultdict(set)
    for m in M:
        dentro[m["o"]].update(range(m["pos"], m["pos"] + m.get("n", 1)))       # la palabra de parentesco no puede ser el propio nombre («el padre»)
        if m["pj"] is not None: por_o[m["o"]][m["pj"]].append(m["pos"])
    por_pj = collections.defaultdict(set)
    for i, d in por_o.items():
        for k in d: por_pj[k].add(i)
    for r in F:
        if r.get("os"): r["c"], r["fuente"] = list(r["os"])[:3], "frase"; continue
        claves = CLAVES_REL.get(r["t"], [sin_tildes(w) for w in r["e"].split()][-1:])
        ambos = sorted(por_pj[r["a"]] & por_pj[r["b"]])
        puntos = []
        for i in ambos:
            toks = O[i]["toks"]
            pk = [j for j, t in enumerate(toks) if j not in dentro[i] and any(_coincide_tok(t, c) for c in claves)]
            if not pk: continue
            pa, pb = por_o[i][r["a"]], por_o[i][r["b"]]
            d = min(abs(x - y) for x in pa for y in pb)
            kd = min(abs(j - x) for j in pk for x in pa + pb)       # la palabra de parentesco, pegada a uno de los dos
            if kd > 4 or d > 25: continue
            puntos.append((kd + d / 5, i))
        puntos.sort()
        r["c"], r["fuente"] = [i for _, i in puntos[:3]], "frase" if puntos else "ninguna"
    return F

def texto_familia(F, R):
    lineas = collections.defaultdict(list)
    for r in F:
        if r["t"] == "hijo": lineas[r["a"]].append(f"{r['e'] if r['e'] in ('hijo', 'hija') else 'hijo'} de {R[r['b']]['nombre']}")
        else: lineas[r["a"]].append(f"{r['e']} de {R[r['b']]['nombre']}")
    return "\n".join(f"{R[a]['nombre']}: " + "; ".join(v) for a, v in lineas.items())

def disponer_arbol(n, F):
    """Generaciones por filas (padres arriba, hijos abajo); las parejas, lado a lado cuando son de la misma generación."""
    padres = collections.defaultdict(set); hijos = collections.defaultdict(set); parejas = collections.defaultdict(set)
    for r in F:
        if r["t"] in ("hijo", "adoptivo"): padres[r["a"]].add(r["b"]); hijos[r["b"]].add(r["a"])
        elif r["t"] in ("pareja", "amante"): parejas[r["a"]].add(r["b"]); parejas[r["b"]].add(r["a"])
    nodos = set(padres) | set(hijos) | set(parejas)
    if not nodos: return {}
    gen = {k: 0 for k in nodos}
    for _ in range(len(nodos) + 2):                     # hijos al menos una fila debajo de cada padre
        cambio = False
        for c, ps in padres.items():
            g = max(gen[p] for p in ps) + 1
            if gen[c] < g: gen[c] = g; cambio = True
        if not cambio: break
    for _ in range(6):                                  # quien no tiene padres en el árbol baja junto a su pareja o encima de sus hijos
        for k in nodos:
            if padres[k]: continue
            op = [gen[c] - 1 for c in hijos[k]] + [gen[q] for q in parejas[k] if padres[q] or hijos[q]]
            if op: gen[k] = max(gen[k], min(op))
    G = max(gen.values())
    x = {}
    for g in range(G + 1):
        fila = [k for k in nodos if gen[k] == g]
        def clave(k):
            ps = [x[p] for p in padres[k] if p in x]
            if ps: return sum(ps) / len(ps)
            qs = [x[q] for q in parejas[k] if q in x]
            for q in parejas[k]:
                qs += [x[p] for p in padres[q] if p in x]
            return sum(qs) / len(qs) if qs else 1e6 + k
        fila.sort(key=clave)
        orden = []
        for k in fila:                                  # pareja junto a pareja
            if k in orden: continue
            orden.append(k)
            for q in sorted(parejas[k], key=lambda q: -len(hijos[q] & hijos[k])):
                if q in fila and q not in orden: orden.append(q)
        deseo = [clave(k) if clave(k) < 1e6 else None for k in orden]
        pos, ult = [], -1e9
        for k, d in zip(orden, deseo):
            v = max(d if d is not None else ult + 1, ult + 1); pos.append(v); ult = v
        for k, v in zip(orden, pos): x[k] = v
    def separar(fila):
        fila.sort(key=lambda k: x[k])
        for i in range(1, len(fila)):
            if x[fila[i]] < x[fila[i - 1]] + 1: x[fila[i]] = x[fila[i - 1]] + 1
    for _ in range(4):                                  # padres sobre sus hijos (los que no tienen padres en el árbol, del todo)
        for g in range(G - 1, -1, -1):
            fila = [k for k in nodos if gen[k] == g]
            for k in fila:
                hs = [x[c] for c in hijos[k]]
                if hs: x[k] = sum(hs) / len(hs) if not padres[k] else (x[k] + sum(hs) / len(hs)) / 2
                elif not padres[k] and parejas[k]: x[k] = sum(x[q] for q in parejas[k]) / len(parejas[k]) + 0.5
            separar(fila)
        for g in range(1, G + 1):                       # y los hijos bajo sus padres
            fila = [k for k in nodos if gen[k] == g]
            for k in fila:
                ps = [x[p] for p in padres[k]]
                if ps: x[k] = (x[k] + sum(ps) / len(ps)) / 2
            separar(fila)
    for g in range(G + 1):                              # al final, cada pareja queda junta (bloques), en el orden deseado
        fila = sorted([k for k in nodos if gen[k] == g], key=lambda k: x[k]); usados, bloques = set(), []
        for k in fila:
            if k in usados: continue
            ps = [q for q in sorted(parejas[k], key=lambda q: x[q]) if gen[q] == g and q not in usados]
            izq = ps[:1] if len(ps) > 1 else []
            b = izq + [k] + [q for q in ps if q not in izq]
            usados.update(b); bloques.append(b)
        bloques.sort(key=lambda b: sum(x[q] for q in b) / len(b)); ult = -1e9
        for b in bloques:
            ini = max(sum(x[q] for q in b) / len(b) - (len(b) - 1) / 2, ult + 1)
            for i, q in enumerate(b): x[q] = ini + i
            ult = x[b[-1]]
    m0 = min(x.values())
    return {k: [gen[k], round(x[k] - m0, 2)] for k in nodos}

# ----------------------- cómo lo llama la obra -----------------------
DETERMINANTES = {"el", "la", "los", "las", "del", "al", "su", "sus", "un", "una"}
PREPOSICIONES = set("a ante bajo con contra de desde en entre hacia hasta para por segun sin sobre tras durante mediante".split())
_SP_CACHE = {}
def sp_(o):
    k = id(o)
    if k not in _SP_CACHE: _SP_CACHE[k] = _spans(o["texto"])
    return _SP_CACHE[k]
def epitetos(O, M, R, NOM, maximo=6):
    """Fórmulas con que el texto nombra a cada personaje: «el coronel», «el viejo Rosendo», «Nasha Suro, la curandera»."""
    malas = STOP | VERBOS_COMUNES | NOM | {"que", "quien", "cual", "cuyo", "cuya"}
    F = collections.defaultdict(lambda: collections.defaultdict(list))
    for m in M:
        k = m["pj"]
        if k is None: continue
        o = O[m["o"]]; toks, orig, tx = o["toks"], o["orig"], o["texto"]
        n_m = len(PAT_TOK.findall(tx[m["a"]:m["b"]])); j0, j1 = m["pos"], m["pos"] + n_m
        # palabras en minúscula al comienzo de la mención («coronel» en «coronel Aureliano Buendía») o justo antes («el viejo Rosendo»)
        lead = []
        for q in range(j0, j1):
            if orig[q][0].isupper(): break
            lead.append(toks[q])
        if lead and len(lead) < n_m:
            det = toks[j0 - 1] if j0 > 0 and toks[j0 - 1] in DETERMINANTES else ""
            F[k][(("el" if det in ("del", "al") else det) + " " + " ".join(lead)).strip()].append(m["o"])
        elif not lead and j0 >= 2 and not orig[j0 - 1][0].isupper() and toks[j0 - 1] not in malas and toks[j0 - 1] not in DETERMINANTES:
            q = j0 - 1; pal = [toks[q]]
            if q >= 2 and toks[q - 1] not in malas and toks[q - 1] not in DETERMINANTES and toks[q - 2] in ("el", "la", "su"): q -= 1; pal.insert(0, toks[q])
            det = toks[q - 1]
            if det in ("el", "la", "su", "los", "las") and (q < 2 or toks[q - 2] not in PREPOSICIONES) and tx[sp_(o)[q - 1][1]:m["a"]].count(",") == 0:
                F[k][f"{det} {' '.join(pal)}"].append(m["o"])
        # aposición: «Nasha Suro, la curandera,»
        resto = tx[m["b"]:]
        am = re.match(r"\s*,\s*((?:el|la|los|las|un|una|su)\s+[^,.;:!?»—\-()]{2,60}?)\s*[,.;:]", resto)
        if am:
            frase = am.group(1); ws = [sin_tildes(w.lower()) for w in PAT_TOK.findall(frase)]
            if 2 <= len(ws) <= 6 and ws[1] not in malas and not ws[1].endswith("ndo") and not any(w in ("que", "quien", "cual") for w in ws):
                F[k][" ".join(ws)].append(m["o"])
    # una fórmula «el X» cuenta si «el X» suele ir pegado a un nombre en la obra (descarta «al principio», «en el cuarto»)
    total_bi = collections.Counter()
    for o in O:
        t = o["toks"]
        for q in range(len(t) - 1):
            if t[q] in ("el", "la", "su", "los", "las"): total_bi[f"{t[q]} {t[q + 1]}"] += 1
    def es_titulo(f, n):
        ws = f.split()
        if len(ws) != 2 or ws[0] not in ("el", "la", "su", "los", "las"): return True
        return n >= 3 and (n >= 0.1 * max(1, total_bi[f]) or n >= 8)
    out = {}
    for k, d in F.items():
        d = {f: v for f, v in d.items() if es_titulo(f, len(v))}
        filas = [(f, sorted(set(os_))) for f, os_ in d.items() if len(set(os_)) >= 2]
        filas.sort(key=lambda x: -len(x[1]))
        out[k] = [{"f": f, "n": len(os_), "s0": O[os_[0]]["seg"], "s1": O[os_[-1]]["seg"],
                   "c": [os_[int(round(t))] for t in np.linspace(0, len(os_) - 1, min(3, len(os_)))]} for f, os_ in filas[:maximo]]
    return out

def personajes(O, obra, tramos, reparto_texto, compartidos="resolver por contexto", conf=None, familia_texto=None):
    conf = conf or {}
    NOM = tokens_nombre(O, conf.get("stop_extra", []))
    R = leer_reparto(reparto_texto, NOM)
    n = len(R)
    M = buscar_menciones(O, R, NOM)
    txt_m = lambda m: O[m["o"]]["texto"][m["a"]:m["b"]]
    amb_formas = collections.Counter(txt_m(m) for m in M if len(m["p"]) > 1)
    resol = resolver_ambiguas(M, O, compartidos)
    # hablantes del teatro → personaje
    voz_pj = {}
    if obra["teatro"]:
        for v in set(o["voz"] for o in O):
            ts = [t for t in tokenizar(v)[1] if not t.isdigit()]
            for k, p in enumerate(R):
                if any(f["t"] == ts or (len(f["t"]) == 1 and f["t"][0] in ts and len(ts) == 1) for f in p["formas"]): voz_pj[v] = k; break
    seg_tr = {s: t for t, ss in enumerate(tramos.values()) for s in ss}
    tok_ini, acc = [], 0
    for o in O: tok_ini.append(acc); acc += len(o["toks"])
    NT = acc or 1
    P = pasajes(O, obra)
    en = [set() for _ in P]; o_p = [0] * len(O)
    for u, (a, b) in enumerate(P):
        for i in range(a, b + 1): o_p[i] = u
    for m in M:
        if m["pj"] is not None: en[o_p[m["o"]]].add(m["pj"])
    parl = collections.Counter(); dichas = collections.Counter()
    if voz_pj:
        vistos = set()
        for i, o in enumerate(O):
            k = voz_pj.get(o["voz"])
            if k is None: continue
            en[o_p[i]].add(k); dichas[k] += len(o["toks"])
            if (o["par"], k) not in vistos: vistos.add((o["par"], k)); parl[k] += 1
    nA = collections.Counter(k for s in en for k in s)
    co = collections.Counter((a, b) for s in en for a in s for b in s if a < b)
    U = len(P)
    minJ = 2 if U < 3000 else 3
    E = [(a, b, c, c / math.sqrt(nA[a] * nA[b]), c * U / (nA[a] * nA[b])) for (a, b), c in co.items() if c >= minJ]
    E.sort(key=lambda e: -e[2]); E = E[:80]
    menc = collections.Counter(m["pj"] for m in M if m["pj"] is not None)
    pos_pj = collections.defaultdict(list)
    for m in M:
        if m["pj"] is not None: pos_pj[m["pj"]].append(m["g"])
    if voz_pj:
        for i, o in enumerate(O):
            k = voz_pj.get(o["voz"])
            if k is not None: pos_pj[k].append(tok_ini[i])
    for k in pos_pj: pos_pj[k].sort()
    activos = [k for k in range(n) if nA[k]]
    grupos = _louvain(n, [(a, b, f) for a, b, c, f, l in E])
    betw = _intermediacion(n, [(a, b, f) for a, b, c, f, l in E])
    grado = collections.Counter()
    for a, b, c, f, l in E: grado[a] += c; grado[b] += c
    pos = _disponer(n, [(a, b, f) for a, b, c, f, l in E], activos)
    T = len(tramos)
    largo_tr = collections.Counter(seg_tr[o["seg"]] for o in O for _ in o["toks"])
    def tr_de(g):
        i = max(0, np.searchsorted(tok_ini, g, side="right") - 1); return seg_tr[O[i]["seg"]], i
    peso = {k: menc[k] + parl[k] for k in range(n)}
    pmax, gmax = max(peso.values() or [1]) or 1, max(grado.values() or [1]) or 1
    score = {k: 0.5 * peso[k] / pmax + 0.5 * grado[k] / gmax for k in range(n)}
    smax = max(score.values() or [1]) or 1
    L = []
    for k in range(n):
        ps = pos_pj.get(k, [])
        trs = collections.Counter(tr_de(g)[0] for g in ps)
        cubre = len(trs) / T if T else 1
        conc = max(trs.values()) / len(ps) if ps else 0
        t_pico = max(trs, key=trs.get) if trs else 0
        ini_o = tr_de(ps[0])[1] if ps else 0; fin_o = tr_de(ps[-1])[1] if ps else 0
        tramo_grande = T and largo_tr[t_pico] / NT > 0.45
        if not ps: rol = "sin apariciones"
        elif score[k] >= 0.7 * smax and cubre >= 0.6: rol = "protagonista"
        elif score[k] >= 0.3 * smax and cubre >= 0.4: rol = "principal"
        elif (T > 1 and conc >= 0.7 and not tramo_grande) or (ps[-1] - ps[0]) / NT <= 0.25: rol = "episódico"
        else: rol = "secundario"
        L.append({"n": R[k]["nombre"], "formas": [" ".join(f["t"]) + (f" [{f['r'][0]}-{f['r'][1]}]" if f["r"] else "") for f in R[k]["formas"]],
                  "rol": rol, "puente": betw[k] >= 0.15 and nA[k] >= 5 and rol != "protagonista", "grupo": grupos[k],
                  "menc": menc[k], "parl": parl[k], "dichas": dichas[k], "pasajes": nA[k], "grado": grado[k], "betw": round(betw[k], 3),
                  "ini": ps[0] / NT if ps else None, "fin": ps[-1] / NT if ps else None, "o_ini": ini_o, "o_fin": fin_o,
                  "cubre": round(cubre, 2), "conc": round(conc, 2), "t_pico": t_pico,
                  "amb": sum(1 for m in M if m["pj"] == k and m.get("amb")), "pos": pos[k], "score": round(score[k] / smax, 3)})
    # roles protagonistas: como mucho 3
    prot = sorted([k for k in range(n) if L[k]["rol"] == "protagonista"], key=lambda k: -score[k])
    for k in prot[3:]: L[k]["rol"] = "principal"
    PJ = {"lista": L, "M": M, "P": P, "E": E, "voz_pj": voz_pj, "en": en, "U": U, "resol": resol,
          "amb_formas": amb_formas, "parentesco": pistas_parentesco(O, M, R), "reparto": R, "texto": reparto_texto.strip(),
          "teatro": obra["teatro"], "compartidos": compartidos, "NT": NT, "O_txt": txt_m,
          "lugar": lambda i: f"{ETQ_SEG[obra['modo']].lower()} {next(x['etiqueta'] for x in obra['segmentos'] if x['num'] == O[i]['seg'])}"}
    # familia: la que escribiste (o la del perfil); si no hay, la que sugieren las frases de parentesco
    if familia_texto and familia_texto.strip():
        F, sin_nombre = leer_familia(familia_texto, R); PJ["familia_origen"] = "escrita"
    else:
        F, sin_nombre = familia_desde_pistas(PJ["parentesco"]), []; PJ["familia_origen"] = "pistas"
    PJ["familia"], PJ["familia_sin_nombre"] = evidencias_familia(F, O, M, P), sin_nombre
    PJ["familia_texto"] = (familia_texto or "").strip() or texto_familia(PJ["familia"], R)
    PJ["arbol"] = disponer_arbol(n, PJ["familia"])
    PJ["epitetos"] = epitetos(O, M, R, NOM)
    PJ["ia"] = {}
    PJ["hallazgos"] = hallazgos_personajes(PJ, tramos)
    return PJ

def _rel(c, na, nb):
    s = c / max(1, min(na, nb))
    return "casi siempre juntos" if s >= 0.5 else "a menudo juntos" if s >= 0.25 else "a veces juntos"

def hallazgos_personajes(PJ, tramos):
    """Frases de entrada. ⟦k⟧ marca el nombre del personaje k (la lectura lo colorea)."""
    L, E, H = PJ["lista"], PJ["E"], []
    nom = lambda k: f"⟦{k}⟧"
    vivos = [k for k, p in enumerate(L) if p["menc"] + p["parl"] > 0]
    if not vivos: return H
    vec = collections.Counter()
    for a, b, *_ in E: vec[a] += 1; vec[b] += 1
    c0 = max(vivos, key=lambda k: (L[k]["score"], L[k]["menc"]))
    if vec[c0]: H.append({"etq": "El centro", "txt": f"Con {nom(c0)} coinciden {vec[c0]} de los {len(vivos) - 1} personajes restantes: la red gira a su alrededor.", "pj": [c0], "ir": {"focal": c0}})
    pu = [k for k in vivos if L[k]["puente"]]
    if pu:
        k = max(pu, key=lambda k: L[k]["betw"])
        H.append({"etq": "Hace de puente", "txt": f"Muchos caminos de la red pasan por {nom(k)}: une grupos de personajes que casi no se tocan entre sí.", "pj": [k], "ir": {"focal": k}})
    E3 = [e for e in E if e[2] >= 3 and c0 not in e[:2]] or [e for e in E if e[2] >= 3]
    if E3:
        a, b, c, f, l = max(E3, key=lambda e: (e[3], e[2]))
        H.append({"etq": "Pareja", "txt": f"{nom(a)} y {nom(b)} van {_rel(c, L[a]['pasajes'], L[b]['pasajes'])}: coinciden en {c} pasajes.", "pj": [a, b], "ir": {"arista": [a, b]}})
    tr = list(tramos)
    epi = [k for k in vivos if L[k]["rol"] == "episódico" and L[k]["menc"] + L[k]["parl"] >= 5]
    if epi and len(tr) > 1:
        k = max(epi, key=lambda k: L[k]["menc"] + L[k]["parl"])
        H.append({"etq": "De paso", "txt": f"Casi todas las apariciones de {nom(k)} están en un solo tramo: <i>{html.escape(tr[L[k]['t_pico']])}</i>.", "pj": [k], "ir": {"focal": k, "tramo": L[k]["t_pico"]}})
    # relevo: dos personajes importantes que casi no coinciden en el tiempo
    imp = sorted([k for k in vivos if L[k]["rol"] in ("protagonista", "principal")], key=lambda k: L[k]["ini"])
    rel = None
    for i, a in enumerate(imp):
        for b in imp[i + 1:]:
            if L[b]["ini"] >= L[a]["fin"] - 0.05 and (rel is None or L[b]["ini"] - L[a]["fin"] > rel[2]): rel = (a, b, L[b]["ini"] - L[a]["fin"])
    if rel:
        H.append({"etq": "Relevo", "txt": f"Cuando entra {nom(rel[1])}, {nom(rel[0])} ya casi no aparece: no comparten época.", "pj": [rel[0], rel[1]], "ir": {"focal": rel[1]}})
    if PJ["amb_formas"]:
        f, c = PJ["amb_formas"].most_common(1)[0]
        a_quien = collections.Counter(m["pj"] for m in PJ["M"] if m.get("amb") and PJ["O_txt"](m) == f)
        quien = [k for k, _ in a_quien.most_common(4)]
        if quien: H.append({"etq": "Nombre compartido", "txt": f"«{html.escape(f)}» aparece {c} veces solo, sin apellido: la herramienta lo repartió por contexto entre " + ", ".join(nom(k) for k in quien) + ". Revísalo.", "pj": quien, "ir": {"focal": quien[0]}})
    # dos épocas: el mismo nombre desaparece mucho tiempo y vuelve (¿otro personaje con el mismo nombre?)
    pos = collections.defaultdict(list)
    for m in PJ["M"]:
        if m["pj"] is not None: pos[m["pj"]].append((m["g"], m["o"]))
    hueco = None
    for k, ps in pos.items():
        ps.sort()
        if len(ps) < 8: continue
        for i in range(3, len(ps) - 2):
            g = (ps[i][0] - ps[i - 1][0]) / PJ["NT"]
            if g >= 0.25 and (hueco is None or g > hueco[1]): hueco = (k, g, ps[i - 1][1], ps[i][1])
    if hueco:
        k, g, o1, o2 = hueco
        H.append({"etq": "¿Dos personajes?", "txt": f"{nom(k)} desaparece durante el {round(100 * g)}% de la obra (desde {PJ['lugar'](o1)} hasta {PJ['lugar'](o2)}) y luego vuelve. "
                  "Si en la obra se repiten nombres, puede ser otro personaje: sepáralos en el reparto con un rango, p. ej. <code>aureliano [1-9]</code>.",
                  "pj": [k], "ir": {"focal": k}})
    # árbol: el nombre que se hereda y la pareja que cruza generaciones
    A = PJ.get("arbol") or {}
    if A:
        por_nombre, vista_w = collections.defaultdict(set), {}
        for k in A:
            for w0 in re.findall(r"[A-Za-zÁÉÍÓÚáéíóúÑñÜü]+", L[k]["n"]):
                w = sin_tildes(w0.lower()); vista_w.setdefault(w, w0)
                if w not in STOP and len(w) > 3 and w not in ("coronel", "padre", "general", "hijo", "bella", "segundo", "seminarista", "renata", "diecisiete"): por_nombre[w].add(k)
        w, ks = max(por_nombre.items(), key=lambda x: (len({A[k][0] for k in x[1]}), len(x[1])), default=(None, set()))
        gens = sorted({A[k][0] for k in ks})
        if w and len(gens) >= 3:
            ks = sorted(ks, key=lambda k: A[k][0])
            H.append({"etq": "Un nombre que se hereda", "txt": f"«{html.escape(vista_w[w])}» se repite en {len(ks)} personajes de {len(gens)} generaciones: " + ", ".join(nom(k) for k in ks) + ". El árbol los separa.",
                      "pj": ks[:6], "ir": {"vista": "arbol", "focal": ks[0]}})
        for r in PJ["familia"]:
            if r["t"] in ("pareja", "amante") and r["a"] in A and r["b"] in A and A[r["a"]][0] != A[r["b"]][0]:
                H.append({"etq": "Pareja entre generaciones", "txt": f"{nom(r['a'])} y {nom(r['b'])} son pareja ({html.escape(r['e'])}) aunque pertenecen a generaciones distintas del árbol.",
                          "pj": [r["a"], r["b"]], "ir": {"vista": "arbol", "arista": [r["a"], r["b"]]}}); break
    tard = [k for k in vivos if L[k]["ini"] >= 0.3 and L[k]["rol"] in ("protagonista", "principal", "secundario")]
    if tard:
        k = max(tard, key=lambda k: L[k]["score"])
        H.append({"etq": "Llega tarde", "txt": f"La primera mención de {nom(k)} llega pasado el {round(100 * L[k]['ini'])}% de la obra, y aun así pesa en la red.", "pj": [k], "ir": {"focal": k}})
    return H[:9]

# ----------------------- presentación en Colab -----------------------
COLORES_PJ = ["#1F5FA8", "#E07B24", "#2E8B6E", "#8E5BC2", "#C0392B", "#3D7A8C", "#B5838D", "#7A8B3A", "#A26A42", "#5C6BC0"]
ROLES_ORDEN = ["protagonista", "principal", "secundario", "episódico", "sin apariciones"]

def _svg_red_pj(PJ, ancho=760, alto=500):
    L, E = PJ["lista"], PJ["E"]
    col = lambda k: COLORES_PJ[L[k]["grupo"] % len(COLORES_PJ)] if L[k]["grupo"] >= 0 else "#9AA0A8"
    P = lambda k: (40 + (ancho - 80) * L[k]["pos"][0], 34 + (alto - 60) * L[k]["pos"][1]) if L[k]["pos"] else None
    cmax = max([e[2] for e in E] + [1]); pmax = max([p["menc"] + p["parl"] for p in L] + [1])
    s = [f'<svg viewBox="0 0 {ancho} {alto}" width="100%" style="max-width:{ancho}px;background:#fbfaf7;border:1px solid #e1e4ea;border-radius:10px" xmlns="http://www.w3.org/2000/svg">']
    for a, b, c, f, l in E:
        pa, pb = P(a), P(b)
        if pa and pb: s.append(f'<line x1="{pa[0]:.0f}" y1="{pa[1]:.0f}" x2="{pb[0]:.0f}" y2="{pb[1]:.0f}" stroke="{col(a) if L[a]["grupo"] == L[b]["grupo"] else "#9AA0A8"}" stroke-opacity=".5" stroke-width="{1 + 8 * c / cmax:.1f}" stroke-linecap="round"/>')
    for k in sorted(range(len(L)), key=lambda k: L[k]["menc"]):
        p = P(k)
        if not p: continue
        r = 5 + 20 * math.sqrt((L[k]["menc"] + L[k]["parl"]) / pmax)
        anillo = ' stroke="#1f2430" stroke-width="3"' if L[k]["rol"] == "protagonista" else ' stroke="#fff" stroke-width="1.5"'
        s.append(f'<circle cx="{p[0]:.0f}" cy="{p[1]:.0f}" r="{r:.1f}" fill="{col(k)}"{anillo}/>')
        s.append(f'<text x="{p[0]:.0f}" y="{p[1] - r - 5:.0f}" text-anchor="middle" font-family="system-ui,sans-serif" font-size="12.5" '
                 f'paint-order="stroke" stroke="#fbfaf7" stroke-width="4" fill="#1f2430">{html.escape(L[k]["n"])}</text>')
    s.append("</svg>")
    return "".join(s)

def _svg_presencia(PJ, O, tramos, ancho=760):
    L = PJ["lista"]; ks = sorted([k for k in range(len(L)) if L[k]["ini"] is not None], key=lambda k: L[k]["ini"])
    fila, izq = 17, 150; W = ancho - izq; H = len(ks) * fila + 18
    col = lambda k: COLORES_PJ[L[k]["grupo"] % len(COLORES_PJ)] if L[k]["grupo"] >= 0 else "#9AA0A8"
    tok_ini, acc = [], 0
    for o in O: tok_ini.append(acc); acc += len(o["toks"])
    seg_tr = {s: t for t, ss in enumerate(tramos.values()) for s in ss}
    cortes = [tok_ini[i] / acc for i in range(1, len(O)) if seg_tr[O[i]["seg"]] != seg_tr[O[i - 1]["seg"]]]
    s = [f'<svg viewBox="0 0 {ancho} {H}" width="100%" style="max-width:{ancho}px" xmlns="http://www.w3.org/2000/svg" font-family="system-ui,sans-serif" font-size="11.5">']
    for x in cortes: s.append(f'<line x1="{izq + W * x:.1f}" x2="{izq + W * x:.1f}" y1="0" y2="{H - 16}" stroke="#d9dde4" stroke-width="1.5"/>')
    pos = collections.defaultdict(list)
    for m in PJ["M"]:
        if m["pj"] is not None: pos[m["pj"]].append(m["g"])
    for r, k in enumerate(ks):
        y = r * fila
        s.append(f'<text x="{izq - 8}" y="{y + 12}" text-anchor="end" fill="#1f2430">{html.escape(L[k]["n"][:22])}</text>')
        s.append(f'<rect x="{izq + W * L[k]["ini"]:.1f}" y="{y + 7}" width="{max(1, W * (L[k]["fin"] - L[k]["ini"])):.1f}" height="2" fill="{col(k)}" opacity=".35"/>')
        for g in pos[k][::max(1, len(pos[k]) // 400)]: s.append(f'<rect x="{izq + W * g / acc:.1f}" y="{y + 3}" width="1.4" height="10" fill="{col(k)}"/>')
    s.append(f'<text x="{izq}" y="{H - 3}" fill="#6b6f78">inicio</text><text x="{ancho}" y="{H - 3}" fill="#6b6f78" text-anchor="end">final</text></svg>')
    return "".join(s)


def _svg_arbol(PJ, ancho_u=None, alto_f=84):
    A, L, F = PJ.get("arbol") or {}, PJ["lista"], PJ.get("familia", [])
    if not A: return ""
    col = lambda k: COLORES_PJ[L[k]["grupo"] % len(COLORES_PJ)] if L[k]["grupo"] >= 0 else "#6b6f78"
    ancho_u = ancho_u or max(118, min(23, max(len(L[k]["n"]) for k in A)) * 6.6 + 44)
    X = lambda k: 20 + ancho_u / 2 + A[k][1] * ancho_u; Y = lambda k: 30 + A[k][0] * alto_f
    W = int(max(X(k) for k in A) + 80); H = int(max(Y(k) for k in A) + 40)
    s = [f'<svg viewBox="0 0 {W} {H}" width="100%" style="max-width:{W}px;background:#fbfaf7;border:1px solid #e1e4ea;border-radius:10px" xmlns="http://www.w3.org/2000/svg" font-family="system-ui,sans-serif" font-size="12">']
    grupos = collections.defaultdict(list)
    for r in F:
        if r["t"] in ("hijo", "adoptivo") and r["a"] in A and r["b"] in A: grupos[r["a"]].append(r)
    hijos_de = collections.defaultdict(list)
    for c, rs in grupos.items(): hijos_de[tuple(sorted(r["b"] for r in rs))].append((c, any(r["t"] == "adoptivo" for r in rs), all(not r.get("c") for r in rs)))
    for r in F:
        if r["t"] in ("pareja", "amante") and r["a"] in A and r["b"] in A:
            x1, y1, x2, y2 = X(r["a"]), Y(r["a"]), X(r["b"]), Y(r["b"])
            dash = ' stroke-dasharray="5 4"' if r["t"] == "amante" else ""
            if y1 == y2 and abs(A[r["a"]][1] - A[r["b"]][1]) <= 1.05: s.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="#E07B24" stroke-width="2.2"{dash}/>')
            elif y1 == y2: s.append(f'<path d="M{x1},{y1 - 12} C{x1},{y1 - 40} {x2},{y2 - 40} {x2},{y2 - 12}" fill="none" stroke="#E07B24" stroke-width="2.2"{dash}/>')
            else: s.append(f'<path d="M{x1},{y1} C{x1},{(y1 + y2) / 2} {x2},{(y1 + y2) / 2} {x2},{y2}" fill="none" stroke="#E07B24" stroke-width="2.2"{dash}/>')
    for ps, cs in hijos_de.items():
        ax = sum(X(p) for p in ps) / len(ps); jy = max(Y(p) for p in ps) + 22
        by = min(Y(c) for c, *_ in cs) - 20
        xs = [X(c) for c, *_ in cs] + [ax]
        d = "".join(f"M{X(p)},{Y(p) + 12} V{jy} " for p in ps) + (f"M{min(X(p) for p in ps)},{jy} H{max(X(p) for p in ps)} " if len(ps) > 1 else "")
        s.append(f'<path d="{d}M{ax},{jy} V{by} M{min(xs)},{by} H{max(xs)}" fill="none" stroke="#9AA0A8" stroke-width="1.5"/>')
        for c, adop, sin in cs:
            punteo = ' stroke-dasharray="3 3"' if adop else ""
            s.append(f'<line x1="{X(c)}" y1="{by}" x2="{X(c)}" y2="{Y(c) - 12}" stroke="#9AA0A8" stroke-width="1.5"{punteo}/>')
    for k in A:
        nombre = html.escape(L[k]["n"] if len(L[k]["n"]) <= 22 else L[k]["n"][:21] + "…"); w = max(60, len(nombre) * 6.6 + 14)
        s.append(f'<rect x="{X(k) - w / 2:.0f}" y="{Y(k) - 12}" width="{w:.0f}" height="24" rx="12" fill="#fff" stroke="{col(k)}" stroke-width="{2.5 if L[k]["rol"] == "protagonista" else 1.5}"/>'
                 f'<text x="{X(k):.0f}" y="{Y(k) + 4}" text-anchor="middle" fill="#1f2430">{nombre}</text>')
    s.append("</svg>")
    return "".join(s)


def _informe_familia(PJ, O, nom):
    L, e, F = PJ["lista"], html.escape, PJ.get("familia", [])
    partes = []
    if F:
        origen = ("escrita por ti o por el perfil" if PJ.get("familia_origen") == "escrita"
                  else "propuesta a partir de frases del tipo «X, hijo de Y» (cópiala al paso 3b para corregirla o completarla)")
        filas = "".join(f'<li>{nom(r["a"])} — {e(r["e"])} de — {nom(r["b"])} '
                        + (f'<span style="color:#2E8B6E">✔ cita</span> <span style="color:#888">«{e(O[r["c"][0]]["texto"][:120])}…»</span>' if r.get("c")
                           else '<span style="color:#888">· sin frase que lo diga explícitamente</span>') + '</li>' for r in F)
        partes.append(f'<h4 style="margin:14px 0 4px">Árbol de familia</h4><div style="color:#6b6f78">Familia {origen}. Línea naranja: pareja (punteada si no es matrimonio); gris: padres e hijos.</div>'
                      f'{_svg_arbol(PJ)}<details><summary style="cursor:pointer"><b>Vínculos y sus citas</b> ({sum(1 for r in F if r.get("c"))} de {len(F)} con una frase del texto)</summary><ul style="margin:4px 0">{filas}</ul></details>')
        if PJ.get("familia_sin_nombre"): partes.append(f'<div style="color:#b03a2e">⚠ Nombres de la familia que no están en el reparto: {e(", ".join(PJ["familia_sin_nombre"]))}</div>')
    else:
        partes.append('<div style="color:#6b6f78;margin-top:10px">Sin árbol de familia: el texto no trae frases como «X, hijo de Y». Puedes escribirla en el paso 3b.</div>')
    EP = PJ.get("epitetos") or {}
    con = [k for k in sorted(EP, key=lambda k: -L[k]["score"]) if EP[k]]
    if con:
        partes.append('<h4 style="margin:14px 0 4px">Cómo los llama la obra</h4><ul style="margin:4px 0">' + "".join(
            f'<li>{nom(k)}: ' + ", ".join(f'«{e(x["f"])}» <span style="color:#888">×{x["n"]}</span>' for x in EP[k][:4]) + "</li>" for k in con[:12]) + "</ul>")
    return "".join(partes)

def _nombres(txt, nom): return re.sub("⟦(\\d+)⟧", lambda m: nom(int(m.group(1))), txt)
def informe_personajes(PJ, O, tramos, info=None, origen=""):
    L, e = PJ["lista"], html.escape
    nom = lambda k: f'<b style="color:{COLORES_PJ[L[k]["grupo"] % len(COLORES_PJ)] if L[k]["grupo"] >= 0 else "#666"}">{e(L[k]["n"])}</b>'
    hall = "".join(f'<div style="border:1px solid #e1e4ea;border-radius:10px;padding:8px 12px;background:#fff"><div style="font-size:11px;letter-spacing:.08em;'
                   f'text-transform:uppercase;color:#6b6f78;font-weight:600">{h["etq"]}</div>'
                   f'{_nombres(h["txt"], nom)}</div>' for h in PJ["hallazgos"])
    por_rol = "".join(f'<div style="margin:3px 0"><b>{r.capitalize()}:</b> ' + ", ".join(
        f'{nom(k)}{" ⇄" if L[k]["puente"] else ""} <span style="color:#888">{L[k]["menc"] + L[k]["parl"]}</span>'
        for k in sorted(range(len(L)), key=lambda k: -L[k]["score"]) if L[k]["rol"] == r) + "</div>" for r in ROLES_ORDEN if any(p["rol"] == r for p in L))
    rels = sorted(PJ["E"], key=lambda x: -x[2])[:10]
    rel_html = "".join(f'<li>{nom(a)} y {nom(b)}: {_rel(c, L[a]["pasajes"], L[b]["pasajes"])} ({c} pasajes)</li>' for a, b, c, f, l in rels)
    par = "".join(f'<li>{nom(a)} — {e(k)} de — {nom(b)} <span style="color:#888">({len(os)} frase{"s" if len(os) > 1 else ""}: «{e(O[os[0]]["texto"][:110])}…»)</span></li>'
                  for a, k, b, os in PJ["parentesco"][:12])
    avisos = []
    if PJ["resol"]["resueltas"] or PJ["resol"]["sin_resolver"]:
        avisos.append(f'Nombres compartidos: {PJ["resol"]["resueltas"]} menciones repartidas por contexto'
                      + (f', {PJ["resol"]["sin_resolver"]} sin resolver (no cuentan)' if PJ["resol"]["sin_resolver"] else "")
                      + " · " + e(", ".join(f"«{f}» ({c})" for f, c in PJ["amb_formas"].most_common(6))))
    if info:
        if info.get("distintos"): avisos.append("Se separaron porque aparecen juntos en una misma frase: " + e("; ".join(f"{a} ≠ {b}" for a, b in info["distintos"][:8])))
        if info.get("familias"): avisos.append("Apellidos de familia (no se cuentan como un personaje): " + e(", ".join(info["familias"][:8])))
        if info.get("lugares"): avisos.append("Parecen lugares y quedaron fuera: " + e(", ".join(info["lugares"])))
        if info.get("teatro_papeles"): avisos.append("En teatro, un papel que se nombra en el texto (el Rey, la Reina, el Príncipe) suele ser el mismo personaje que un hablante: únelos en una sola línea del reparto (por ejemplo «Claudio: claudio, rey»).")
        if info.get("papeles"): avisos.append("Papeles sin nombre que se repiten (puedes sumarlos al reparto): " + e(", ".join(info["papeles"][:10])))
    vacios = [p["n"] for p in L if p["rol"] == "sin apariciones"]
    if vacios: avisos.append("Sin apariciones (revisa sus formas): " + e(", ".join(vacios)))
    unidad = "pasaje (parlamentos seguidos, ≈120 palabras)" if PJ["teatro"] else "pasaje (párrafos seguidos, ≈120 palabras)"
    return (f'<div style="font-family:system-ui,sans-serif;font-size:14px;line-height:1.5;max-width:900px;color:#1f2430">'
            f'<h3 style="margin:4px 0">Red de personajes · {sum(1 for p in L if p["rol"] != "sin apariciones")} personajes</h3>'
            + (f'<div style="color:#6b6f78">Reparto: {e(origen)}</div>' if origen else "") +
            f'<div style="color:#6b6f78">Dos personajes se unen cuando aparecen en el mismo {unidad}. Tamaño = cuántas veces se le nombra'
            f'{" o habla" if PJ["teatro"] else ""}; color = círculo de personajes que suelen ir juntos; borde oscuro = protagonista; ⇄ = puente.</div>'
            f'<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(250px,1fr));gap:8px;margin:10px 0">{hall}</div>'
            f'{_svg_red_pj(PJ)}<div style="margin:10px 0">{por_rol}</div>'
            f'<h4 style="margin:12px 0 4px">Quién está cuándo</h4>{_svg_presencia(PJ, O, tramos)}'
            + _informe_familia(PJ, O, nom) +
            f'<details open><summary style="cursor:pointer"><b>Relaciones más frecuentes</b></summary><ul style="margin:4px 0">{rel_html or "<li>—</li>"}</ul></details>'
            + "".join(f'<div style="background:#eef2f8;border-radius:8px;padding:6px 10px;margin:6px 0;font-size:13px">⚠ {a}</div>' for a in avisos)
            + f'<details><summary style="cursor:pointer"><b>El reparto usado</b> (cópialo al paso 3b para corregirlo)</summary>'
            f'<pre style="background:#f4f5f7;padding:8px 10px;border-radius:8px;white-space:pre-wrap;font-size:12.5px">{e(PJ["texto"])}</pre>'
            + (f'<b>La familia usada</b><pre style="background:#f4f5f7;padding:8px 10px;border-radius:8px;white-space:pre-wrap;font-size:12.5px">{e(PJ["familia_texto"])}</pre>' if PJ.get("familia_texto") else "")
            + '</details></div>')

def datos_personajes(PJ, O):
    """Lo que la lectura necesita para la pestaña Personajes."""
    L = PJ["lista"]
    return {"lista": [{"n": p["n"], "f": p["formas"], "rol": p["rol"], "pu": p["puente"], "g": p["grupo"], "m": p["menc"], "h": p["parl"],
                       "d": p["dichas"], "u": p["pasajes"], "gr": p["grado"], "b": p["betw"], "s": p["score"], "amb": p["amb"],
                       "xy": [round(p["pos"][0], 4), round(p["pos"][1], 4)] if p["pos"] else None} for p in L],
            "M": [[m["o"], m["a"], m["b"], m["pj"], 1 if m.get("amb") else 0] for m in PJ["M"] if m["pj"] is not None],
            "P": [list(p) for p in PJ["P"]],
            "voz": {str(i): PJ["voz_pj"][v] for i, v in enumerate(dict.fromkeys(o["voz"] for o in O)) if v in PJ["voz_pj"]},
            "par": [[a, k, b, os[:6]] for a, k, b, os in PJ["parentesco"][:40]],
            "hall": PJ["hallazgos"],
            "amb": {"res": PJ["resol"]["resueltas"], "sin": PJ["resol"]["sin_resolver"], "formas": PJ["amb_formas"].most_common(8)},
            "teatro": PJ["teatro"], "minJ": 2 if PJ["U"] < 3000 else 3, "reparto": PJ["texto"],
            "fam": [{"a": r["a"], "b": r["b"], "t": r["t"], "e": r["e"], "c": r.get("c", [])[:3]} for r in PJ.get("familia", [])],
            "fam_origen": PJ.get("familia_origen", ""), "fam_texto": PJ.get("familia_texto", ""),
            "arbol": {str(k): v for k, v in (PJ.get("arbol") or {}).items()},
            "epi": {str(k): [[e["f"], e["n"], e["s0"] - 1, e["s1"] - 1, e["c"]] for e in v] for k, v in (PJ.get("epitetos") or {}).items() if v},
            "ia": {str(k): v for k, v in (PJ.get("ia") or {}).items()}}

# ----------------------- IA: rol literal y simbólico de un personaje -----------------------
def ia_rol_personaje(PJ, O, nombre, motivos, titulo, k=14):
    L, R = PJ["lista"], PJ["reparto"]
    j = _buscar_pj(nombre, R) if nombre.strip() else None
    if j is None: print("Escribe un personaje del reparto:", ", ".join(p["n"] for p in L if p["menc"] + p["parl"])); return None
    idx = []
    for e in (PJ.get("epitetos") or {}).get(j, [])[:3]: idx += e["c"][:1]
    for r in PJ.get("familia", []):
        if j in (r["a"], r["b"]) and r.get("c"): idx += r["c"][:1]
    ms = sorted({m["o"] for m in PJ["M"] if m["pj"] == j})
    if ms: idx += [ms[int(round(t))] for t in np.linspace(0, len(ms) - 1, min(k, len(ms)))]
    idx = sorted(set(idx))[:k + 6]
    if not idx: print(f"«{L[j]['n']}» no tiene menciones en el texto."); return None
    # motivos que van con el personaje (en sus pasajes)
    con = collections.Counter()
    for u, (a, b) in enumerate(PJ["P"]):
        if j in PJ["en"][u]:
            toks = {t for i in range(a, b + 1) for t in O[i]["toks"]}
            for m in motivos:
                if any(coincide(t, m["formas"]) for t in toks): con[m["nombre"]] += 1
    fam = [f"{L[r['a']]['n']} — {r['e']} de — {L[r['b']]['n']}" for r in PJ.get("familia", []) if j in (r["a"], r["b"])]
    epi = [f"«{e['f']}» ({e['n']} veces)" for e in (PJ.get("epitetos") or {}).get(j, [])]
    frags = "\n".join(f"[{n + 1}] ({lugar(O[i])}) {O[i]['texto'][:320]}" for n, i in enumerate(idx))
    r = preguntar(sistema_ia(titulo), f"Personaje: «{L[j]['n']}» (rol en la red: {L[j]['rol']}; nombrado {L[j]['menc']} veces).\n"
        f"Cómo lo nombra la obra: {', '.join(epi) or 'sin fórmulas repetidas'}.\nFamilia y vínculos: {'; '.join(fam) or 'no registrados'}.\n"
        f"Motivos que aparecen en sus pasajes: {', '.join(f'{m} ({c})' for m, c in con.most_common(8)) or '—'}.\n"
        f"Fragmentos donde aparece, en orden:\n{frags}\n\n"
        'Describe su papel. JSON: {"literal": str (2-4 frases: qué hace en la trama, cita fragmentos como [n]), '
        '"simbolico": str (2-4 frases: qué representa o encarna, apoyado en fragmentos [n] y motivos), '
        '"cambia": str (si su papel cambia a lo largo de la obra, con [n], o "no se ve en los fragmentos"), '
        '"fuera_de_los_fragmentos": str (lo que dices por conocimiento previo de la obra, o "nada")}')
    usados = sorted({int(x) for campo in ("literal", "simbolico", "cambia") for x in re.findall(r"\[(\d+)\]", str(r.get(campo, "")))})
    malas = [n for n in usados if not 1 <= n <= len(idx)]
    citas = [[n, idx[n - 1]] for n in usados if 1 <= n <= len(idx)]
    PJ.setdefault("ia", {})[j] = {"literal": str(r.get("literal", "")), "simbolico": str(r.get("simbolico", "")), "cambia": str(r.get("cambia", "")),
                                 "fuera": str(r.get("fuera_de_los_fragmentos", "")), "citas": citas, "modelo": MODELO_IA}
    e = lambda x: html.escape(str(x))
    citado = "".join(f"<li><b>[{n}]</b> <small>{e(lugar(O[i]))}</small> — {e(O[i]['texto'])}</li>" for n, i in citas)
    display(HTML(f'<div style="font-family:system-ui;font-size:14px;line-height:1.5;max-width:880px"><h3>{e(L[j]["n"])}</h3>'
        f'<p><b>En la trama:</b> {e(r.get("literal", ""))}</p><p><b>Simbólicamente:</b> {e(r.get("simbolico", ""))}</p>'
        f'<p><b>¿Cambia?</b> {e(r.get("cambia", ""))}</p>'
        f'<p style="background:#fff3d6;padding:6px 10px;border-radius:6px"><b>Fuera de los fragmentos (no verificado):</b> {e(r.get("fuera_de_los_fragmentos", ""))}</p>'
        + (f'<p style="color:#b03a2e">⚠ Citó fragmentos inexistentes: {malas}</p>' if malas else "")
        + f'<b>Frases que cita</b><ul>{citado or "<li><i>Ninguna: la lectura no está anclada en el texto.</i></li>"}</ul></div>'))
    IA_REGISTRO.append({"tarea": "rol_personaje", "personaje": L[j]["n"], "fragmentos": [O[i]["texto"] for i in idx], "respuesta": r})
    return j
print("Motor de personajes listo.")

In [ ]:
#@title 🎨 Diseño de la lectura en red (ejecútala una vez; no necesitas abrirla)
PLANTILLA = r"""<div id="lectura-red" class="lr">
<style>
.lr{--bg:#f7f4ee;--card:#fffdf8;--ink:#1f2430;--sub:#6b6f78;--line:#e3ddd0;--soft:#efe9dd;--acc:#b5523b;--mA:#f4cfa3;--mB:#c7d5f0;
  --serif:"Iowan Old Style","Palatino Linotype",Palatino,"Book Antiqua",Georgia,serif;
  font-family:system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;color:var(--ink);background:var(--bg);
  padding:22px 22px 16px;max-width:1240px;margin:0 auto;box-sizing:border-box;font-size:14.5px;line-height:1.5}
@media (prefers-color-scheme:dark){.lr{--bg:#16181d;--card:#1e2128;--ink:#ece8e1;--sub:#a2a6ae;--line:#343844;--soft:#262a33;--acc:#e08a6d;--mA:#6e4824;--mB:#2d3f66}}
.lr *{box-sizing:border-box}
.lr button,.lr select,.lr input,.lr textarea{font:inherit;color:var(--ink)}
.lr .cab{display:flex;justify-content:space-between;align-items:flex-end;gap:16px;flex-wrap:wrap;border-bottom:1px solid var(--line);padding-bottom:12px}
.lr h1{font-family:var(--serif);font-weight:600;font-size:30px;margin:0;line-height:1.15}
.lr .autor{font-family:var(--serif);font-style:italic;color:var(--sub);font-size:16px}
.lr .datos{color:var(--sub);font-size:13px;margin-top:4px}
.lr .interruptor{display:flex;align-items:center;gap:8px;font-size:13px;color:var(--sub);cursor:pointer;user-select:none}
.lr .interruptor i{width:34px;height:19px;border-radius:10px;background:var(--line);position:relative;transition:.2s}
.lr .interruptor i:after{content:"";position:absolute;top:2px;left:2px;width:15px;height:15px;border-radius:50%;background:var(--card);transition:.2s}
.lr .interruptor.on i{background:var(--acc)} .lr .interruptor.on i:after{left:17px}
.lr nav{display:flex;gap:4px;margin:14px 0 18px;flex-wrap:wrap}
.lr nav button{border:0;background:transparent;padding:7px 14px;border-radius:20px;cursor:pointer;color:var(--sub);font-weight:500}
.lr nav button.on{background:var(--ink);color:var(--bg)}
.lr nav button .n{font-size:11px;background:var(--acc);color:#fff;border-radius:9px;padding:0 6px;margin-left:4px}
.lr .vista{display:none} .lr .vista.on{display:block}
.lr .intro{font-family:var(--serif);font-size:17px;max-width:760px;margin:0 0 16px;color:var(--ink)}
.lr .grilla{display:grid;grid-template-columns:repeat(auto-fill,minmax(280px,1fr));gap:14px}
.lr .tarjeta{background:var(--card);border:1px solid var(--line);border-radius:12px;padding:14px 16px;display:flex;flex-direction:column;gap:8px}
.lr .etq{font-size:11px;letter-spacing:.09em;text-transform:uppercase;color:var(--sub);font-weight:600}
.lr .frase{font-family:var(--serif);font-size:17px;line-height:1.4}
.lr .mot{font-weight:600;white-space:nowrap}
.lr .btn{border:1px solid var(--line);background:var(--card);border-radius:8px;padding:5px 11px;cursor:pointer;font-size:13px}
.lr .btn:hover{border-color:var(--sub)} .lr .btn.pri{background:var(--ink);color:var(--bg);border-color:var(--ink)}
.lr .btn.link{border:0;background:none;padding:0;color:var(--acc);font-weight:600;text-align:left}
.lr .tarjeta .btn.link{margin-top:auto}
.lr .tira{width:100%;height:auto;display:block}
.lr .redgrid{display:grid;grid-template-columns:minmax(0,1.6fr) minmax(300px,1fr);gap:16px}
@media (max-width:860px){.lr .redgrid{grid-template-columns:1fr}}
.lr .ctl{display:flex;flex-wrap:wrap;gap:12px 18px;align-items:center;margin-bottom:10px;font-size:13.5px}
.lr .ctl label{display:flex;align-items:center;gap:7px}
.lr select,.lr input[type=text],.lr textarea{background:var(--card);border:1px solid var(--line);border-radius:7px;padding:5px 8px}
.lr .tramo{display:flex;align-items:center;gap:8px;flex:1 1 320px}
.lr .tramo input{flex:1} .lr .tramo b{font-family:var(--serif);font-weight:600;min-width:150px}
.lr .lienzo{background:var(--card);border:1px solid var(--line);border-radius:12px;overflow:hidden}
.lr svg.red{width:100%;height:auto;display:block}
.lr .red text{font-size:13px;paint-order:stroke;stroke:var(--card);stroke-width:4px;stroke-linejoin:round;fill:var(--ink);pointer-events:none}
.lr .red .n{cursor:pointer} .lr .red line{cursor:pointer}
.lr .panel{background:var(--card);border:1px solid var(--line);border-radius:12px;padding:14px 16px;max-height:680px;overflow:auto}
.lr .panel h3{font-family:var(--serif);font-size:22px;margin:0 0 2px;font-weight:600}
.lr .sub{color:var(--sub);font-size:13px}
.lr .tit{font-size:11px;letter-spacing:.08em;text-transform:uppercase;color:var(--sub);font-weight:600;margin:14px 0 6px}
.lr .vec{display:flex;justify-content:space-between;gap:10px;padding:4px 6px;border-radius:6px;cursor:pointer}
.lr .vec:hover{background:var(--soft)} .lr .vec > span:last-child{color:var(--sub);font-size:12.5px;text-align:right}
.lr .cita{margin:8px 0;padding:2px 0 2px 12px;border-left:3px solid var(--line);font-family:var(--serif);font-size:15px;line-height:1.45}
.lr .cita .lug{display:block;font-family:system-ui,sans-serif;font-size:11.5px;color:var(--sub);font-weight:600;letter-spacing:.02em}
.lr .cita .mas{font-family:system-ui,sans-serif;font-size:11.5px;border:0;background:none;color:var(--acc);cursor:pointer;padding:0;margin-left:6px}
.lr mark.a{background:var(--mA);color:inherit;border-radius:3px;padding:0 2px} .lr mark.b{background:var(--mB);color:inherit;border-radius:3px;padding:0 2px}
.lr .ley{display:flex;flex-wrap:wrap;gap:6px 14px;font-size:12.5px;color:var(--sub);margin-top:8px}
.lr .ley i{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:5px;vertical-align:-1px}
.lr .num{font-variant-numeric:tabular-nums}
.lr:not(.investigar) .inv{display:none !important}
.lr table{border-collapse:collapse;font-size:12.5px;width:100%} .lr td,.lr th{padding:4px 6px;border-bottom:1px solid var(--line);text-align:left}
.lr .taller{display:grid;grid-template-columns:minmax(0,1.5fr) minmax(260px,1fr);gap:16px}
@media (max-width:860px){.lr .taller{grid-template-columns:1fr}}
.lr .fila{display:grid;grid-template-columns:14px minmax(90px,.8fr) minmax(120px,2fr) 54px 26px;gap:8px;align-items:center;margin:5px 0}
.lr .fila input{width:100%} .lr .fila .c{font-size:12px;color:var(--sub);text-align:right} .lr .fila .c.cero{color:#c0392b;font-weight:600}
.lr .fila .x{border:0;background:none;cursor:pointer;color:var(--sub);font-size:16px}
.lr .chip{display:inline-block;border:1px solid var(--line);border-radius:14px;padding:2px 9px;margin:3px 3px 0 0;font-size:12.5px;cursor:pointer;background:var(--card)}
.lr .chip:hover{border-color:var(--acc)} .lr .chip.usado{opacity:.4;cursor:default}
.lr .aviso{background:var(--soft);border-radius:8px;padding:8px 12px;font-size:13px;margin:8px 0}
.lr .nota{background:var(--card);border:1px solid var(--line);border-radius:12px;padding:12px 16px;margin-bottom:12px}
.lr .nota textarea{width:100%;min-height:70px;margin-top:6px}
.lr .toast{position:fixed;bottom:18px;left:50%;transform:translateX(-50%);background:var(--ink);color:var(--bg);padding:8px 16px;border-radius:20px;font-size:13px;opacity:0;transition:.3s;pointer-events:none;z-index:9}
.lr .toast.on{opacity:1}
.lr footer{margin-top:22px;padding-top:10px;border-top:1px solid var(--line);color:var(--sub);font-size:12px}
.lr details summary{cursor:pointer;color:var(--sub);font-size:13px}
.lr .redpj .n text{font-size:13.5px}
.lr .cuando text{font-size:12px;fill:var(--ink)} .lr .cuando .fila-pj{cursor:pointer} .lr .cuando .fila-pj:hover rect.fondo-fila{fill:var(--soft)}
.lr mark.c{background:none;color:inherit;border-bottom:1.5px dotted var(--sub);padding:0}
.lr .rol{display:inline-block;font-size:11px;border-radius:9px;padding:0 7px;background:var(--soft);color:var(--sub);font-weight:600;letter-spacing:.02em;margin-left:6px;vertical-align:2px}
.lr .grilla.hallpj{grid-template-columns:repeat(auto-fill,minmax(240px,1fr))} .lr .hallpj .frase{font-size:15.5px}
.lr .pjn{font-weight:600;cursor:pointer;white-space:nowrap}
.lr .vpjs{display:flex;gap:4px} .lr .vpjs .btn.on{background:var(--ink);color:var(--bg);border-color:var(--ink)}
.lr .arbol-lienzo{overflow:auto;max-height:720px} .lr .redgrid.ancho{grid-template-columns:minmax(0,1fr)} .lr .redgrid.ancho .panel{max-height:none} .lr svg.arbolpj{display:block} .lr svg.arbolpj text{font-size:12.5px;fill:var(--ink)}
.lr svg.arbolpj .n{cursor:pointer} .lr svg.arbolpj .v{cursor:pointer} .lr svg.arbolpj .v:hover{stroke-width:5}
.lr .chips-epi .chip{cursor:pointer} .lr .chip.on{border-color:var(--acc);background:var(--soft)}
.lr .famrow{display:flex;gap:6px;align-items:baseline;padding:3px 6px;border-radius:6px;cursor:pointer;flex-wrap:wrap} .lr .famrow:hover{background:var(--soft)}
.lr .ia{background:var(--soft);border-radius:8px;padding:8px 12px;margin:6px 0;font-size:14px} .lr .ia p{margin:4px 0}
.lr .ref{font-size:11px;color:var(--acc);font-weight:700;cursor:pointer} .lr .pjn:hover{text-decoration:underline}

</style>

<div class="cab">
  <div><h1></h1><div class="autor"></div><div class="datos"></div></div>
  <div class="interruptor" title="Muestra cifras, unidad de análisis y tablas"><i></i>Modo investigación</div>
</div>
<nav>
  <button data-v="hallazgos" class="on">Hallazgos</button><button data-v="red">Red</button><button data-v="personajes" style="display:none">Personajes</button><button data-v="motivos">Motivos</button>
  <button data-v="taller">Taller</button><button data-v="cuaderno">Cuaderno <span class="n" style="display:none"></span></button>
</nav>

<section class="vista on" data-v="hallazgos">
  <p class="intro">Puntos de entrada a tu lectura. Cada uno señala algo que la red encontró en la obra; ábrelo y comprueba en las frases si lo reconoces, si te sorprende o si no se sostiene.</p>
  <div class="grilla hall"></div>
  <details style="margin-top:18px"><summary>Cómo leer esta herramienta</summary>
    <ul class="sub" style="max-width:760px">
      <li>Un <b>motivo</b> es un grupo de palabras que nombran una misma imagen, objeto, espacio, idea o personaje. Tú decides cuáles (pestaña <b>Taller</b>).</li>
      <li>Dos motivos están <b>unidos</b> cuando aparecen en la misma frase más de lo que se esperaría por azar. Es vecindad, no sentido: el sentido lo confirma tu lectura.</li>
      <li>Los <b>colores</b> son familias de motivos que suelen ir juntos. Una línea <b>punteada</b> es una conexión frágil: desaparece si se mide de otra forma.</li>
      <li>La pestaña <b>Personajes</b> (si la generaste en Colab) arma otra red: quién aparece con quién, cuánto y cuándo.</li>
      <li>Cada cita trae su página. Con <b>＋ citar</b> la guardas en el <b>Cuaderno</b>, donde puedes escribir tus notas y descargarlas.</li>
    </ul></details>
</section>

<section class="vista" data-v="red">
  <div class="ctl">
    <div class="tramo"><button class="btn play" title="Recorrer la obra">▶</button><input type="range" class="s-tramo" min="0" step="1" value="0"><b class="v-tramo"></b></div>
    <label>Motivo <select class="s-focal"></select></label>
    <label class="inv">Unidad <select class="s-unidad"></select></label>
    <label class="inv">Conexiones <input type="range" class="s-top" min="10" max="80" step="5" value="40" style="width:90px"><span class="v-top num">40</span></label>
  </div>
  <div class="redgrid"><div><div class="lienzo"><svg class="red" viewBox="0 0 1000 660"></svg></div><div class="ley"></div>
    <div class="inv" style="margin-top:12px"><div class="tit">Conexiones de esta vista <button class="btn link csv" style="margin-left:8px">descargar CSV</button></div><div class="tabla"></div></div></div>
    <div class="panel"></div></div>
</section>

<section class="vista" data-v="personajes">
  <p class="intro">Quién aparece, cuánto, cuándo y con quién. Dos personajes se unen cuando el texto los nombra (o, en teatro, hablan) en el mismo <b>pasaje</b>: unos párrafos seguidos. Sirve para orientarse en obras con muchos personajes o con nombres que se repiten.</p>
  <div class="ctl">
    <div class="vpjs" style="display:none"><button class="btn vpj on" data-vpj="red">Red</button><button class="btn vpj" data-vpj="arbol">Árbol de familia</button></div>
    <div class="tramo"><button class="btn pj-play" title="Recorrer la obra">▶</button><input type="range" class="pj-tramo" min="0" step="1" value="0"><b class="pj-vtramo"></b></div>
    <label>Personaje <select class="s-pj"></select></label>
  </div>
  <div class="redgrid"><div><div class="lienzo red-lienzo"><svg class="red redpj" viewBox="0 0 1000 640"></svg></div>
    <div class="lienzo arbol-lienzo" style="display:none"><svg class="arbolpj"></svg></div><div class="ley leypj"></div></div>
    <div class="panel panelpj"></div></div>
  <div class="tit" style="margin-top:20px">Para empezar</div>
  <div class="grilla hallpj"></div>
  <div class="tit" style="margin-top:20px">Quién está cuándo</div>
  <div class="sub" style="margin-bottom:6px">Cada raya es una mención (o un parlamento); las líneas verticales separan los tramos. Haz clic en un nombre para abrir su ficha.</div>
  <div class="lienzo" style="padding:6px 8px"><svg class="cuando" viewBox="0 0 1000 100"></svg></div>
  <div class="inv" style="margin-top:14px"><div class="tit">Personajes en cifras <button class="btn link csvpj" style="margin-left:8px">descargar CSV</button></div><div class="tablapj"></div></div>
  <details style="margin-top:16px"><summary>Cómo se arma esta red y qué no ve</summary>
    <ul class="sub" style="max-width:780px">
      <li>Cuenta las <b>menciones explícitas</b>: nombres y papeles («el padre», «la hermana»). Los pronombres («él», «ella») no se cuentan, así que un personaje muy presente pero poco nombrado parece más pequeño.</li>
      <li>Cuando dos personajes comparten un nombre («Aureliano»), cada mención suelta se asigna al que se nombró más cerca en el mismo capítulo. Es una conjetura: revisa las citas.</li>
      <li>El <b>rol</b> combina cuánto se le nombra, con cuántos coincide y en qué parte de la obra está. <b>⇄ Puente</b>: une grupos que casi no se tocan.</li>
      <li>Los <b>colores</b> son círculos de personajes que suelen aparecer juntos.</li>
      <li>El <b>árbol de familia</b> viene del perfil, de lo que escribiste en Colab (paso 3b) o, si no hay, de frases como «X, hijo de Y». Cada vínculo busca una frase del texto que lo diga; si no la encuentra, la línea se ve punteada y gris.</li>
      <li><b>Cómo lo llama la obra</b> recoge las fórmulas con que el texto nombra a un personaje («el coronel», «la bella»). <b>Motivos que lo acompañan</b> cruza esta red con tus motivos: es una base para pensar su papel simbólico, no una interpretación.</li>
      <li>El reparto (quién es quién y con qué nombres) se corrige en Colab, paso 3b.</li>
    </ul></details>
</section>

<section class="vista" data-v="motivos">
  <p class="intro">Cada motivo, de un vistazo: dónde aparece a lo largo de la obra (cada raya es una aparición; las divisiones son los tramos), con qué va y dos frases donde se ve.</p>
  <div class="grilla fichas"></div>
</section>

<section class="vista" data-v="taller">
  <p class="intro">Aquí decides qué es un motivo. Cambia nombres, agrupa palabras o suma sugerencias; al pulsar <b>Aplicar</b>, toda la lectura se recalcula.</p>
  <div class="taller">
    <div class="tarjeta">
      <div class="etq">Tus motivos</div>
      <div class="sub">Palabras separadas por comas, sin tildes. El asterisco significa «cualquier terminación»: <code>pozo*</code> cuenta <i>pozo</i> y <i>pozos</i>; <code>mir*</code>, <i>mirar, miró, mirada</i>… (y también <i>mirlo</i>: revisa las frases).</div>
      <div class="filas"></div>
      <div style="display:flex;gap:8px;flex-wrap:wrap;margin-top:8px">
        <button class="btn nuevo">＋ Nuevo motivo</button><button class="btn pri aplicar">Aplicar cambios</button>
        <button class="btn restaurar">Volver al diccionario inicial</button><button class="btn bajar-dic">Descargar diccionario</button>
        <label class="btn" style="cursor:pointer">Cargar diccionario<input type="file" accept=".json" class="subir-dic" style="display:none"></label>
      </div><div class="aviso estado" style="display:none"></div>
    </div>
    <div class="tarjeta"><div class="etq">Sugerencias del programa</div><div class="sub">Haz clic para sumar una palabra como motivo nuevo (luego puedes unirla a otro).</div><div class="sugs"></div></div>
  </div>
</section>

<section class="vista" data-v="cuaderno">
  <p class="intro">Tus notas y las citas que guardaste, con su página. Descárgalas para llevarlas a tu trabajo.</p>
  <div style="display:flex;gap:8px;margin-bottom:12px;flex-wrap:wrap"><button class="btn pri bajar-notas">Descargar notas (.md)</button><button class="btn nueva-nota">＋ Nota libre</button></div>
  <div class="aviso persist" style="display:none">Este navegador no permite guardar datos: tus notas se perderán al cerrar. Descárgalas antes.</div>
  <div class="notas"></div>
</section>

<footer>Explorador de motivos · La red mide vecindad entre palabras, no sentido: cada hallazgo se confirma leyendo. Las citas son fragmentos para uso de estudio e investigación.</footer>
<div class="toast"></div>

<script>
(function(){
"use strict";
const D = __DATOS__;
const R = document.getElementById("lectura-red");
const $ = s => R.querySelector(s), $$ = s => [...R.querySelectorAll(s)];
const esc = s => String(s).replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const NS = "http://www.w3.org/2000/svg";
const PALETA = ["#C8553D","#2E86AB","#5B8C5A","#8E5BC2","#D9822B","#3D7A8C","#B5838D","#7A8B3A","#A26A42","#5C6BC0"];
const GRIS = "#9AA0A8";
const almacen = (() => { try { const k = "__p"; localStorage.setItem(k, "1"); localStorage.removeItem(k); return localStorage; } catch (e) { return null; } })();
const CLAVE = "explorador-motivos:" + D.titulo;
const leer = (k, d) => { try { const v = almacen && almacen.getItem(CLAVE + ":" + k); return v ? JSON.parse(v) : d; } catch (e) { return d; } };
const guardar = (k, v) => { try { almacen && almacen.setItem(CLAVE + ":" + k, JSON.stringify(v)); } catch (e) {} };

// ---------- texto ----------
const norm = s => s.toLowerCase().replace(/ñ/g, "\u0001").normalize("NFD").replace(/[̀-ͯ]/g, "").replace(/\u0001/g, "ñ");
const RX_TOK = /\d+(?:[.,]\d+)*|[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+/g;
const O = D.O.map(([seg, par, pag, voz, tx], i) => ({i, seg, par, pag, voz: D.voces[voz], tx, toks: (tx.match(RX_TOK) || []).map(norm)}));
let NTOK = 0; O.forEach(o => { o.ini = NTOK; NTOK += o.toks.length; o.tr = D.segmentos[o.seg].tr; });
const FREC = new Map(); O.forEach(o => o.toks.forEach(t => FREC.set(t, (FREC.get(t) || 0) + 1)));
const VOCAB = [...FREC.keys()];
const T = D.tramos.length;
const TR_INI = D.tramos.map((_, t) => { const o = O.find(x => x.tr === t); return o ? o.ini : NTOK; });
const TR_LARGO = D.tramos.map((_, t) => O.filter(o => o.tr === t).reduce((s, o) => s + o.toks.length, 0));
const lugarSeg = s => `${D.tipo_seg.toLowerCase()} ${D.segmentos[s].e}`;
const FEM = ["Escena", "Parte", "Sección"].includes(D.tipo_seg);
const elSeg = s => (FEM ? "la " : "el ") + lugarSeg(s), delSeg = s => (FEM ? "de la " : "del ") + lugarSeg(s);
const pagina = p => D.cita === "impresa" ? `p. ${p}` : `p. ${p} del PDF`;
const oracionEn = pos => { let lo = 0, hi = O.length - 1; while (lo < hi) { const m = (lo + hi + 1) >> 1; if (O[m].ini <= pos) lo = m; else hi = m - 1; } return O[lo]; };
const NOMBRE_UNIDAD = {oracion: "oración", parrafo: D.teatro ? "parlamento" : "párrafo", ventana: "ventana de 60 palabras"};
const PLURAL_UNIDAD = {oracion: "oraciones", parrafo: D.teatro ? "parlamentos" : "párrafos", ventana: "ventanas de 60 palabras"};
const MISMAS = {oracion: "las mismas", parrafo: "los mismos", ventana: "las mismas"};

// ---------- estado ----------
const FIRMA = JSON.stringify(D.motivos).length + ":" + D.motivos.map(m => m.n).join("|");   // si Colab genera otro diccionario, no se recupera el viejo
const guardados = leer("motivos", null);
const S = {motivos: guardados && guardados.firma === FIRMA ? guardados.m : D.motivos.map(m => ({...m})), unidad: D.unidad, tramo: -1, focal: -1, arista: null,
           top: 40, inv: false, notas: leer("notas", []), borrador: null};
let W2M, POSM, UNIS, CACHE, BASE, COLOR, FAM, POS, HALL, BETW;
const K = () => S.motivos.length;
const nombre = k => S.motivos[k].n;
const coincide = (w, fs) => fs.some(f => f.endsWith("*") ? w.startsWith(f.slice(0, -1)) : w === f);
const contar = fs => { let c = 0; for (const [w, n] of FREC) if (coincide(w, fs)) c += n; return c; };

function recalcular() {
  W2M = new Map();
  for (const w of VOCAB) { const ids = []; S.motivos.forEach((m, k) => { if (coincide(w, m.f)) ids.push(k); }); if (ids.length) W2M.set(w, ids); }
  POSM = S.motivos.map(() => []);
  O.forEach(o => { const s = new Set(); o.toks.forEach((t, j) => { const ids = W2M.get(t); if (ids) ids.forEach(k => { s.add(k); POSM[k].push(o.ini + j); }); }); o.m = [...s]; });
  UNIS = {oracion: O.map(o => ({os: [o.i], tr: o.tr, m: o.m}))};
  for (const tipo of ["parrafo", "ventana"]) {
    const g = new Map();
    O.forEach(o => { const k = tipo === "parrafo" ? o.par : Math.floor(o.ini / 60); if (!g.has(k)) g.set(k, {os: [], tr: o.tr, m: new Set()}); const u = g.get(k); u.os.push(o.i); o.m.forEach(x => u.m.add(x)); });
    UNIS[tipo] = [...g.values()].map(u => ({os: u.os, tr: u.tr, m: [...u.m]}));
  }
  CACHE = {}; BASE = red(S.unidad, -1);
  const otras = ["oracion", "parrafo", "ventana"].map(u => new Set(red(u, -1).edges.map(e => e.a * 1000 + e.b)));
  BASE.edges.forEach(e => e.sol = otras.filter(s => s.has(e.a * 1000 + e.b)).length);
  familias(); POS = disponer(); BETW = intermediacion(BASE.edges); HALL = hallazgos();
}

function red(tipo, tr) {
  const key = tipo + "|" + tr + "|" + S.top; if (CACHE[key]) return CACHE[key];
  const US = UNIS[tipo].filter(u => tr < 0 || u.tr === tr), n = US.length, nA = new Array(K()).fill(0), co = new Map();
  US.forEach((u, ui) => { const m = u.m; m.forEach(a => nA[a]++);
    for (let i = 0; i < m.length; i++) for (let j = i + 1; j < m.length; j++) {
      const a = Math.min(m[i], m[j]), b = Math.max(m[i], m[j]), k = a * 1000 + b;
      let e = co.get(k); if (!e) { e = {a, b, c: 0, us: []}; co.set(k, e); } e.c++; e.us.push(ui); } });
  const minJ = tr < 0 ? (n < 5000 ? 2 : Math.max(3, Math.round(n / 3000))) : 2;
  const edges = [...co.values()].filter(e => e.c >= minJ)
    .map(e => ({...e, f: e.c / Math.sqrt(nA[e.a] * nA[e.b]), l: e.c * n / (nA[e.a] * nA[e.b])}))
    .filter(e => e.l > 1).sort((x, y) => y.f - x.f).slice(0, S.top);
  return CACHE[key] = {US, n, nA, edges, co};
}

// ---------- familias (Louvain, una pasada de movimientos locales) ----------
function familias() {
  const k = K(), w = Array.from({length: k}, () => new Map()), deg = new Array(k).fill(0);
  BASE.edges.forEach(e => { w[e.a].set(e.b, e.f); w[e.b].set(e.a, e.f); deg[e.a] += e.f; deg[e.b] += e.f; });
  const m2 = deg.reduce((s, x) => s + x, 0) || 1, com = [...Array(k).keys()], tot = deg.slice();
  for (let it = 0, mov = true; mov && it < 20; it++) { mov = false;
    for (let i = 0; i < k; i++) { if (!deg[i]) continue;
      const ci = com[i]; tot[ci] -= deg[i]; const enlaces = new Map();
      w[i].forEach((f, j) => enlaces.set(com[j], (enlaces.get(com[j]) || 0) + f));
      let mejor = ci, ganancia = (enlaces.get(ci) || 0) - tot[ci] * deg[i] / m2;
      enlaces.forEach((v, c) => { const g = v - tot[c] * deg[i] / m2; if (g > ganancia + 1e-9) { ganancia = g; mejor = c; } });
      com[i] = mejor; tot[mejor] += deg[i]; if (mejor !== ci) mov = true; } }
  const tam = new Map(); com.forEach((c, i) => deg[i] && tam.set(c, (tam.get(c) || 0) + 1));
  const orden = [...tam.entries()].sort((a, b) => b[1] - a[1]).map(x => x[0]);
  FAM = com.map((c, i) => deg[i] ? orden.indexOf(c) : -1);
  COLOR = FAM.map(f => f < 0 ? GRIS : PALETA[f % PALETA.length]);
}

// ---------- disposición: resortes para el grupo principal, fila inferior para el resto ----------
function disponer() {
  const k = K(), par = [...Array(k).keys()], raiz = x => par[x] === x ? x : (par[x] = raiz(par[x]));
  BASE.edges.forEach(e => { par[raiz(e.a)] = raiz(e.b); });
  const comps = new Map(); for (let i = 0; i < k; i++) { if (!POSM[i].length) continue; const r = raiz(i); if (!comps.has(r)) comps.set(r, []); comps.get(r).push(i); }
  const lista = [...comps.values()].sort((a, b) => b.length - a.length), pos = new Array(k).fill(null);
  const resortes = nodos => {
    const n = nodos.length, idx = new Map(nodos.map((x, i) => [x, i]));
    if (n === 1) return [[0.5, 0.5]];
    const p = nodos.map((_, i) => [0.5 + 0.4 * Math.cos(2 * Math.PI * i / n), 0.5 + 0.4 * Math.sin(2 * Math.PI * i / n)]);
    const es = BASE.edges.filter(e => idx.has(e.a) && idx.has(e.b)), fmax = Math.max(...es.map(e => e.f), 1e-6);
    const kk = Math.sqrt(1 / n) * 0.9;
    for (let it = 0; it < 500; it++) {
      const t = 0.08 * (1 - it / 500) + 0.002, d = p.map(() => [0, 0]);
      for (let i = 0; i < n; i++) for (let j = i + 1; j < n; j++) {
        let dx = p[i][0] - p[j][0], dy = p[i][1] - p[j][1], dist = Math.hypot(dx, dy) || 1e-3, f = kk * kk / dist;
        d[i][0] += dx / dist * f; d[i][1] += dy / dist * f; d[j][0] -= dx / dist * f; d[j][1] -= dy / dist * f; }
      es.forEach(e => { const i = idx.get(e.a), j = idx.get(e.b); let dx = p[i][0] - p[j][0], dy = p[i][1] - p[j][1];
        const dist = Math.hypot(dx, dy) || 1e-3, f = dist * dist / kk * (0.4 + e.f / fmax);
        d[i][0] -= dx / dist * f; d[i][1] -= dy / dist * f; d[j][0] += dx / dist * f; d[j][1] += dy / dist * f; });
      p.forEach((q, i) => { const len = Math.hypot(d[i][0], d[i][1]) || 1; q[0] += d[i][0] / len * Math.min(len, t); q[1] += d[i][1] / len * Math.min(len, t); });
    }
    const xs = p.map(q => q[0]), ys = p.map(q => q[1]), [x0, x1, y0, y1] = [Math.min(...xs), Math.max(...xs), Math.min(...ys), Math.max(...ys)];
    return p.map(q => [(q[0] - x0) / ((x1 - x0) || 1), (q[1] - y0) / ((y1 - y0) || 1)]);
  };
  if (lista.length) resortes(lista[0]).forEach((q, i) => pos[lista[0][i]] = [q[0], q[1] * 0.8]);
  const resto = lista.slice(1), anchoTot = resto.reduce((s, c) => s + c.length, 0); let x = 0;
  resto.forEach(c => { const w = c.length / anchoTot; resortes(c).forEach((q, i) => pos[c[i]] = [x + w * (0.15 + 0.7 * q[0]), 0.93 + 0.07 * q[1]]); x += w; });
  return pos;
}

// ---------- intermediación (Brandes con pesos 1/fuerza) ----------
function intermediacion(edges) {
  const k = K(), adj = Array.from({length: k}, () => []), cb = new Array(k).fill(0);
  edges.forEach(e => { adj[e.a].push([e.b, 1 / e.f]); adj[e.b].push([e.a, 1 / e.f]); });
  for (let s = 0; s < k; s++) {
    if (!adj[s].length) continue;
    const dist = new Array(k).fill(Infinity), sigma = new Array(k).fill(0), pred = Array.from({length: k}, () => []), orden = [], hecho = new Array(k).fill(false);
    dist[s] = 0; sigma[s] = 1;
    for (;;) { let u = -1; for (let i = 0; i < k; i++) if (!hecho[i] && dist[i] < Infinity && (u < 0 || dist[i] < dist[u])) u = i;
      if (u < 0) break; hecho[u] = true; orden.push(u);
      adj[u].forEach(([v, w]) => { const nd = dist[u] + w;
        if (nd < dist[v] - 1e-12) { dist[v] = nd; sigma[v] = sigma[u]; pred[v] = [u]; }
        else if (Math.abs(nd - dist[v]) < 1e-12) { sigma[v] += sigma[u]; pred[v].push(u); } }); }
    const delta = new Array(k).fill(0);
    for (let i = orden.length - 1; i >= 0; i--) { const w = orden[i]; pred[w].forEach(v => delta[v] += sigma[v] / sigma[w] * (1 + delta[w])); if (w !== s) cb[w] += delta[w]; }
  }
  const n = cb.filter((_, i) => adj[i].length).length, norma = n > 2 ? (n - 1) * (n - 2) : 1;
  return cb.map(x => x / norma);
}

// ---------- palabras en lugar de números ----------
const union = e => e.l >= 5 && e.c >= 5 ? "inseparables" : e.l >= 3 && e.c >= 3 ? "muy unidos" : e.l >= 1.8 ? "unidos" : "cercanos";
const solidez = s => s >= 3 ? "sólida" : s === 2 ? "probable" : "frágil";
const motHTML = k => `<span class="mot" style="color:${COLOR[k]}">«${esc(nombre(k))}»</span>`;
const pct = x => Math.round(100 * x) + "%";
const veces = n => n === 1 ? "1 vez" : `${n} veces`;

// ---------- hallazgos ----------
function trDePos(p) { return oracionEn(p).tr; }
function hallazgos() {
  const H = [], usados = new Set(), k = K(), minN = Math.max(5, Math.round(NTOK / 20000));
  const vivos = [...Array(k).keys()].filter(i => POSM[i].length >= minN);
  const add = (h, ms) => { if (ms.some(m => usados.has(m)) && H.length > 2) return; ms.forEach(m => usados.add(m)); H.push(h); };
  // puente
  const p = vivos.filter(i => BETW[i] > 0.05).sort((a, b) => BETW[b] - BETW[a])[0];
  if (p !== undefined) add({etq: "El puente", html: `${motHTML(p)} es el puente de tu red: muchas conexiones entre otros motivos pasan por él.`, ms: [p], ir: {focal: p}}, [p]);
  // pareja sólida
  const ps = BASE.edges.filter(e => e.sol >= 3 && e.c >= 3).sort((a, b) => b.l - a.l)[0];
  if (ps) add({etq: "Pareja constante", html: `${motHTML(ps.a)} y ${motHTML(ps.b)} van juntos: aparecen en ${MISMAS[S.unidad]} ${ps.c} ${PLURAL_UNIDAD[S.unidad]} y la conexión se mantiene se mida como se mida.`, ms: [ps.a, ps.b], ir: {arista: [ps.a, ps.b]}}, [ps.a, ps.b]);
  // nace tarde
  const tarde = vivos.map(i => [i, POSM[i][0] / NTOK]).filter(x => x[1] >= 0.3).sort((a, b) => b[1] - a[1])[0];
  if (tarde) { const o = oracionEn(POSM[tarde[0]][0]);
    add({etq: "Nace tarde", html: `${motHTML(tarde[0])} no aparece hasta ${elSeg(o.seg)} (${pagina(o.pag)}), pasado el ${pct(tarde[1])} de la obra.`, ms: [tarde[0]], ir: {focal: tarde[0], tramo: o.tr}}, [tarde[0]]); }
  // se apaga
  const apaga = vivos.map(i => [i, POSM[i][POSM[i].length - 1] / NTOK]).filter(x => x[1] <= 0.7).sort((a, b) => a[1] - b[1])[0];
  if (apaga) { const o = oracionEn(POSM[apaga[0]][POSM[apaga[0]].length - 1]);
    add({etq: "Se apaga", html: `${motHTML(apaga[0])} desaparece después ${delSeg(o.seg)} (${pagina(o.pag)}): no vuelve en el último ${pct(1 - apaga[1])} de la obra.`, ms: [apaga[0]], ir: {focal: apaga[0]}}, [apaga[0]]); }
  // ráfaga
  if (T > 1) { const raf = vivos.map(i => { const c = new Array(T).fill(0); POSM[i].forEach(p => c[trDePos(p)]++); const t = c.indexOf(Math.max(...c));
      return [i, c[t] / POSM[i].length, t]; }).filter(x => x[1] >= 0.6 && TR_LARGO[x[2]] / NTOK <= 0.45).sort((a, b) => b[1] - a[1])[0];
    if (raf) add({etq: "En ráfaga", html: `El ${pct(raf[1])} de ${motHTML(raf[0])} está en un solo tramo: <i>${esc(D.tramos[raf[2]])}</i>.`, ms: [raf[0]], ir: {focal: raf[0], tramo: raf[2]}}, [raf[0]]); }
  // encuentro localizado
  if (T > 1) { const enc = BASE.edges.filter(e => e.c >= 4).map(e => { const c = new Array(T).fill(0); e.us.forEach(u => c[BASE.US[u].tr]++);
      const t = c.indexOf(Math.max(...c)); return [e, c[t], t]; }).filter(x => x[1] / x[0].c >= 0.6 && TR_LARGO[x[2]] / NTOK <= 0.45).sort((a, b) => b[1] - a[1])[0];
    if (enc) add({etq: "Un encuentro", html: `${motHTML(enc[0].a)} y ${motHTML(enc[0].b)} se encuentran sobre todo en <i>${esc(D.tramos[enc[2]])}</i> (${enc[1]} de sus ${enc[0].c} coincidencias).`, ms: [enc[0].a, enc[0].b], ir: {arista: [enc[0].a, enc[0].b], tramo: enc[2]}}, [enc[0].a, enc[0].b]); }
  // se evitan
  let ev = null;
  for (let a = 0; a < k; a++) for (let b = a + 1; b < k; b++) {
    const esp = BASE.nA[a] * BASE.nA[b] / BASE.n; if (esp < 4 || BASE.nA[a] < 10 || BASE.nA[b] < 10) continue;
    const c = (BASE.co.get(a * 1000 + b) || {c: 0}).c, r = c / esp;
    if (r <= 0.35 && (!ev || r < ev[3])) ev = [a, b, c, r, esp]; }
  if (ev) add({etq: "Se evitan", html: `${motHTML(ev[0])} y ${motHTML(ev[1])} son frecuentes, pero casi nunca coinciden: ${veces(ev[2])}, cuando por azar se esperarían unas ${Math.round(ev[4])}.`, ms: [ev[0], ev[1]], ir: {focal: ev[0]}}, [ev[0], ev[1]]);
  return H;
}

// ---------- piezas visuales ----------
function tira(ks, alto = 16) {
  const W = 600, H = ks.length * alto + 14;
  let s = `<svg class="tira" viewBox="0 0 ${W} ${H}" preserveAspectRatio="none" role="img" aria-label="Apariciones a lo largo de la obra">`;
  s += `<rect x="0" y="0" width="${W}" height="${H - 12}" fill="var(--soft)" rx="3"/>`;
  TR_INI.slice(1).forEach(p => s += `<line x1="${p / NTOK * W}" x2="${p / NTOK * W}" y1="0" y2="${H - 12}" stroke="var(--line)" stroke-width="2"/>`);
  ks.forEach((k, r) => POSM[k].forEach(p => s += `<rect x="${(p / NTOK * W).toFixed(1)}" y="${r * alto + 2}" width="1.6" height="${alto - 4}" fill="${COLOR[k]}"/>`));
  s += `<text x="0" y="${H - 1}" font-size="10" fill="var(--sub)">inicio</text><text x="${W}" y="${H - 1}" font-size="10" fill="var(--sub)" text-anchor="end">final</text></svg>`;
  return s;
}
function ventana(tx, tramos, largo) {
  if (tx.length <= largo) return [0, tx.length];
  if (!tramos.length) return [0, largo];
  const i = Math.min(...tramos.map(t => t[0])), f = Math.max(...tramos.map(t => t[1]));
  largo = Math.max(largo, Math.min(f - i + 80, 2 * largo));
  const a = Math.max(0, Math.min(i - Math.max(0, Math.floor((largo - (f - i)) / 3)), tx.length - largo));
  return [a, Math.min(tx.length, a + largo)];
}
function fragmento(os, marcas, largo = 300) {   // os = índices de oraciones de la unidad; marcas = {k: "a"|"b"}
  const partes = os.map(i => O[i]); const tx = partes.map(o => o.tx).join(" ");
  const hits = []; let m; RX_TOK.lastIndex = 0;
  while ((m = RX_TOK.exec(tx))) { const ids = W2M.get(norm(m[0])); if (ids) { const c = ids.find(x => x in marcas); if (c !== undefined) hits.push([m.index, m.index + m[0].length, marcas[c], c]); } }
  const primeros = []; const vistos = new Set(); hits.forEach(h => { if (!vistos.has(h[3])) { vistos.add(h[3]); primeros.push(h); } });
  const [a, b] = ventana(tx, primeros, largo);
  let out = "", p = a;
  hits.forEach(([s, e, cls]) => { if (s < a || e > b) return; out += esc(tx.slice(p, s)) + `<mark class="${cls}">${esc(tx.slice(s, e))}</mark>`; p = e; });
  out += esc(tx.slice(p, b));
  let off = 0, o0 = partes[0]; for (const o of partes) { if (off + o.tx.length >= (primeros[0] ? primeros[0][0] : 0)) { o0 = o; break; } off += o.tx.length + 1; }
  const voz = D.teatro ? o0.voz : (o0.voz === "diálogo" ? "diálogo" : "");
  const lug = `${lugarSeg(o0.seg)} · ${pagina(o0.pag)}${voz ? " · " + voz : ""}`;
  const plano = (a > 0 ? "…" : "") + tx.slice(a, b).trim() + (b < tx.length ? "…" : "");
  return {html: `<div class="cita"><span class="lug">${esc(lug)}<button class="mas" data-cita="${esc(JSON.stringify({tx: plano, lug}))}">＋ citar</button></span>${a > 0 ? "…" : ""}${out.trim()}${b < tx.length ? "…" : ""}</div>`};
}
const repartir = (arr, n) => arr.length <= n ? arr : Array.from({length: n}, (_, j) => arr[Math.round(j * (arr.length - 1) / (n - 1))]);

// ---------- vista: hallazgos ----------
function pintarHallazgos() {
  $(".hall").innerHTML = HALL.length ? HALL.map((h, i) => `<div class="tarjeta"><div class="etq">${h.etq}</div><div class="frase">${h.html}</div>${tira(h.ms, 14)}
    <button class="btn link ir" data-i="${i}">Ver en la red →</button></div>`).join("")
    : `<div class="aviso">Con estos motivos la red no muestra patrones claros todavía. Prueba con más motivos o agrupa variantes en el Taller.</div>`;
  $$(".hall .ir").forEach(b => b.onclick = () => { const h = HALL[+b.dataset.i].ir; irRed(h); });
  if (DP && DP.hall.length) {
    const d = document.createElement("div"); d.className = "tarjeta";
    d.innerHTML = `<div class="etq">Personajes · ${esc(DP.hall[0].etq)}</div><div class="frase">${txtPJ(DP.hall[0].txt)}</div>${tiraPJ(DP.hall[0].pj, 14)}<button class="btn link">Ver la red de personajes →</button>`;
    d.querySelector("button").onclick = () => irPJ(DP.hall[0].ir); enlazarPJ(d);
    d.querySelectorAll(".pjn").forEach(x => x.onclick = () => irPJ({focal: +x.dataset.pj}));
    $(".hall").appendChild(d);
  }
}
function irRed(h) {
  S.tramo = h.tramo !== undefined ? h.tramo : -1; S.focal = h.focal !== undefined ? h.focal : -1;
  S.arista = h.arista ? h.arista : null; if (h.arista) S.focal = h.arista[0];
  mostrar("red");
}

// ---------- vista: red ----------
function vistaRed() { return red(S.unidad, S.tramo); }
function vecinos(r, k) { return r.edges.filter(e => e.a === k || e.b === k).map(e => ({...e, otro: e.a === k ? e.b : e.a})).sort((x, y) => y.f - x.f); }
function pintarRed() {
  const r = vistaRed(), svg = $("svg.red"), k = K();
  $(".s-tramo").max = T; $(".s-tramo").value = S.tramo + 1;
  $(".v-tramo").textContent = S.tramo < 0 ? "Toda la obra" : `${S.tramo + 1}/${T} · ${D.tramos[S.tramo]}`;
  $(".s-focal").value = S.focal;
  const fmax = Math.max(0.01, ...r.edges.map(e => e.f)), nmax = Math.max(1, ...r.nA);
  const solidas = new Map(BASE.edges.map(e => [e.a * 1000 + e.b, e.sol]));
  const veci = new Set(S.focal >= 0 ? vecinos(r, S.focal).map(v => v.otro).concat([S.focal]) : []);
  const P = i => POS[i] ? [70 + 860 * POS[i][0], 40 + 590 * POS[i][1]] : null;
  let s = `<rect x="0" y="0" width="1000" height="660" fill="transparent" class="fondo"/>`;
  r.edges.forEach((e, idx) => { const pa = P(e.a), pb = P(e.b); if (!pa || !pb) return;
    const toca = S.focal < 0 || e.a === S.focal || e.b === S.focal;
    const esSel = S.arista && ((S.arista[0] === e.a && S.arista[1] === e.b) || (S.arista[0] === e.b && S.arista[1] === e.a));
    const sol = S.tramo < 0 ? e.sol : (solidas.get(e.a * 1000 + e.b) || 1);
    const col = esSel ? "var(--acc)" : (FAM[e.a] >= 0 && FAM[e.a] === FAM[e.b] ? COLOR[e.a] : GRIS);
    s += `<line data-e="${idx}" x1="${pa[0]}" y1="${pa[1]}" x2="${pb[0]}" y2="${pb[1]}" stroke="${col}" stroke-linecap="round"
      stroke-width="${(1.2 + 10 * e.f / fmax).toFixed(1)}" stroke-opacity="${toca ? (S.focal >= 0 ? .9 : .55) : .07}" ${sol <= 1 ? 'stroke-dasharray="2 7"' : ""}>
      <title>${esc(nombre(e.a))} — ${esc(nombre(e.b))}: ${union(e)} (${veces(e.c)} juntos)</title></line>`; });
  for (let i = 0; i < k; i++) { const p = P(i); if (!p) continue;
    const c = r.nA[i], rad = c ? 6 + 20 * Math.sqrt(c / nmax) : 4, op = S.focal >= 0 && !veci.has(i) ? .16 : (c ? 1 : .35);
    s += `<g class="n" data-k="${i}" opacity="${op}"><circle cx="${p[0]}" cy="${p[1]}" r="${rad.toFixed(1)}" pointer-events="all" fill="${c ? COLOR[i] : "var(--card)"}" stroke="${i === S.focal ? "var(--ink)" : (c ? "var(--card)" : COLOR[i])}" stroke-width="${i === S.focal ? 3.5 : 1.5}"/>
      <text x="${p[0]}" y="${p[1] - rad - 6}" text-anchor="middle">${esc(nombre(i))}</text><title>${esc(nombre(i))}: ${veces(c)} en esta vista</title></g>`; }
  svg.innerHTML = s;
  svg.querySelector(".fondo").onclick = () => { S.focal = -1; S.arista = null; pintarRed(); };
  svg.querySelectorAll("g.n").forEach(g => g.onclick = ev => { ev.stopPropagation(); const i = +g.dataset.k; S.focal = S.focal === i ? -1 : i; S.arista = null; pintarRed(); });
  svg.querySelectorAll("line[data-e]").forEach(l => l.onclick = ev => { ev.stopPropagation(); const e = r.edges[+l.dataset.e]; S.arista = [e.a, e.b]; pintarRed(); });
  const fams = [...new Set(FAM.filter(f => f >= 0))].sort((a, b) => a - b);
  $(".ley").innerHTML = fams.map(f => `<span><i style="background:${PALETA[f % PALETA.length]}"></i>${FAM.map((x, i) => x === f ? nombre(i) : null).filter(Boolean).slice(0, 3).map(esc).join(", ")}${FAM.filter(x => x === f).length > 3 ? "…" : ""}</span>`).join("")
    + `<span>— línea continua: conexión sólida · <span style="letter-spacing:2px">· · ·</span> frágil</span>`;
  if (S.arista) panelArista(r); else if (S.focal >= 0) panelNodo(r); else panelResumen(r);
  if (S.inv) tablaRed(r);
}
function panelResumen(r) {
  const top = r.edges.slice(0, 8);
  $(".panel").innerHTML = `<h3>${S.tramo < 0 ? "Toda la obra" : esc(D.tramos[S.tramo])}</h3>
    <div class="sub">${S.tramo < 0 ? "Mueve el control de arriba (o pulsa ▶) para recorrer la obra por tramos." : "La red solo con este tramo. Los motivos quedan en el mismo lugar para comparar."}</div>
    <div class="tit">Lo que más se une aquí</div>${top.length ? top.map((e, i) => `<div class="vec" data-i="${i}"><span>${motHTML(e.a)} y ${motHTML(e.b)}</span><span>${union(e)}<span class="inv"> · ${e.c} · ×${e.l.toFixed(1)}</span></span></div>`).join("") : '<div class="sub">Sin conexiones en esta vista.</div>'}
    <div class="tit">Para empezar</div><div class="sub">Haz clic en un motivo para ver con qué se une y en qué frases; o en una línea para leer las frases donde dos motivos coinciden.</div>`;
  $$(".panel .vec").forEach(d => d.onclick = () => { const e = top[+d.dataset.i]; S.arista = [e.a, e.b]; pintarRed(); });
}
function panelNodo(r) {
  const k = S.focal, vs = vecinos(r, k), us = r.US.map((u, i) => [u, i]).filter(([u]) => u.m.includes(k)).map(([u]) => u);
  const conM = us.sort((a, b) => b.m.length - a.m.length).slice(0, 12).sort((a, b) => a.os[0] - b.os[0]);
  const citas = repartir(conM, 4).map(u => fragmento(u.os, {[k]: "a"}).html).join("");
  $(".panel").innerHTML = `<h3 style="color:${COLOR[k]}">${esc(nombre(k))}</h3>
    <div class="sub">${veces(r.nA[k])} en ${S.tramo < 0 ? "toda la obra" : "este tramo"} <span class="inv">· en ${r.nA[k]} ${PLURAL_UNIDAD[S.unidad]} · palabras: ${esc(S.motivos[k].f.join(", "))}</span></div>
    ${tira([k])}<div class="tit">Va con</div>
    ${vs.length ? vs.slice(0, 10).map((v, i) => `<div class="vec" data-i="${i}"><span>${motHTML(v.otro)}</span><span>${union(v)}<span class="inv"> · ${v.c} · ×${v.l.toFixed(1)}</span></span></div>`).join("") : '<div class="sub">No se une con otros motivos en esta vista: aparece solo.</div>'}
    <div class="tit">Dónde se ve</div>${citas || '<div class="sub">No aparece en este tramo.</div>'}
    <button class="btn anotar" style="margin-top:6px">✎ Anotar sobre «${esc(nombre(k))}»</button>`;
  $$(".panel .vec").forEach(d => d.onclick = () => { const v = vs[+d.dataset.i]; S.arista = [k, v.otro]; pintarRed(); });
  conectarCitas($(".panel"), `«${nombre(k)}»`);
  $(".panel .anotar").onclick = () => abrirNota(`«${nombre(k)}»`);
}
function panelArista(r) {
  const [a, b] = S.arista, lo = Math.min(a, b), hi = Math.max(a, b);
  const e = r.co.get(lo * 1000 + hi), ed = r.edges.find(x => x.a === lo && x.b === hi);
  const us = e ? e.us.map(i => r.US[i]) : [];
  const citas = repartir(us, 6).map(u => fragmento(u.os, {[a]: "a", [b]: "b"}).html).join("");
  const base = BASE.edges.find(x => x.a === lo && x.b === hi);
  $(".panel").innerHTML = `<h3><mark class="a">${esc(nombre(a))}</mark> y <mark class="b">${esc(nombre(b))}</mark></h3>
    <div class="sub">${e ? `${veces(e.c)} juntos${ed ? " · " + union(ed) : ""}${base ? " · conexión " + solidez(base.sol) : ""}` : "No coinciden en esta vista"}
    <span class="inv">${ed ? ` · fuerza ${ed.f.toFixed(3)} · ×${ed.l.toFixed(1)} lo esperado` : ""}${base ? ` · aparece con ${base.sol} de 3 unidades` : ""}</span></div>
    ${tira([a, b])}<div class="tit">Donde coinciden</div>${citas || '<div class="sub">—</div>'}
    <div style="display:flex;gap:8px;flex-wrap:wrap;margin-top:6px"><button class="btn anotar">✎ Anotar</button><button class="btn volver">← volver a «${esc(nombre(a))}»</button></div>`;
  conectarCitas($(".panel"), `«${nombre(a)}» y «${nombre(b)}»`);
  $(".panel .anotar").onclick = () => abrirNota(`«${nombre(a)}» y «${nombre(b)}»`);
  $(".panel .volver").onclick = () => { S.arista = null; S.focal = a; pintarRed(); };
}
function tablaRed(r) {
  const solidas = new Map(BASE.edges.map(e => [e.a * 1000 + e.b, e.sol]));
  $(".tabla").innerHTML = `<table><tr><th>motivo A</th><th>motivo B</th><th class="num">juntos</th><th class="num">fuerza</th><th class="num">× esperado</th><th>solidez</th></tr>` +
    r.edges.map(e => `<tr><td>${esc(nombre(e.a))}</td><td>${esc(nombre(e.b))}</td><td class="num">${e.c}</td><td class="num">${e.f.toFixed(3)}</td><td class="num">${e.l.toFixed(1)}</td><td>${solidas.has(e.a * 1000 + e.b) ? solidas.get(e.a * 1000 + e.b) + "/3" : "—"}</td></tr>`).join("") + `</table>`;
}

// ---------- vista: motivos ----------
function pintarFichas() {
  const orden = [...Array(K()).keys()].filter(k => POSM[k].length).sort((a, b) => POSM[b].length - POSM[a].length);
  const vacios = [...Array(K()).keys()].filter(k => !POSM[k].length);
  const usadas = new Set();
  $(".fichas").innerHTML = orden.map(k => {
    const ini = oracionEn(POSM[k][0]), fin = oracionEn(POSM[k][POSM[k].length - 1]), vs = vecinos(BASE, k).slice(0, 3);
    const peso = u => u.os.reduce((s, i) => s + O[i].toks.filter(t => (W2M.get(t) || []).includes(k)).length, 0);
    const us = BASE.US.filter(u => u.m.includes(k)), tercios = [0, 1, 2].map(t => us.filter(u => O[u.os[0]].ini >= t * NTOK / 3 && O[u.os[0]].ini < (t + 1) * NTOK / 3));
    const elegidas = tercios.map(g => g.filter(u => !usadas.has(u.os[0])).sort((a, b) => peso(b) - peso(a) || a.os.length - b.os.length)[0]).filter(Boolean).slice(0, 2);
    elegidas.forEach(u => usadas.add(u.os[0]));
    const citas = elegidas.map(u => fragmento(u.os, {[k]: "a"}, 220).html).join("");
    return `<div class="tarjeta"><div style="display:flex;justify-content:space-between;align-items:baseline;gap:8px"><span class="frase" style="color:${COLOR[k]};font-weight:600">${esc(nombre(k))}</span>
      <span class="sub">${veces(POSM[k].length)}</span></div><div class="sub">Desde ${elSeg(ini.seg)} hasta ${elSeg(fin.seg)}<span class="inv"> · ${esc(S.motivos[k].f.join(", "))}</span></div>${tira([k])}
      <div class="sub">${vs.length ? "Va con " + vs.map(v => `${motHTML(v.otro)} <span class="sub">(${union(v)})</span>`).join(", ") : "No se une con otros motivos."}</div>${citas}
      <button class="btn link ver" data-k="${k}">Ver en la red →</button></div>`; }).join("")
    + (vacios.length ? `<div class="aviso">Sin apariciones (revisa sus palabras en el Taller): ${vacios.map(k => esc(nombre(k))).join(", ")}</div>` : "");
  $$(".fichas .ver").forEach(b => b.onclick = () => irRed({focal: +b.dataset.k}));
  $$(".fichas .tarjeta").forEach(t => conectarCitas(t, null));
}

// ---------- vista: taller ----------
function pintarTaller() {
  if (!S.borrador) S.borrador = S.motivos.map(m => ({n: m.n, f: m.f.join(", "), t: m.t}));
  const filas = $(".filas");
  filas.innerHTML = S.borrador.map((m, i) => { const fs = formas(m.f), c = contar(fs);
    return `<div class="fila" data-i="${i}"><i style="width:10px;height:10px;border-radius:50%;background:${i < K() && S.motivos[i] && S.motivos[i].n === m.n ? COLOR[i] : GRIS}"></i>
    <input type="text" class="nom" value="${esc(m.n)}" aria-label="nombre"><input type="text" class="fs" value="${esc(m.f)}" aria-label="palabras">
    <span class="c ${c ? "" : "cero"}">${c ? c : "0"}</span><button class="x" title="Quitar">✕</button></div>`; }).join("");
  filas.querySelectorAll(".fila").forEach(f => { const i = +f.dataset.i;
    f.querySelector(".nom").oninput = ev => { S.borrador[i].n = ev.target.value; marcarCambios(); };
    f.querySelector(".fs").oninput = ev => { S.borrador[i].f = ev.target.value; const c = contar(formas(ev.target.value)); const sp = f.querySelector(".c"); sp.textContent = c; sp.classList.toggle("cero", !c); marcarCambios(); };
    f.querySelector(".x").onclick = () => { S.borrador.splice(i, 1); marcarCambios(); pintarTaller(); }; });
  const actuales = S.borrador.map(m => formas(m.f));
  const usado = c => c.f.some(w => actuales.some(fs => coincide(w, fs)));
  const titulos = {recurrente: "Recorren toda la obra", concentrado: "Aparecen en ráfagas", "característico": "Propias de esta obra", nombre: "Nombres"};
  $(".sugs").innerHTML = Object.entries(D.candidatos).filter(([, l]) => l.length).map(([lista, l]) => `<div class="tit">${titulos[lista] || lista}</div>` +
    l.map((c, j) => `<span class="chip ${usado(c) ? "usado" : ""}" data-l="${lista}" data-j="${j}">+ ${esc(c.n)} <span class="sub">${c.c}</span></span>`).join("")).join("");
  $$(".sugs .chip").forEach(ch => ch.onclick = () => { if (ch.classList.contains("usado")) return; const c = D.candidatos[ch.dataset.l][+ch.dataset.j];
    S.borrador.push({n: c.n, f: c.f.join(", "), t: ch.dataset.l}); marcarCambios(); pintarTaller(); });
}
const formas = s => s.split(",").map(x => norm(x.trim())).filter(Boolean);
function marcarCambios(txt) { const e = $(".estado"); e.style.display = "block"; e.textContent = txt || "Hay cambios sin aplicar."; }
function aplicar() {
  const nuevos = S.borrador.map(m => ({n: m.n.trim() || "sin nombre", f: formas(m.f), t: m.t || "propio"})).filter(m => m.f.length);
  if (!nuevos.length) { marcarCambios("Necesitas al menos un motivo con palabras."); return; }
  S.motivos = nuevos; guardar("motivos", {firma: FIRMA, m: S.motivos}); S.borrador = null; S.focal = -1; S.arista = null;
  recalcular(); pintarTodo(); marcarCambios(`Listo: ${K()} motivos. La lectura se recalculó.`); aviso("Motivos aplicados");
}
function descargar(nombreArchivo, contenido, tipo) {
  try { const a = document.createElement("a"); a.href = URL.createObjectURL(new Blob([contenido], {type: tipo})); a.download = nombreArchivo; document.body.appendChild(a); a.click(); a.remove(); }
  catch (e) { aviso("Este entorno no permite descargas: abre el archivo HTML en tu navegador."); }
}

// ---------- vista: cuaderno ----------
function abrirNota(ref) {
  let n = S.notas.find(x => x.ref === ref);
  if (!n) { n = {id: Date.now(), ref, texto: "", citas: [], fecha: new Date().toLocaleDateString("es")}; S.notas.unshift(n); }
  guardar("notas", S.notas); mostrar("cuaderno");
  const ta = R.querySelector(`.nota[data-id="${n.id}"] textarea`); if (ta) ta.focus();
}
function conectarCitas(cont, ref) {
  cont.querySelectorAll(".mas").forEach(b => b.onclick = ev => { ev.stopPropagation();
    const c = JSON.parse(b.dataset.cita), r = ref || (b.closest(".tarjeta") ? "«" + b.closest(".tarjeta").querySelector(".frase").textContent.trim() + "»" : "Citas sueltas");
    let n = S.notas.find(x => x.ref === r); if (!n) { n = {id: Date.now(), ref: r, texto: "", citas: [], fecha: new Date().toLocaleDateString("es")}; S.notas.unshift(n); }
    if (!n.citas.some(x => x.tx === c.tx)) n.citas.push(c); guardar("notas", S.notas); contarNotas(); aviso("Cita guardada en el Cuaderno"); });
}
function contarNotas() { const n = $("nav .n"); n.textContent = S.notas.length; n.style.display = S.notas.length ? "inline" : "none"; }
function pintarCuaderno() {
  $(".persist").style.display = almacen ? "none" : "block";
  $(".notas").innerHTML = S.notas.length ? S.notas.map(n => `<div class="nota" data-id="${n.id}"><div style="display:flex;justify-content:space-between;gap:8px">
    <b class="frase" style="font-size:16px">${esc(n.ref)}</b><span class="sub">${esc(n.fecha)} <button class="btn link borrar" style="margin-left:8px">borrar</button></span></div>
    ${n.citas.map((c, j) => `<div class="cita"><span class="lug">${esc(c.lug)}<button class="mas quitar" data-j="${j}">quitar</button></span>${esc(c.tx)}</div>`).join("")}
    <textarea placeholder="Tu lectura: qué ves aquí, qué te sorprende, qué vas a revisar…">${esc(n.texto)}</textarea></div>`).join("")
    : `<div class="aviso">Todavía no hay notas. En la Red o en las fichas, usa <b>＋ citar</b> para guardar una frase con su página, o <b>✎ Anotar</b> para escribir sobre un motivo o una conexión.</div>`;
  $$(".nota").forEach(d => { const n = S.notas.find(x => x.id === +d.dataset.id);
    d.querySelector("textarea").oninput = ev => { n.texto = ev.target.value; guardar("notas", S.notas); };
    d.querySelector(".borrar").onclick = () => { S.notas = S.notas.filter(x => x !== n); guardar("notas", S.notas); pintarCuaderno(); contarNotas(); };
    d.querySelectorAll(".quitar").forEach(q => q.onclick = () => { n.citas.splice(+q.dataset.j, 1); guardar("notas", S.notas); pintarCuaderno(); }); });
  contarNotas();
}
function notasMarkdown() {
  return `# Notas de lectura: ${D.titulo}${D.autor ? " (" + D.autor + ")" : ""}\n\n` + S.notas.map(n => `## ${n.ref}\n\n${n.texto ? n.texto + "\n\n" : ""}` +
    n.citas.map(c => `> ${c.tx}\n> — ${D.titulo}, ${c.lug}\n`).join("\n")).join("\n") +
    `\n---\nMotivos usados: ${S.motivos.map(m => `${m.n} (${m.f.join(", ")})`).join("; ")}\n`;
}

// ---------- vista: personajes ----------
const DP = D.personajes || null;
const KP = DP ? DP.lista.length : 0;
const SP = {focal: -1, arista: null, tramo: -1, vista: "red", vinculo: -1};
const HAY_ARBOL = DP && DP.arbol && Object.keys(DP.arbol).length > 0;
let PJ_O, PJ_U, PJ_POS, PJ_TR, PJ_CACHE = {}, relojPJ = null;
const pjCol = k => DP.lista[k].g >= 0 ? PALETA[DP.lista[k].g % PALETA.length] : GRIS;
const pjNom = k => DP.lista[k].n;
const pjHTML = k => `<span class="pjn" data-pj="${k}" style="color:${pjCol(k)}">${esc(pjNom(k))}</span>`;
const ROL = {"protagonista": "protagonista", "principal": "principal", "secundario": "secundario", "episódico": "episódico", "sin apariciones": "sin apariciones"};
const ROL_TXT = {"protagonista": "el más nombrado y conectado, presente en casi toda la obra", "principal": "pesa en la red y recorre buena parte de la obra",
  "secundario": "aparece a lo largo de la obra, con menos peso", "episódico": "se concentra en una parte de la obra", "sin apariciones": "sus nombres no aparecen en el texto"};
const relPJ = (c, a, b) => { const s = c / Math.max(1, Math.min(a, b)); return s >= .5 ? "casi siempre juntos" : s >= .25 ? "a menudo juntos" : "a veces juntos"; };
function prepararPJ() {
  if (!DP) return;
  PJ_O = O.map(() => []);
  DP.M.forEach(([o, a, b, k, amb]) => PJ_O[o].push({a, b, k, amb}));
  const VOZ = {}; Object.entries(DP.voz || {}).forEach(([v, k]) => VOZ[D.voces[+v]] = k);
  PJ_POS = DP.lista.map(() => []);
  O.forEach(o => PJ_O[o.i].forEach(m => PJ_POS[m.k].push(o.ini)));
  const visto = new Set();
  O.forEach(o => { const k = VOZ[o.voz]; if (k === undefined) return; const c = o.par + ":" + k; if (!visto.has(c)) { visto.add(c); PJ_POS[k].push(o.ini); } });
  PJ_POS.forEach(p => p.sort((x, y) => x - y));
  PJ_TR = PJ_POS.map(ps => { const c = new Array(T).fill(0); ps.forEach(p => c[oracionEn(p).tr]++); return c; });
  PJ_U = DP.P.map(([a, b]) => { const s = new Set(), os = [];
    for (let i = a; i <= b; i++) { os.push(i); PJ_O[i].forEach(m => s.add(m.k)); const k = VOZ[O[i].voz]; if (k !== undefined) s.add(k); }
    return {os, tr: O[a].tr, s: [...s], voz: VOZ}; });
}
function redPJ(tr) {
  if (PJ_CACHE[tr]) return PJ_CACHE[tr];
  const US = PJ_U.filter(u => tr < 0 || u.tr === tr), nA = new Array(KP).fill(0), co = new Map();
  US.forEach((u, ui) => { const m = u.s; m.forEach(a => nA[a]++);
    for (let i = 0; i < m.length; i++) for (let j = i + 1; j < m.length; j++) {
      const a = Math.min(m[i], m[j]), b = Math.max(m[i], m[j]), key = a * 1000 + b;
      let e = co.get(key); if (!e) { e = {a, b, c: 0, us: []}; co.set(key, e); } e.c++; e.us.push(ui); } });
  const minJ = tr < 0 ? DP.minJ : 2;
  const edges = [...co.values()].filter(e => e.c >= minJ).sort((x, y) => y.c - x.c).slice(0, tr < 0 ? 80 : 50);
  return PJ_CACHE[tr] = {US, nA, co, edges};
}
const pesoPJ = (k, tr) => tr < 0 ? PJ_POS[k].length : PJ_TR[k][tr];
function tiraPJ(ks, alto = 16) {
  const W = 600, H = ks.length * alto + 14;
  let s = `<svg class="tira" viewBox="0 0 ${W} ${H}" preserveAspectRatio="none" role="img" aria-label="Apariciones a lo largo de la obra"><rect x="0" y="0" width="${W}" height="${H - 12}" fill="var(--soft)" rx="3"/>`;
  TR_INI.slice(1).forEach(p => s += `<line x1="${p / NTOK * W}" x2="${p / NTOK * W}" y1="0" y2="${H - 12}" stroke="var(--line)" stroke-width="2"/>`);
  ks.forEach((k, r) => PJ_POS[k].forEach(p => s += `<rect x="${(p / NTOK * W).toFixed(1)}" y="${r * alto + 2}" width="1.6" height="${alto - 4}" fill="${pjCol(k)}"/>`));
  return s + `<text x="0" y="${H - 1}" font-size="10" fill="var(--sub)">inicio</text><text x="${W}" y="${H - 1}" font-size="10" fill="var(--sub)" text-anchor="end">final</text></svg>`;
}
function fragmentoPJ(u, marcas, largo = 320, marcasM = null) {
  let tx = "", spans = [], offs = [];
  u.os.forEach(i => { if (tx) tx += " "; offs.push([tx.length, i]); PJ_O[i].forEach(m => spans.push([tx.length + m.a, tx.length + m.b, m.k])); tx += O[i].tx; });
  if (marcasM) { let mm; RX_TOK.lastIndex = 0;
    while ((mm = RX_TOK.exec(tx))) { const ids = W2M.get(norm(mm[0])); const c = ids && ids.find(x => x in marcasM);
      if (c !== undefined && !spans.some(z => z[0] <= mm.index && mm.index < z[1])) spans.push([mm.index, mm.index + mm[0].length, "m" + c]); }
    spans.sort((x, y) => x[0] - y[0]); Object.entries(marcasM).forEach(([m, cl]) => marcas["m" + m] = cl); }
  const hits = spans.filter(x => x[2] in marcas), vistos = new Set(), primeros = [];
  hits.forEach(h => { if (!vistos.has(h[2])) { vistos.add(h[2]); primeros.push(h); } });
  const [a, b] = ventana(tx, primeros, largo);
  let out = "", p = a;
  spans.forEach(([s0, e0, k]) => { if (s0 < a || e0 > b || s0 < p) return;
    out += esc(tx.slice(p, s0)) + `<mark class="${k in marcas ? marcas[k] : "c"}">${esc(tx.slice(s0, e0))}</mark>`; p = e0; });
  out += esc(tx.slice(p, b));
  const ancla = primeros.length ? primeros[0][0] : a; let o0 = O[u.os[0]];
  offs.forEach(([off, i]) => { if (off <= ancla) o0 = O[i]; });
  const voz = D.teatro ? o0.voz : (o0.voz === "diálogo" ? "diálogo" : "");
  const lug = `${lugarSeg(o0.seg)} · ${pagina(o0.pag)}${voz ? " · " + voz : ""}`;
  const plano = (a > 0 ? "…" : "") + tx.slice(a, b).trim() + (b < tx.length ? "…" : "");
  return `<div class="cita"><span class="lug">${esc(lug)}<button class="mas" data-cita="${esc(JSON.stringify({tx: plano, lug}))}">＋ citar</button></span>${a > 0 ? "…" : ""}${out.trim()}${b < tx.length ? "…" : ""}</div>`;
}
const nombraA = (u, k) => u.os.some(i => PJ_O[i].some(m => m.k === k));
const txtPJ = t => t.replace(/⟦(\d+)⟧/g, (_, k) => pjHTML(+k));
function irPJ(h) {
  SP.tramo = h.tramo !== undefined ? h.tramo : -1; SP.focal = h.focal !== undefined ? h.focal : -1; SP.arista = h.arista || null; SP.vinculo = -1;
  SP.vista = h.vista === "arbol" && HAY_ARBOL ? "arbol" : "red";
  if (h.arista) { SP.focal = h.arista[0];
    if (SP.vista === "arbol") { const i = DP.fam.findIndex(f => (f.a === h.arista[0] && f.b === h.arista[1]) || (f.a === h.arista[1] && f.b === h.arista[0])); if (i >= 0) { SP.vinculo = i; SP.arista = null; } } }
  mostrar("personajes");
}
function enlazarPJ(cont) { cont.querySelectorAll(".pjn").forEach(x => x.onclick = ev => { ev.stopPropagation(); SP.focal = +x.dataset.pj; SP.arista = null; SP.vinculo = -1; pintarPersonajes(); }); }
function pintarPersonajes() {
  const r = redPJ(SP.tramo), svg = $("svg.redpj");
  $(".hallpj").innerHTML = DP.hall.map((h, i) => `<div class="tarjeta"><div class="etq">${esc(h.etq)}</div><div class="frase">${txtPJ(h.txt)}</div>
    <button class="btn link irpj" data-i="${i}">Ver →</button></div>`).join("");
  $$(".hallpj .irpj").forEach(b => b.onclick = () => irPJ(DP.hall[+b.dataset.i].ir));
  enlazarPJ($(".hallpj"));
  $("section[data-v=personajes] .redgrid").classList.toggle("ancho", SP.vista === "arbol");
  $(".red-lienzo").style.display = SP.vista === "red" ? "" : "none"; $(".arbol-lienzo").style.display = SP.vista === "arbol" ? "" : "none";
  $$(".vpj").forEach(b => b.classList.toggle("on", b.dataset.vpj === SP.vista));
  $(".pj-tramo").max = T; $(".pj-tramo").value = SP.tramo + 1;
  $(".pj-vtramo").textContent = SP.tramo < 0 ? "Toda la obra" : `${SP.tramo + 1}/${T} · ${D.tramos[SP.tramo]}`;
  $(".s-pj").value = SP.focal;
  const cmax = Math.max(1, ...r.edges.map(e => e.c)), pmax = Math.max(1, ...DP.lista.map((_, k) => pesoPJ(k, SP.tramo)));
  const veci = new Set(SP.focal >= 0 ? r.edges.filter(e => e.a === SP.focal || e.b === SP.focal).flatMap(e => [e.a, e.b]).concat([SP.focal]) : []);
  const P = k => DP.lista[k].xy ? [70 + 860 * DP.lista[k].xy[0], 44 + 560 * DP.lista[k].xy[1]] : null;
  let s = `<rect x="0" y="0" width="1000" height="640" fill="transparent" class="fondo"/>`;
  r.edges.forEach((e, idx) => { const pa = P(e.a), pb = P(e.b); if (!pa || !pb) return;
    const toca = SP.focal < 0 || e.a === SP.focal || e.b === SP.focal;
    const sel = SP.arista && ((SP.arista[0] === e.a && SP.arista[1] === e.b) || (SP.arista[0] === e.b && SP.arista[1] === e.a));
    const col = sel ? "var(--acc)" : (DP.lista[e.a].g >= 0 && DP.lista[e.a].g === DP.lista[e.b].g ? pjCol(e.a) : GRIS);
    s += `<line data-e="${idx}" x1="${pa[0]}" y1="${pa[1]}" x2="${pb[0]}" y2="${pb[1]}" stroke="${col}" stroke-linecap="round" stroke-width="${(1.2 + 10 * e.c / cmax).toFixed(1)}"
      stroke-opacity="${toca ? (SP.focal >= 0 ? .9 : .5) : .06}"><title>${esc(pjNom(e.a))} y ${esc(pjNom(e.b))}: ${relPJ(e.c, r.nA[e.a], r.nA[e.b])} (${e.c} pasajes)</title></line>`; });
  const orden = [...Array(KP).keys()].sort((a, b) => pesoPJ(a, SP.tramo) - pesoPJ(b, SP.tramo)), etiquetas = [];
  orden.forEach(k => { const p = P(k); if (!p) return;
    const w = pesoPJ(k, SP.tramo), rad = w ? 6 + 22 * Math.sqrt(w / pmax) : 4, L = DP.lista[k];
    const op = SP.focal >= 0 && !veci.has(k) ? .15 : (w ? 1 : .3), prot = L.rol === "protagonista";
    s += `<g class="n" data-k="${k}" opacity="${op}"><circle cx="${p[0]}" cy="${p[1]}" r="${rad.toFixed(1)}" pointer-events="all" fill="${w ? pjCol(k) : "var(--card)"}"
      stroke="${k === SP.focal ? "var(--ink)" : (prot ? "var(--ink)" : (w ? "var(--card)" : pjCol(k)))}" stroke-width="${k === SP.focal ? 4 : prot ? 3 : 1.5}"/>
      <title>${esc(L.n)} · ${L.rol}: ${veces(w)} en esta vista</title></g>`;
    etiquetas.push({k, x: p[0], y: p[1] - rad - 6, w, txt: L.n + (L.pu ? " ⇄" : ""), op}); });
  // etiquetas: primero las de más peso; una que se monta sobre otra se omite (aparece al pasar el cursor o al elegirla)
  const puestas = [];
  etiquetas.sort((a, b) => (b.k === SP.focal) - (a.k === SP.focal) || (veci.has(b.k) - veci.has(a.k)) || b.w - a.w).forEach(e => {
    const ancho = e.txt.length * 7.2, caja = [e.x - ancho / 2, e.y - 13, e.x + ancho / 2, e.y + 3];
    if (puestas.some(q => caja[0] < q[2] && caja[2] > q[0] && caja[1] < q[3] && caja[3] > q[1])) return;
    puestas.push(caja); s += `<text x="${e.x}" y="${e.y}" text-anchor="middle" opacity="${e.op}" pointer-events="none">${esc(e.txt)}</text>`; });
  svg.innerHTML = s;
  svg.querySelector(".fondo").onclick = () => { SP.focal = -1; SP.arista = null; SP.vinculo = -1; pintarPersonajes(); };
  svg.querySelectorAll("g.n").forEach(g => g.onclick = ev => { ev.stopPropagation(); const k = +g.dataset.k; SP.focal = SP.focal === k ? -1 : k; SP.arista = null; SP.vinculo = -1; pintarPersonajes(); });
  svg.querySelectorAll("line[data-e]").forEach(l => l.onclick = ev => { ev.stopPropagation(); const e = r.edges[+l.dataset.e]; SP.arista = [e.a, e.b]; SP.vinculo = -1; pintarPersonajes(); });
  const grupos = [...new Set(DP.lista.map(p => p.g).filter(g => g >= 0))].sort((a, b) => a - b);
  if (SP.vista === "arbol") $(".leypj").innerHTML = `<span><i style="background:#E07B24;border-radius:0;height:3px;width:16px"></i>pareja (punteada: no es matrimonio)</span><span><i style="background:${GRIS};border-radius:0;height:2px;width:16px"></i>padres e hijos</span><span>línea gris punteada: el texto no lo dice en una frase · clic en una línea → la cita</span>${SP.tramo >= 0 ? "<span>huecos: no aparecen en este tramo</span>" : ""}`;
  else $(".leypj").innerHTML = grupos.map(g => { const ns = DP.lista.map((p, k) => p.g === g ? k : -1).filter(k => k >= 0).sort((a, b) => DP.lista[b].s - DP.lista[a].s);
    return `<span><i style="background:${PALETA[g % PALETA.length]}"></i>${ns.slice(0, 3).map(k => esc(pjNom(k))).join(", ")}${ns.length > 3 ? "…" : ""}</span>`; }).join("")
    + `<span>borde oscuro: protagonista · ⇄ puente · tamaño: cuánto se le nombra${DP.teatro ? " o habla" : ""}</span>`;
  if (SP.vista === "arbol") pintarArbol();
  if (SP.vinculo >= 0) panelVinculo(SP.vinculo); else if (SP.arista) panelAristaPJ(r); else if (SP.focal >= 0) panelPJ(r); else panelResumenPJ(r);
  pintarCuando();
  if (S.inv) tablaPJ(r);
}
function panelResumenPJ(r) {
  const roles = ["protagonista", "principal", "secundario", "episódico", "sin apariciones"];
  const amb = DP.amb && DP.amb.res ? `<div class="aviso">Hay nombres compartidos (${DP.amb.formas.slice(0, 4).map(f => "«" + esc(f[0]) + "»").join(", ")}): ${DP.amb.res} menciones se asignaron por contexto${DP.amb.sin ? ` y ${DP.amb.sin} quedaron sin asignar` : ""}. Revisa las citas de esos personajes.</div>` : "";
  const top = r.edges.slice(0, 8);
  $(".panelpj").innerHTML = `<h3>${SP.tramo < 0 ? "El reparto" : esc(D.tramos[SP.tramo])}</h3>
    <div class="sub">${SP.tramo < 0 ? "Haz clic en un personaje para ver con quién se relaciona y dónde aparece; o en una línea para leer los pasajes que comparten." : "Solo este tramo. Los personajes quedan en su lugar para comparar; los que no aparecen se ven huecos."}</div>
    ${roles.filter(ro => DP.lista.some(p => p.rol === ro)).map(ro => `<div class="tit">${ro}</div><div class="sub" style="margin-bottom:4px">${ROL_TXT[ro]}</div>
      <div>${DP.lista.map((p, k) => p.rol === ro ? k : -1).filter(k => k >= 0).sort((a, b) => DP.lista[b].s - DP.lista[a].s).map(k => pjHTML(k) + (DP.lista[k].pu ? " ⇄" : "")).join(", ")}</div>`).join("")}
    <div class="tit">Las relaciones más frecuentes${SP.tramo < 0 ? "" : " aquí"}</div>${top.length ? top.map((e, i) => `<div class="vec" data-i="${i}"><span>${pjHTML(e.a)} y ${pjHTML(e.b)}</span><span>${relPJ(e.c, r.nA[e.a], r.nA[e.b])}<span class="inv"> · ${e.c}</span></span></div>`).join("") : '<div class="sub">Sin relaciones en esta vista.</div>'}
    ${amb}${HAY_ARBOL ? `<div class="aviso">Hay un <b>árbol de familia</b> con ${DP.fam.filter(f => f.t !== "otro").length} vínculos (${DP.fam.filter(f => f.c.length).length} con cita del texto). Ábrelo con el botón «Árbol de familia».</div>` : ""}`;
  $$(".panelpj .vec").forEach(d => d.onclick = () => { const e = top[+d.dataset.i]; SP.arista = [e.a, e.b]; pintarPersonajes(); });
  enlazarPJ($(".panelpj"));
}
function panelPJ(r) {
  const k = SP.focal, L = DP.lista[k], vs = r.edges.filter(e => e.a === k || e.b === k).map(e => ({...e, otro: e.a === k ? e.b : e.a}));
  let us = r.US.filter(u => u.s.includes(k)); const conNombre = us.filter(u => nombraA(u, k)); if (conNombre.length >= 3) us = conNombre;
  const citas = repartir(us, 4).map(u => fragmentoPJ(u, {[k]: "a"})).join("");
  const fam = DP.fam.map((f, i) => ({...f, i})).filter(f => f.a === k || f.b === k);
  const epi = (DP.epi && DP.epi[k]) || [], mot = motivosDe(k), ia = DP.ia && DP.ia[k];
  const cifras = `${veces(L.m)} nombrado${DP.teatro ? ` · ${L.h} parlamentos` : ""} · en ${L.u} pasajes`;
  $(".panelpj").innerHTML = `<h3 style="color:${pjCol(k)}">${esc(L.n)}<span class="rol">${L.rol}${L.pu ? " · puente ⇄" : ""}</span></h3>
    <div class="sub">${ROL_TXT[L.rol]}${L.pu ? "; además une grupos de personajes que casi no se tocan" : ""}.</div>
    <div class="sub" style="margin-top:4px">${cifras}${SP.tramo >= 0 ? ` · ${veces(pesoPJ(k, SP.tramo))} en este tramo` : ""}<span class="inv"> · formas: ${esc(L.f.join(", "))} · intermediación ${L.b}</span></div>
    ${L.amb ? `<div class="aviso">${L.amb} de sus menciones vienen de un nombre compartido y se le asignaron por contexto: revisa las citas.</div>` : ""}
    ${tiraPJ([k])}<div class="tit">Se relaciona con</div>
    ${vs.length ? vs.slice(0, 10).map((v, i) => `<div class="vec" data-i="${i}"><span>${pjHTML(v.otro)}</span><span>${relPJ(v.c, r.nA[k], r.nA[v.otro])}<span class="inv"> · ${v.c} pasajes</span></span></div>`).join("") : '<div class="sub">No coincide con otros personajes en esta vista.</div>'}
    ${epi.length ? `<div class="tit">Cómo lo llama la obra</div><div class="chips-epi">${epi.map((e, i) => `<span class="chip" data-epi="${i}">«${esc(e[0])}» <span class="sub">×${e[1]} · ${e[2] === e[3] ? elSeg(e[2]) : `${lugarSeg(e[2])}–${D.segmentos[e[3]].e}`}</span></span>`).join("")}</div><div class="epi-citas"></div>` : ""}
    ${fam.length ? `<div class="tit">Familia y vínculos</div>${fam.map(f => `<div class="famrow" data-f="${f.i}">${pjHTML(f.a)} <span class="sub">${esc(f.e)} de</span> ${pjHTML(f.b)} <span class="sub">${f.c.length ? "✔ con cita" : "· sin frase"}</span></div>`).join("")}${HAY_ARBOL && SP.vista !== "arbol" ? '<button class="btn link verarbol" style="margin:4px 6px">Ver en el árbol →</button>' : ""}` : ""}
    ${mot.length ? `<div class="tit">Motivos que lo acompañan</div><div class="sub" style="margin-bottom:4px">Motivos que aparecen en sus pasajes más de lo esperado: material para pensar su papel simbólico.</div><div class="chips-mot">${mot.map((x, i) => `<span class="chip" data-mot="${i}">${motHTML(x.m)} <span class="sub">${x.l >= 2.5 ? "muy ligado" : x.l >= 1.6 ? "ligado" : "algo ligado"}<span class="inv"> · ${x.n} · ×${x.l.toFixed(1)}</span></span></span>`).join("")}</div><div class="mot-citas"></div>` : ""}
    ${ia ? `<div class="tit">Lectura propuesta por la IA <span class="sub" style="text-transform:none;letter-spacing:0">(hipótesis: compruébala en las citas)</span></div><div class="ia">
      <p><b>En la trama:</b> ${refsIA(ia.literal)}</p><p><b>Simbólicamente:</b> ${refsIA(ia.simbolico)}</p>${ia.cambia ? `<p><b>¿Cambia?</b> ${refsIA(ia.cambia)}</p>` : ""}
      ${ia.fuera && ia.fuera !== "nada" ? `<p class="sub"><b>Fuera de las citas (no verificado):</b> ${esc(ia.fuera)}</p>` : ""}</div><div class="ia-citas"></div>` : ""}
    <div class="tit">Dónde aparece</div>${citas || '<div class="sub">No aparece en este tramo.</div>'}
    <button class="btn anotar" style="margin-top:6px">✎ Anotar sobre ${esc(L.n)}</button>`;
  $$(".panelpj .chips-epi .chip").forEach(c => c.onclick = () => { const e = epi[+c.dataset.epi], on = !c.classList.contains("on");
    $$(".panelpj .chips-epi .chip").forEach(x => x.classList.remove("on")); c.classList.toggle("on", on);
    $(".panelpj .epi-citas").innerHTML = on ? e[4].map(i => fragmentoPJ({os: [i]}, {[k]: "a"}, 260)).join("") : ""; conectarCitas($(".panelpj .epi-citas"), L.n); });
  $$(".panelpj .famrow").forEach(d => d.onclick = ev => { if (ev.target.closest(".pjn")) return; SP.vinculo = +d.dataset.f; pintarPersonajes(); });
  const va = $(".panelpj .verarbol"); if (va) va.onclick = () => { SP.vista = "arbol"; pintarPersonajes(); };
  $$(".panelpj .chips-mot .chip").forEach(c => c.onclick = () => { const x = mot[+c.dataset.mot], on = !c.classList.contains("on");
    $$(".panelpj .chips-mot .chip").forEach(y => y.classList.remove("on")); c.classList.toggle("on", on);
    const cand = [], conK = [];
    PJ_U.forEach(u => { if (!u.s.includes(k)) return; u.os.forEach(i => { if ((O[i].m || []).includes(x.m)) { cand.push(i); if (PJ_O[i].some(z => z.k === k)) conK.push(i); } }); });
    const os = repartir(conK.length >= 3 ? conK : cand, 3);
    $(".panelpj .mot-citas").innerHTML = on ? os.map(i => fragmentoPJ({os: [i]}, {[k]: "a"}, 360, {[x.m]: "b"})).join("") : "";
    conectarCitas($(".panelpj .mot-citas"), `${L.n} y «${nombre(x.m)}»`); });
  $$(".panelpj .ia .ref").forEach(rf => rf.onclick = () => { const c = (ia.citas || []).find(z => z[0] === +rf.dataset.n); if (!c) return;
    $(".panelpj .ia-citas").innerHTML = fragmentoPJ({os: [c[1]]}, {[k]: "a"}, 400); conectarCitas($(".panelpj .ia-citas"), L.n); });
  $$(".panelpj .vec").forEach(d => d.onclick = () => { const v = vs[+d.dataset.i]; SP.arista = [k, v.otro]; pintarPersonajes(); });
  enlazarPJ($(".panelpj")); conectarCitas($(".panelpj"), L.n);
  $(".panelpj .anotar").onclick = () => abrirNota(L.n);
}
function motivosDe(k) {
  if (!K()) return [];
  const base = new Map(), c = new Map(); let nk = 0;
  PJ_U.forEach(u => { const ms = new Set(); u.os.forEach(i => (O[i].m || []).forEach(m => ms.add(m)));
    ms.forEach(m => base.set(m, (base.get(m) || 0) + 1));
    if (u.s.includes(k)) { nk++; ms.forEach(m => c.set(m, (c.get(m) || 0) + 1)); } });
  return [...c.entries()].map(([m, n]) => ({m, n, l: n / (nk * base.get(m) / PJ_U.length)}))
    .filter(x => x.n >= 3 && x.l >= 1.3).sort((a, b) => b.l * Math.sqrt(b.n) - a.l * Math.sqrt(a.n)).slice(0, 6);
}
const refsIA = t => esc(t || "").replace(/\[(\d+)\]/g, (_, n) => `<span class="ref" data-n="${n}">[${n}]</span>`);
function pintarArbol() {
  const A = DP.arbol, ks = Object.keys(A).map(Number), svg = $("svg.arbolpj");
  const maxL = Math.min(22, Math.max(...ks.map(k => pjNom(k).length))), U = Math.max(118, maxL * 6.9 + 46), FILA = 88;
  const X = k => 64 + U / 2 + A[k][1] * U, Y = k => 42 + A[k][0] * FILA;
  const W = Math.max(...ks.map(X)) + U / 2 + 20, H = Math.max(...ks.map(Y)) + 46;
  const cerca = new Set();
  if (SP.focal >= 0) { cerca.add(SP.focal); DP.fam.forEach(f => { if (f.a === SP.focal) cerca.add(f.b); if (f.b === SP.focal) cerca.add(f.a); }); }
  const vf = SP.vinculo >= 0 ? DP.fam[SP.vinculo] : null; if (vf) { cerca.add(vf.a); cerca.add(vf.b); }
  const tenue = k => cerca.size && !cerca.has(k);
  let s = `<rect x="0" y="0" width="${W}" height="${H}" fill="transparent" class="fondo"/>`;
  const gens = [...new Set(ks.map(k => A[k][0]))].sort((a, b) => a - b);
  gens.forEach(g => s += `<text x="10" y="${42 + g * FILA + 4}" font-size="11" fill="var(--sub)" style="font-size:11px">${g + 1}.ª gen.</text>`);
  // padres e hijos, agrupados por pareja de padres
  const grupos = new Map();
  DP.fam.forEach((f, i) => { if ((f.t === "hijo" || f.t === "adoptivo") && f.a in A && f.b in A) { if (!grupos.has(f.a)) grupos.set(f.a, []); grupos.get(f.a).push(i); } });
  const porPadres = new Map();
  grupos.forEach((is, c) => { const key = is.map(i => DP.fam[i].b).sort((a, b) => a - b).join(","); if (!porPadres.has(key)) porPadres.set(key, []); porPadres.get(key).push([c, is]); });
  porPadres.forEach((cs, key) => { const ps = key.split(",").map(Number), ax = ps.reduce((t, p) => t + X(p), 0) / ps.length, jy = Math.max(...ps.map(Y)) + 22;
    const by = Math.min(...cs.map(([c]) => Y(c))) - 22, xs = cs.map(([c]) => X(c)).concat([ax]);
    const op = cerca.size && !ps.some(p => cerca.has(p)) && !cs.some(([c]) => cerca.has(c)) ? .2 : 1;
    let d = ps.map(p => `M${X(p)},${Y(p) + 13} V${jy}`).join(" ") + (ps.length > 1 ? ` M${Math.min(...ps.map(X))},${jy} H${Math.max(...ps.map(X))}` : "");
    s += `<path d="${d} M${ax},${jy} V${by} M${Math.min(...xs)},${by} H${Math.max(...xs)}" fill="none" stroke="${GRIS}" stroke-width="1.6" opacity="${op}"/>`;
    cs.forEach(([c, is]) => { const i = is.find(j => DP.fam[j].c.length) ?? is[0], f = DP.fam[i], sel = vf && is.includes(SP.vinculo);
      const sin = is.every(j => !DP.fam[j].c.length), adop = is.some(j => DP.fam[j].t === "adoptivo");
      s += `<line class="v" data-f="${i}" x1="${X(c)}" y1="${by}" x2="${X(c)}" y2="${Y(c) - 13}" stroke="${sel ? "var(--acc)" : GRIS}" stroke-width="${sel ? 4 : 2.2}" ${sin || adop ? 'stroke-dasharray="3 4"' : ""} opacity="${cerca.size && !cerca.has(c) ? .2 : 1}"><title>${esc(pjNom(c))}: ${esc(f.e)} de ${is.map(j => esc(pjNom(DP.fam[j].b))).join(" y ")}${sin ? " (sin frase en el texto)" : ""}</title></line>`; }); });
  DP.fam.forEach((f, i) => { if (!(f.t === "pareja" || f.t === "amante") || !(f.a in A) || !(f.b in A)) return;
    const x1 = X(f.a), y1 = Y(f.a), x2 = X(f.b), y2 = Y(f.b), sel = SP.vinculo === i, op = cerca.size && !(cerca.has(f.a) && cerca.has(f.b)) ? .2 : 1;
    const est = `stroke="${sel ? "var(--acc)" : f.c.length ? "#E07B24" : GRIS}" stroke-width="${sel ? 4.5 : 2.6}" ${f.t === "amante" ? 'stroke-dasharray="6 5"' : (!f.c.length ? 'stroke-dasharray="3 4"' : "")} fill="none" opacity="${op}"`;
    const d = y1 === y2 && Math.abs(A[f.a][1] - A[f.b][1]) <= 1.05 ? `M${x1},${y1} L${x2},${y2}` : y1 === y2 ? `M${x1},${y1 - 13} C${x1},${y1 - 44} ${x2},${y2 - 44} ${x2},${y2 - 13}` : `M${x1},${y1} C${x1},${(y1 + y2) / 2} ${x2},${(y1 + y2) / 2} ${x2},${y2}`;
    s += `<path class="v" data-f="${i}" d="${d}" ${est}><title>${esc(pjNom(f.a))}: ${esc(f.e)} de ${esc(pjNom(f.b))}</title></path>`; });
  ks.forEach(k => { const nm = pjNom(k).length > 22 ? pjNom(k).slice(0, 21) + "…" : pjNom(k), w = Math.max(64, nm.length * 6.9 + 16), L = DP.lista[k];
    const presente = SP.tramo < 0 || pesoPJ(k, SP.tramo) > 0, op = tenue(k) ? .25 : presente ? 1 : .45;
    s += `<g class="n" data-k="${k}" opacity="${op}"><rect x="${X(k) - w / 2}" y="${Y(k) - 13}" width="${w}" height="26" rx="13" fill="${presente ? "var(--card)" : "var(--soft)"}" stroke="${pjCol(k)}" stroke-width="${k === SP.focal ? 4 : L.rol === "protagonista" ? 3 : 1.6}" ${presente ? "" : 'stroke-dasharray="3 3"'}/>
      <text x="${X(k)}" y="${Y(k) + 4.5}" text-anchor="middle" font-weight="${k === SP.focal || L.rol === "protagonista" ? 700 : 400}">${esc(nm)}</text><title>${esc(L.n)} · ${L.rol}${SP.tramo >= 0 ? ` · ${veces(pesoPJ(k, SP.tramo))} en este tramo` : ""}</title></g>`; });
  svg.setAttribute("viewBox", `0 0 ${W} ${H}`); svg.setAttribute("width", W); svg.setAttribute("height", H); svg.innerHTML = s;
  svg.querySelector(".fondo").onclick = () => { SP.focal = -1; SP.vinculo = -1; SP.arista = null; pintarPersonajes(); };
  svg.querySelectorAll("g.n").forEach(g => g.onclick = ev => { ev.stopPropagation(); const k = +g.dataset.k; SP.focal = SP.focal === k ? -1 : k; SP.vinculo = -1; SP.arista = null; pintarPersonajes(); });
  const alPanel = () => { const pn = $(".panelpj"), r0 = pn.getBoundingClientRect(); if (r0.top > window.innerHeight - 120) pn.scrollIntoView({behavior: "smooth", block: "start"}); };
  svg.querySelectorAll(".v").forEach(l => l.onclick = ev => { ev.stopPropagation(); SP.vinculo = +l.dataset.f; SP.arista = null; pintarPersonajes(); alPanel(); });
  svg.querySelectorAll("g.n").forEach(g => g.addEventListener("click", alPanel));
  if (SP.focal >= 0 && SP.focal in A) { const cont = $(".arbol-lienzo"), x = X(SP.focal) * cont.clientWidth / Math.max(cont.clientWidth, W);
    if (W > cont.clientWidth) cont.scrollLeft = Math.max(0, X(SP.focal) - cont.clientWidth / 2); }
}
function panelVinculo(i) {
  const f = DP.fam[i], a = f.a, b = f.b, ref = `${pjNom(a)} y ${pjNom(b)} (${f.e})`;
  let citas = f.c.map(o => fragmentoPJ({os: [o]}, {[a]: "a", [b]: "b"}, 380)).join(""), extra = "";
  if (!f.c.length) { const e = redPJ(-1).co.get(Math.min(a, b) * 1000 + Math.max(a, b)), us = e ? e.us.map(j => redPJ(-1).US[j]).filter(u => nombraA(u, a) && nombraA(u, b)) : [];
    extra = `<div class="aviso">No encontré una frase que lo diga con una palabra de parentesco junto a los dos nombres. Puede estar dicho con pronombres, apodos o a lo largo de varias frases.</div>` +
      (us.length ? `<div class="tit">Pasajes donde aparecen juntos</div>` + repartir(us, 3).map(u => fragmentoPJ(u, {[a]: "a", [b]: "b"})).join("") : `<div class="sub">Tampoco coinciden en un mismo pasaje.</div>`); }
  $(".panelpj").innerHTML = `<h3 style="font-size:20px">${pjHTML(a)} <span class="sub" style="font-family:system-ui">${esc(f.e)} de</span> ${pjHTML(b)}</h3>
    <div class="sub">${DP.fam_origen === "escrita" ? "Vínculo del reparto (escrito en el perfil o por ti en Colab, paso 3b)." : "Vínculo propuesto a partir de una frase del texto."}</div>
    ${f.c.length ? `<div class="tit">Dónde lo dice el texto</div>${citas}` : extra}
    <div style="display:flex;gap:8px;flex-wrap:wrap;margin-top:8px"><button class="btn anotar">✎ Anotar</button><button class="btn fa">Ficha: ${esc(pjNom(a))}</button><button class="btn fb">Ficha: ${esc(pjNom(b))}</button></div>`;
  enlazarPJ($(".panelpj")); conectarCitas($(".panelpj"), ref);
  $(".panelpj .anotar").onclick = () => abrirNota(ref);
  $(".panelpj .fa").onclick = () => { SP.vinculo = -1; SP.focal = a; pintarPersonajes(); };
  $(".panelpj .fb").onclick = () => { SP.vinculo = -1; SP.focal = b; pintarPersonajes(); };
}
function panelAristaPJ(r) {
  const [a, b] = SP.arista, e = r.co.get(Math.min(a, b) * 1000 + Math.max(a, b));
  const us = e ? e.us.map(i => r.US[i]) : [], ambos = us.filter(u => nombraA(u, a) && nombraA(u, b));
  const citas = repartir(ambos.length >= 3 ? ambos : us, 6).map(u => fragmentoPJ(u, {[a]: "a", [b]: "b"})).join("");
  const ref = `${pjNom(a)} y ${pjNom(b)}`;
  $(".panelpj").innerHTML = `<h3><mark class="a">${esc(pjNom(a))}</mark> y <mark class="b">${esc(pjNom(b))}</mark></h3>
    <div class="sub">${e ? `${relPJ(e.c, r.nA[a], r.nA[b])}: comparten ${e.c} pasajes${SP.tramo >= 0 ? " en este tramo" : ""}` : "No coinciden en esta vista"}</div>
    ${tiraPJ([a, b])}<div class="tit">Pasajes que comparten</div>${citas || '<div class="sub">—</div>'}
    <div style="display:flex;gap:8px;flex-wrap:wrap;margin-top:6px"><button class="btn anotar">✎ Anotar</button><button class="btn volver">← volver a ${esc(pjNom(a))}</button></div>`;
  conectarCitas($(".panelpj"), ref);
  $(".panelpj .anotar").onclick = () => abrirNota(ref);
  $(".panelpj .volver").onclick = () => { SP.arista = null; SP.focal = a; pintarPersonajes(); };
}
function pintarCuando() {
  const ks = [...Array(KP).keys()].filter(k => PJ_POS[k].length).sort((a, b) => PJ_POS[a][0] - PJ_POS[b][0]);
  const fila = 19, izq = 190, W = 1000 - izq - 6, H = ks.length * fila + 18, svg = $("svg.cuando");
  svg.setAttribute("viewBox", `0 0 1000 ${H}`);
  let s = "";
  if (SP.tramo >= 0) { const x0 = TR_INI[SP.tramo] / NTOK, x1 = (TR_INI[SP.tramo] + TR_LARGO[SP.tramo]) / NTOK; s += `<rect x="${izq + W * x0}" y="0" width="${W * (x1 - x0)}" height="${H - 16}" fill="var(--acc)" opacity=".08"/>`; }
  TR_INI.slice(1).forEach(p => s += `<line x1="${izq + W * p / NTOK}" x2="${izq + W * p / NTOK}" y1="0" y2="${H - 16}" stroke="var(--line)" stroke-width="1.5"/>`);
  ks.forEach((k, r) => { const y = r * fila, ps = PJ_POS[k], foco = SP.focal < 0 || SP.focal === k;
    s += `<g class="fila-pj" data-k="${k}" opacity="${foco ? 1 : .35}"><rect class="fondo-fila" x="0" y="${y}" width="1000" height="${fila}" fill="transparent"/>
      <text x="${izq - 10}" y="${y + 13}" text-anchor="end" font-weight="${SP.focal === k ? 700 : 400}">${esc(pjNom(k))}</text>
      <rect x="${izq + W * ps[0] / NTOK}" y="${y + 8.5}" width="${Math.max(1, W * (ps[ps.length - 1] - ps[0]) / NTOK)}" height="2" fill="${pjCol(k)}" opacity=".35"/>`;
    const paso = Math.max(1, Math.floor(ps.length / 500));
    for (let i = 0; i < ps.length; i += paso) s += `<rect x="${(izq + W * ps[i] / NTOK).toFixed(1)}" y="${y + 3}" width="1.6" height="13" fill="${pjCol(k)}"/>`;
    s += `</g>`; });
  s += `<text x="${izq}" y="${H - 3}" fill="var(--sub)" font-size="11">inicio</text><text x="1000" y="${H - 3}" fill="var(--sub)" font-size="11" text-anchor="end">final</text>`;
  svg.innerHTML = s;
  svg.querySelectorAll(".fila-pj").forEach(g => g.onclick = () => { SP.focal = +g.dataset.k; SP.arista = null; pintarPersonajes(); $("svg.redpj").scrollIntoView({behavior: "smooth", block: "nearest"}); });
}
function tablaPJ(r) {
  const filas = [...Array(KP).keys()].sort((a, b) => DP.lista[b].s - DP.lista[a].s);
  $(".tablapj").innerHTML = `<table><tr><th>personaje</th><th>rol</th><th class="num">menciones</th>${DP.teatro ? '<th class="num">parlamentos</th>' : ""}<th class="num">pasajes</th><th class="num">vínculos</th><th class="num">intermediación</th><th class="num">${SP.tramo < 0 ? "" : "en el tramo"}</th></tr>` +
    filas.map(k => { const L = DP.lista[k]; return `<tr><td>${esc(L.n)}</td><td>${L.rol}${L.pu ? " ⇄" : ""}</td><td class="num">${L.m}</td>${DP.teatro ? `<td class="num">${L.h}</td>` : ""}<td class="num">${L.u}</td><td class="num">${L.gr}</td><td class="num">${L.b}</td><td class="num">${SP.tramo < 0 ? "" : pesoPJ(k, SP.tramo)}</td></tr>`; }).join("") + `</table>`;
}
function iniciarPJ() {
  if (!DP) return;
  prepararPJ();
  $('nav button[data-v="personajes"]').style.display = "";
  if (HAY_ARBOL) { $(".vpjs").style.display = ""; $$(".vpj").forEach(b => b.onclick = () => { SP.vista = b.dataset.vpj; pintarPersonajes(); }); }
  $(".s-pj").innerHTML = `<option value="-1">(ninguno)</option>` + [...Array(KP).keys()].filter(k => PJ_POS[k].length).sort((a, b) => pjNom(a).localeCompare(pjNom(b), "es")).map(k => `<option value="${k}">${esc(pjNom(k))}</option>`).join("");
  $(".s-pj").onchange = ev => { SP.focal = +ev.target.value; SP.arista = null; pintarPersonajes(); };
  $(".pj-tramo").oninput = ev => { SP.tramo = +ev.target.value - 1; SP.arista = null; pintarPersonajes(); };
  $(".pj-play").onclick = () => { if (relojPJ) { clearInterval(relojPJ); relojPJ = null; $(".pj-play").textContent = "▶"; return; }
    $(".pj-play").textContent = "❚❚"; SP.arista = null; SP.tramo = SP.tramo >= T - 1 ? 0 : SP.tramo + 1; pintarPersonajes();
    relojPJ = setInterval(() => { if (SP.tramo >= T - 1) { clearInterval(relojPJ); relojPJ = null; $(".pj-play").textContent = "▶"; return; } SP.tramo++; pintarPersonajes(); }, 1700); };
  $(".csvpj").onclick = () => { const r = redPJ(SP.tramo);
    descargar(`personajes_${D.titulo}.csv`, "personaje,rol,puente,menciones,parlamentos,pasajes,vinculos,intermediacion\n" + DP.lista.map(L => `"${L.n}",${L.rol},${L.pu ? "sí" : "no"},${L.m},${L.h},${L.u},${L.gr},${L.b}`).join("\n")
      + "\n\npersonaje_a,personaje_b,pasajes_juntos\n" + r.edges.map(e => `"${pjNom(e.a)}","${pjNom(e.b)}",${e.c}`).join("\n"), "text/csv"); };
}

// ---------- navegación ----------
let vista = "hallazgos", reloj = null;
function mostrar(v) {
  vista = v; $$("nav button").forEach(b => b.classList.toggle("on", b.dataset.v === v)); $$(".vista").forEach(s => s.classList.toggle("on", s.dataset.v === v));
  if (v === "hallazgos") pintarHallazgos(); if (v === "red") pintarRed(); if (v === "personajes") pintarPersonajes(); if (v === "motivos") pintarFichas(); if (v === "taller") pintarTaller(); if (v === "cuaderno") pintarCuaderno();
  if (v !== "red" && reloj) { clearInterval(reloj); reloj = null; $(".play").textContent = "▶"; }
  if (v !== "personajes" && relojPJ) { clearInterval(relojPJ); relojPJ = null; $(".pj-play").textContent = "▶"; }
}
function pintarTodo() {
  $(".s-focal").innerHTML = `<option value="-1">(ninguno)</option>` + [...Array(K()).keys()].sort((a, b) => nombre(a).localeCompare(nombre(b), "es")).map(k => `<option value="${k}">${esc(nombre(k))}</option>`).join("");
  $(".datos").textContent = `${NTOK.toLocaleString("es")} palabras · ${D.segmentos.length} ${D.teatro ? "escenas" : "segmentos"} · ${T} tramos · ${K()} motivos${DP ? ` · ${DP.lista.filter(p => p.rol !== "sin apariciones").length} personajes` : ""}`;
  mostrar(vista);
}
let tAviso; function aviso(t) { const e = $(".toast"); e.textContent = t; e.classList.add("on"); clearTimeout(tAviso); tAviso = setTimeout(() => e.classList.remove("on"), 1800); }

// ---------- inicio ----------
$("h1").textContent = D.titulo; $(".autor").textContent = D.autor;
$$("nav button").forEach(b => b.onclick = () => mostrar(b.dataset.v));
$(".interruptor").onclick = () => { S.inv = !S.inv; R.classList.toggle("investigar", S.inv); $(".interruptor").classList.toggle("on", S.inv); mostrar(vista); };
$(".s-unidad").innerHTML = ["oracion", "parrafo", "ventana"].map(u => `<option value="${u}">${NOMBRE_UNIDAD[u]}</option>`).join(""); $(".s-unidad").value = S.unidad;
$(".s-unidad").onchange = ev => { S.unidad = ev.target.value; recalcular(); pintarTodo(); };
$(".s-top").oninput = ev => { S.top = +ev.target.value; $(".v-top").textContent = S.top; CACHE = {}; BASE = red(S.unidad, -1); const otras = ["oracion", "parrafo", "ventana"].map(u => new Set(red(u, -1).edges.map(e => e.a * 1000 + e.b))); BASE.edges.forEach(e => e.sol = otras.filter(s => s.has(e.a * 1000 + e.b)).length); pintarRed(); };
$(".s-tramo").oninput = ev => { S.tramo = +ev.target.value - 1; S.arista = null; pintarRed(); };
$(".s-focal").onchange = ev => { S.focal = +ev.target.value; S.arista = null; pintarRed(); };
$(".play").onclick = () => { if (reloj) { clearInterval(reloj); reloj = null; $(".play").textContent = "▶"; return; }
  $(".play").textContent = "❚❚"; S.arista = null; S.tramo = S.tramo >= T - 1 ? 0 : S.tramo + 1; pintarRed();
  reloj = setInterval(() => { if (S.tramo >= T - 1) { clearInterval(reloj); reloj = null; $(".play").textContent = "▶"; return; } S.tramo++; pintarRed(); }, 1700); };
$(".csv").onclick = () => { const r = vistaRed(); descargar(`conexiones_${D.titulo}.csv`, "motivo_a,motivo_b,juntos,fuerza,lift\n" + r.edges.map(e => `"${nombre(e.a)}","${nombre(e.b)}",${e.c},${e.f.toFixed(4)},${e.l.toFixed(2)}`).join("\n"), "text/csv"); };
$(".nuevo").onclick = () => { S.borrador.push({n: "", f: "", t: "propio"}); pintarTaller(); const fs = $$(".fila .nom"); fs[fs.length - 1].focus(); marcarCambios(); };
$(".aplicar").onclick = aplicar;
$(".restaurar").onclick = () => { S.borrador = D.motivos.map(m => ({n: m.n, f: m.f.join(", "), t: m.t})); pintarTaller(); marcarCambios("Diccionario inicial cargado: pulsa Aplicar."); };
$(".bajar-dic").onclick = () => descargar(`motivos_${D.titulo}.json`, JSON.stringify({obra: D.titulo, motivos: S.motivos}, null, 1), "application/json");
$(".subir-dic").onchange = ev => { const f = ev.target.files[0]; if (!f) return; f.text().then(t => { try { const j = JSON.parse(t); S.borrador = (j.motivos || j).map(m => ({n: m.n, f: (m.f || []).join(", "), t: m.t || "propio"})); pintarTaller(); marcarCambios("Diccionario cargado: pulsa Aplicar."); } catch (e) { marcarCambios("Ese archivo no es un diccionario válido."); } }); };
$(".bajar-notas").onclick = () => descargar(`notas_${D.titulo}.md`, notasMarkdown(), "text/markdown");
$(".nueva-nota").onclick = () => { S.notas.unshift({id: Date.now(), ref: "Nota libre", texto: "", citas: [], fecha: new Date().toLocaleDateString("es")}); guardar("notas", S.notas); pintarCuaderno(); };
iniciarPJ(); recalcular(); pintarTodo();
})();
</script>
</div>
"""
print("Diseño listo.")

## 1 · Sube la obra y revisa la limpieza
Al ejecutar la celda se abre un cuadro para elegir el PDF (o escribe su nombre en `archivo` si ya está en Colab). La herramienta:
1. quita **cabeceras, pies, números de página, índices, adornos y notas editoriales** (lo que se repite en muchas páginas, las URL, el colofón);
2. une las **palabras partidas** al final de línea;
3. busca los **títulos de capítulo** (`CAPÍTULO 3`, `III`, `Acto I / Escena II`…) y se queda con la secuencia 1, 2, 3… más larga, para que un número suelto o el índice no se confundan con capítulos; lo anterior al primer capítulo (portada, prólogo, dedicatoria) se descarta;
4. separa **frases**, **diálogo y narración** (o parlamentos, en teatro) y guarda la **página** de cada frase: la impresa si el PDF la trae al pie, o la del PDF si no.

Si el informe muestra un problema, usa los campos del formulario (frase donde empieza o termina la obra, o forzar el tipo de estructura) y vuelve a ejecutar.

In [ ]:
#@title 1 · Sube la obra y revisa la limpieza
archivo    = ""      #@param {type:"string"}
perfil     = "auto"  #@param ["auto", "ninguno", "El principito", "La metamorfosis", "El mundo es ancho y ajeno", "Hamlet", "Cien años de soledad"]
#@markdown **Solo si el informe muestra un problema** (déjalos vacíos la primera vez):
frase_inicio = ""    #@param {type:"string"}
frase_final  = ""    #@param {type:"string"}
estructura   = "auto" #@param ["auto", "capitulo", "romano", "numero", "parte", "titulos", "teatro", "bloques"]

if not archivo:
    pdfs = sorted(glob.glob("*.pdf"), key=os.path.getmtime, reverse=True)
    if EN_COLAB and not pdfs:
        print("Elige el PDF de la obra:"); pdfs = [n for n in files.upload() if n.lower().endswith(".pdf")]
    if not pdfs: raise FileNotFoundError("Falta el PDF de la obra (es obligatorio).")
    archivo = pdfs[0]
print("Leyendo", archivo, "…")
PAGINAS = leer_pdf(archivo)
if sum(len(l) for p in PAGINAS for l in p) < 40 * len(PAGINAS):
    print("⚠ Casi no hay texto: el PDF parece escaneado (imágenes). Este prototipo solo lee PDF con texto seleccionable.")

plano = lambda t: re.sub(r"[^a-z0-9ñ]+", " ", sin_tildes(t.lower()))
PERFIL = None
if perfil == "auto":
    donde = plano(archivo) + " " + plano(" ".join(l for p in PAGINAS[:15] for l in p))
    PERFIL = next((p for p, c in PERFILES.items() if any(plano(k) in donde for k in c["claves"])), None)
elif perfil != "ninguno":
    PERFIL = perfil
CONF = dict(PERFILES.get(PERFIL, {}))
CONF.update({k: v for k, v in {"inicio": frase_inicio, "final": frase_final,
             "estructura": None if estructura == "auto" else estructura}.items() if v})

OBRA, INFORME = limpiar(PAGINAS, CONF)
O = armar_oraciones(OBRA)
TRAMOS = armar_tramos(OBRA, CONF, O)
TITULO = PERFIL or re.sub(r"(?<=[a-zé])(?=[A-Z])", " ", re.sub(r"[_\-]+", " ", os.path.splitext(os.path.basename(archivo))[0])).strip()
AUTOR = CONF.get("autor", "")
SEG_ETQ = {s["num"]: s["etiqueta"] for s in OBRA["segmentos"]}; ETQ_SEG_ACTUAL = ETQ_SEG[OBRA["modo"]].lower()
display(HTML(informe_html(OBRA, INFORME, O, TRAMOS, PERFIL)))

**Qué sale y qué no.**
- ✅ Funcionó sin ajustes con las cuatro ediciones: 27 capítulos en *El principito*, 3 en *La metamorfosis* (con su letra decorativa y sus números «~ 12 ~»), 24 en *El mundo es ancho y ajeno* (incluido un título dañado, «APÍTULO 21», y dos páginas del libro por hoja) 85 escenas en *Hamlet* y los 20 capítulos (en romanos) de *Cien años de soledad*, que se cita por página impresa.
- ✅ Citas por **página impresa** en *El principito* y *La metamorfosis*; por **página del PDF** en las otras dos (en Alegría, cada hoja del PDF trae dos páginas del libro, así que hay que convertir la cita).
- ✅ También reconoce capítulos con **ordinales** («CAPÍTULO PRIMERO»), con **números sueltos** («1», «2»…, sin confundirlos con los números de página), con **título sin número** («EL DESPERTAR», «LA MANZANA») y en minúsculas («Capítulo 3»); quita **notas al pie**, cabeceras alternas («AUTOR / TÍTULO») y **ligaduras** tipográficas (ﬁ, ﬂ). Esto se probó con cinco maquetaciones simuladas a partir de textos reales.
- ⚠️ Una obra **sin títulos de capítulo** se divide en bloques de unas 2 500 palabras: la red funciona, pero los tramos no son capítulos reales.
- ⚠️ Prólogos o estudios **dentro** del cuerpo del libro (después del primer capítulo) no se detectan: recórtalos con `frase_inicio` o `frase_final`.
- ❌ PDF **escaneados** (imágenes sin texto) o maquetados a **dos columnas** no funcionan en este prototipo.

## 2 · Elige el punto de partida de los motivos
Un **motivo** es un grupo de palabras que nombran lo mismo: *agua, sed, pozo* → «agua y pozo». Hay tres formas de empezar:
- **Perfil**: las obras probadas traen un diccionario inicial pensado para discutir en clase (no es la respuesta).
- **Sugerencias automáticas**: el programa elige entre cuatro listas: palabras que **recorren toda la obra**, que aparecen **en ráfagas**, que son **propias de esta obra** frente al español general y **nombres**.
- **IA (OpenAI)**: un modelo agrupa las sugerencias en familias de motivos (necesitas tu clave; ver paso 5).

Sea cual sea el punto de partida, los motivos se cambian después, dentro de la lectura (pestaña **Taller**).

In [ ]:
#@title 2 · Elige el punto de partida de los motivos
punto_de_partida = "perfil si existe; si no, sugerencias" #@param ["perfil si existe; si no, sugerencias", "sugerencias automáticas", "IA (OpenAI)"]
modelo_ia = "gpt-6-luna" #@param {type:"string"}

CAND, LISTAS, FREC = candidatos(O, CONF)
if punto_de_partida == "IA (OpenAI)" and (CLIENTE or conectar_ia(modelo_ia)):
    MOTIVOS = ia_sugerir_motivos(LISTAS, O, FREC, TITULO)
    MOTIVOS += [m for m in diccionario_automatico(LISTAS, {"nombre": 3}) if m["nombre"] not in {x["nombre"] for x in MOTIVOS}]
    origen = f"propuestos por la IA ({MODELO_IA}) + 3 nombres"
elif punto_de_partida.startswith("perfil") and CONF.get("motivos"):
    MOTIVOS, origen = leer_diccionario(CONF["motivos"], "perfil"), f"diccionario del perfil «{PERFIL}»"
else:
    MOTIVOS, origen = diccionario_automatico(LISTAS), "elegidos automáticamente entre las sugerencias"
MOTIVOS = [m for m in MOTIVOS if contar_motivo(m, FREC)]

e = html.escape
filas = "".join(f'<span style="display:inline-block;border:1px solid #ddd5c6;border-radius:14px;padding:2px 10px;margin:3px">'
                f'<b>{e(m["nombre"])}</b> <span style="color:#888">{contar_motivo(m, FREC)}</span></span>' for m in MOTIVOS)
listas = "".join(f'<div style="margin:4px 0"><b>{t}:</b> {e(", ".join(LISTAS[l].index[:12])) or "—"}</div>' for l, t in
                 [("recurrente", "Recorren toda la obra"), ("concentrado", "Aparecen en ráfagas"),
                  ("característico", "Propias de esta obra" + ("" if zipf_frequency else " (requiere wordfreq)")), ("nombre", "Nombres")])
display(HTML(f'<div style="font-family:system-ui;font-size:14px;line-height:1.5;max-width:900px"><h3 style="margin:4px 0">'
             f'{len(MOTIVOS)} motivos para empezar</h3><div style="color:#777">{e(origen)}. Se ajustan después, dentro de la lectura (pestaña Taller).</div>'
             f'<div style="margin:8px 0">{filas}</div><details><summary style="cursor:pointer">Ver las sugerencias del programa</summary>{listas}</details></div>'))

**Qué sale y qué no.**
- ✅ Las listas encuentran motivos reconocibles: *zorro, baobabs, rosa* en *El principito*; *coca, caucho, selva* en Alegría; *huéspedes, violín, manzana* en Kafka; *calavera, copa, sepultura* en *Hamlet*.
- ⚠️ Sin la librería `wordfreq` (se instala sola en Colab), la primera lista se llena de palabras generales (*hombre, día, tiempo*). Por eso, en una obra nueva, conviene revisar en el Taller o usar la IA.
- ⚠️ El programa une singular y plural, pero no tiempos verbales ni sinónimos: eso lo decides tú con el asterisco (`mir*`) o agrupando palabras.
- ⚠️ Un motivo es una **palabra**: no capta una imagen que nunca se nombra (el insecto de Kafka casi no se nombra, y eso ya es un hallazgo).

## 3 · Personajes: quién aparece, cuánto, cuándo, con quién y de quién es familia
Además de la red de motivos, la lectura trae una sección de **personajes**. Sirve sobre todo para orientarse en obras con muchos personajes o con **nombres que se repiten entre generaciones**, como *Cien años de soledad*.

Cómo se arma:
1. **El reparto.** Si la obra tiene perfil, se usa el suyo. Si no, el programa propone uno: busca nombres propios (palabras que casi siempre van con mayúscula), une las formas cortas con la completa (*Rosendo* → *Rosendo Maqui*), deja fuera lo que parece un lugar (*en Rumi*, *de Umay*) y, en teatro, suma a los **hablantes**.
2. **Nombres compartidos.** Si un nombre suelto sirve para varios personajes (*Maqui*), cada mención se asigna al que se nombró más cerca en el mismo capítulo. Si una forma corta y una larga aparecen **en la misma frase**, se tratan como personas distintas (*Aureliano* ≠ *Aureliano Segundo*). Con un rango (`aureliano [15-20]`) decides tú a quién se refiere un nombre en cada parte de la obra.
3. **La red.** Dos personajes se unen cuando aparecen en el mismo **pasaje** (párrafos o parlamentos seguidos, unas 120 palabras). El tamaño es cuánto se le nombra (o habla), el color es su círculo de personajes y el **rol** (protagonista, principal, secundario, episódico, ⇄ puente) sale de su peso, sus vínculos y la parte de la obra que ocupa.
4. **El árbol de familia.** Viene del perfil o de lo que escribas en el paso 3b (`Aureliano Segundo: hijo de Arcadio y Santa Sofía de la Piedad; esposo de Fernanda del Carpio`). Sin eso, se arma con frases como «X, hijo de Y». Cada vínculo **busca en el texto una frase que lo diga** (los dos nombres y una palabra de parentesco cerca) y la muestra como cita; si no la encuentra, la línea se dibuja punteada.
5. **La ficha de cada personaje** reúne: cómo lo llama la obra («el coronel», «la bella»), su familia con citas, con quién coincide, los **motivos que lo acompañan** (base para pensar su papel simbólico) y dónde aparece. Con la IA (paso 5) puedes sumar una lectura de su papel literal y simbólico, anclada en frases numeradas.

Si el reparto o la familia tienen errores, corrígelos en el paso **3b** y vuelve a ejecutar el paso 4.

In [ ]:
#@title 3 · Arma el reparto y la red de personajes
punto_de_partida_personajes = "perfil si existe; si no, propuesta automática" #@param ["perfil si existe; si no, propuesta automática", "propuesta automática", "no incluir personajes"]
max_personajes = 25 #@param {type:"slider", min:5, max:40, step:1}
nombres_compartidos = "resolver por contexto" #@param ["resolver por contexto", "dejarlos sin asignar"]

PJ, INFO_PJ = None, None
if punto_de_partida_personajes.startswith("no"):
    print("La lectura no tendrá pestaña de personajes.")
else:
    FAMILIA = None
    if punto_de_partida_personajes.startswith("perfil") and CONF.get("personajes"):
        REPARTO, FAMILIA, origen_pj = CONF["personajes"], CONF.get("familia"), f"del perfil «{PERFIL}»"
    else:
        REPARTO, INFO_PJ = proponer_reparto(O, OBRA, CONF, max_personajes)
        origen_pj = "propuesta automática (nombres propios del texto" + (" y hablantes)" if OBRA["teatro"] else ")")
    PJ = personajes(O, OBRA, TRAMOS, REPARTO, nombres_compartidos, CONF, FAMILIA)
    display(HTML(informe_personajes(PJ, O, TRAMOS, INFO_PJ, origen_pj)))

In [ ]:
#@title 3b · (Opcional) Corrige el reparto y escribe la familia
#@markdown Copia lo que mostró el paso 3 (desplegable «El reparto usado»), pégalo entre las comillas, corrígelo y ejecuta esta celda. Después vuelve a ejecutar el paso 4. Lo que dejes vacío no cambia.
REPARTO_A_MANO = """

"""
FAMILIA_A_MANO = """

"""
# REPARTO · una línea por personaje →  Nombre que verás: forma, otra forma, forma*
#   · formas de varias palabras:   jose arcadio buendia      (la más larga gana: «José Arcadio Buendía» no cuenta como «José Arcadio»)
#   · el asterisco = cualquier final:   huesped*
#   · [a-b] limita una forma a esos capítulos (número de orden, como en el informe del paso 1):
#         Coronel Aureliano Buendía: coronel aureliano buendia, coronel, aureliano [1-14]
#         Aureliano Babilonia: aureliano babilonia, aureliano [15-20]
#   · si dos personajes comparten una forma sin rango, cada mención se asigna al que se nombró más cerca.
# FAMILIA · una línea por personaje, con los nombres tal como están en el reparto →  Nombre: vínculo de A y B; vínculo de C
#         Aureliano Segundo: hijo de Arcadio y Santa Sofía de la Piedad; esposo de Fernanda del Carpio; amante de Petra Cotes
#         Rebeca: hija adoptiva de José Arcadio Buendía y Úrsula Iguarán
#   · arman el árbol: hijo/hija, hijo adoptivo, padre/madre, esposo/esposa/marido/mujer, pareja/amante
#   · se muestran en la ficha (no en el árbol): hermano, abuelo, tío, sobrino, primo, novio, pretendiente, amigo…
#   · cada vínculo busca solo una frase del texto que lo diga; si no la encuentra, se dibuja punteado.
if REPARTO_A_MANO.strip() or FAMILIA_A_MANO.strip():
    if REPARTO_A_MANO.strip(): REPARTO = REPARTO_A_MANO
    FAMILIA = FAMILIA_A_MANO if FAMILIA_A_MANO.strip() else globals().get("FAMILIA")
    PJ = personajes(O, OBRA, TRAMOS, REPARTO, nombres_compartidos, CONF, FAMILIA)
    display(HTML(informe_personajes(PJ, O, TRAMOS, None, "corregido a mano")))
else:
    print("Sin cambios: se usa el reparto del paso 3.")

**Qué sale y qué no.**
- ✅ ***Cien años de soledad*** (con su perfil): 32 personajes en 6 generaciones. Los rangos separan a los dos José Arcadio (hijo y seminarista), al coronel de Aureliano Babilonia y a Remedios Moscote de Remedios, la bella. El árbol muestra que «José» se repite en 5 personajes de 5 generaciones y que Amaranta Úrsula y Aureliano Babilonia son pareja desde generaciones distintas. 11 de los 41 vínculos traen una frase del texto que los dice («Aureliano Buendía y Remedios Moscote se casaron…»); los demás se dicen con pronombres o a lo largo de varias frases.
- ✅ **Sin perfil**, en *Cien años* la propuesta automática ya separa «Aureliano» de «Aureliano Segundo» y «Aureliano Buendía», encuentra a Santa Sofía de la Piedad y a Fernanda, y deja fuera Macondo y Riohacha; pero no puede saber qué José Arcadio es cuál: para eso están los rangos.
- ✅ En *El mundo es ancho y ajeno* propone 25 personajes reconocibles y deja fuera los lugares (Rumi, Umay, Muncha, Yanañahui); en *Hamlet* cuenta los parlamentos; en *El principito*, los habitantes de los planetas forman su propio círculo. Con un texto inventado con nombres repetidos (Tomás, Tomás Segundo, Tomás Ávila), arma solo el árbol a partir de «Tomás Segundo, hijo de Tomás y de Remigia».
- ⚠️ El árbol **automático** casi nunca sale: los novelistas rara vez escriben «X, hijo de Y» entre personajes con nombre (en Alegría, no hay ninguna). El árbol útil es el que escribe el perfil o tú; la herramienta aporta las citas que lo respaldan.
- ⚠️ «Cómo lo llama la obra» solo encuentra fórmulas que se repiten pegadas al nombre: «el coronel» (196 veces, para Aureliano Buendía), «la bella» (Remedios), «el padre» (Nicanor), «la curandera» (Nasha Suro), «el señor» (el padre Samsa). Muchos personajes no tienen ninguna.
- ⚠️ Los **pronombres** no se cuentan, y un papel común también se cuenta cuando no es el personaje («rey» en *Hamlet* a veces es el padre muerto).
- ❌ La red y el árbol dicen **quién coincide con quién y quién es familia de quién**, no qué significa. El papel simbólico lo decides tú (o lo propone la IA como hipótesis) a partir de las citas y de los motivos que acompañan al personaje.

## 4 · La lectura en red
Esta celda genera la página interactiva, la muestra aquí abajo y la **descarga** como archivo `.html`, que se abre en cualquier navegador sin Colab ni internet. Ese archivo es el producto: puedes volver a él, mostrarlo en clase o enviarlo a tu asesor.

| Pestaña | Para qué |
|---|---|
| **Hallazgos** | Puntos de entrada calculados: el motivo **puente**, la **pareja** que no se separa, lo que **nace tarde** o **se apaga**, lo que ocurre **en ráfaga**, los motivos que **se evitan** |
| **Red** | Los motivos y sus uniones. Con ▶ recorres la obra tramo a tramo; los motivos quedan quietos y cambian sus uniones. Clic en un motivo o en una línea → frases con página |
| **Personajes** | La red de personajes y el **árbol de familia** (si hiciste el paso 3), recorribles por tramos. Clic en un vínculo del árbol → la frase que lo dice. Una ficha por personaje: rol, cómo lo llama la obra, familia, con quién coincide, motivos que lo acompañan y citas. Y «Quién está cuándo», una línea de tiempo con todas sus apariciones |
| **Motivos** | Una ficha por motivo: dónde aparece a lo largo de la obra, con qué va y dos frases |
| **Taller** | Cambias, unes o sumas motivos; al aplicar, todo se recalcula |
| **Cuaderno** | Tus notas y las citas que guardaste con **＋ citar**, descargables en Markdown |

**Cómo leerla.** Los **colores** son familias de motivos que suelen ir juntos, calculadas por el programa. Las uniones se dicen con palabras (*cercanos, unidos, muy unidos, inseparables*) según cuánto más de lo esperado por azar coinciden. Una línea **punteada** es una unión **frágil**: desaparece si se mide por párrafo o por ventana de palabras en vez de por oración. El **Modo investigación** muestra las cifras, la unidad de análisis y una tabla descargable.

In [ ]:
#@title 4 · Genera la lectura en red
descargar_lectura = True  #@param {type:"boolean"}
descargar_materiales = False  #@param {type:"boolean"}

DATOS = datos_lectura(OBRA, O, TRAMOS, MOTIVOS, LISTAS, TITULO, AUTOR)
if globals().get("PJ"): DATOS["personajes"] = datos_personajes(PJ, O)
FRAGMENTO = generar_lectura(DATOS, PLANTILLA)
base = re.sub(r"\W+", "_", sin_tildes(TITULO.lower()).replace("ñ", "n")).strip("_")
ARCHIVO = f"lectura_{base}.html"
with open(ARCHIVO, "w", encoding="utf-8") as f: f.write(documento(FRAGMENTO, TITULO))
print(f"Lectura guardada en {ARCHIVO} ({os.path.getsize(ARCHIVO) / 1e6:.1f} MB).")
display(HTML(FRAGMENTO))
if descargar_materiales:
    with open(f"texto_limpio_{base}.txt", "w", encoding="utf-8") as f:
        for s in OBRA["segmentos"]:
            f.write(f"\n## {s['etiqueta']} {s['titulo']}\n\n")
            f.write("\n".join(f"[{pagina}] {t}" for pagina, t in [(o["pag"], o["texto"]) for o in O if o["seg"] == s["num"]]) + "\n")
    CAND.assign(formas=CAND.formas.map(", ".join)).to_csv(f"candidatos_{base}.csv", encoding="utf-8-sig")
    with open(f"motivos_{base}.json", "w", encoding="utf-8") as f:
        json.dump({"obra": TITULO, "motivos": DATOS["motivos"]}, f, ensure_ascii=False, indent=1)
    if globals().get("PJ"):
        with open(f"reparto_{base}.txt", "w", encoding="utf-8") as f:
            f.write("# REPARTO\n" + PJ["texto"] + "\n\n# FAMILIA\n" + PJ.get("familia_texto", "") + "\n")
        pd.DataFrame([{"personaje": p["n"], "rol": p["rol"], "puente": p["puente"], "menciones": p["menc"], "parlamentos": p["parl"],
                       "pasajes": p["pasajes"], "grado": p["grado"], "intermediacion": p["betw"], "formas": "; ".join(p["formas"])}
                      for p in PJ["lista"]]).to_csv(f"personajes_{base}.csv", index=False, encoding="utf-8-sig")
        L_ = PJ["lista"]
        pd.DataFrame([{"a": L_[a]["n"], "b": L_[b]["n"], "pasajes_juntos": c, "fuerza": round(f_, 4), "lift": round(l_, 2)} for a, b, c, f_, l_ in PJ["E"]]
                     ).to_csv(f"relaciones_personajes_{base}.csv", index=False, encoding="utf-8-sig")
    if IA_REGISTRO:
        with open(f"lecturas_ia_{base}.json", "w", encoding="utf-8") as f: json.dump(IA_REGISTRO, f, ensure_ascii=False, indent=1, default=str)
    zipname = f"materiales_{base}.zip"
    with zipfile.ZipFile(zipname, "w") as z:
        for fn in [ARCHIVO] + [x for x in os.listdir(".") if x.startswith(("texto_limpio_", "candidatos_", "motivos_", "lecturas_ia_", "reparto_", "personajes_", "relaciones_personajes_")) and base in x]:
            z.write(fn)
if EN_COLAB:
    if descargar_lectura: files.download(ARCHIVO)
    if descargar_materiales: files.download(zipname)

**Qué sale y qué no.**
- ✅ Hallazgos que un lector reconoce: en *El principito*, «corazón» no aparece hasta el capítulo XXI y el 95 % de «domesticar» está en «El zorro»; en *Hamlet*, «venganza» e «infierno» van juntos; en Alegría, «comunidad» es el puente de la red; en Kafka, «música» y «huéspedes» son inseparables y «cuerpo» y «madre» casi nunca coinciden.
- ✅ Todo se recalcula en el navegador: el Taller no obliga a volver a Colab.
- ⚠️ Dentro de Colab, algunas funciones (descargar notas, guardar el Cuaderno) pueden estar bloqueadas: úsalas en el **archivo descargado**.
- ⚠️ Las notas se guardan en el navegador donde abres el archivo; descárgalas para no perderlas.
- ⚠️ La página contiene las frases de la obra para poder recalcular y citar. Úsala para estudio e investigación; si la obra tiene derechos de autor vigentes (como la de Alegría), no la publiques abiertamente.
- ❌ La red mide **vecindad**, no sentido: dos motivos «unidos» pueden estar en la misma frase por casualidad o por contraste. La lectura de las frases decide.

## 5 · (Opcional) Lectura asistida con IA
Con tu clave de OpenAI, el modelo hace dos tareas, **siempre sobre frases numeradas de la obra**:
- **Leer una conexión**: escribe dos motivos de tu diccionario. El modelo propone qué vínculo construye la obra entre ellos, si cambia, qué lo matiza y qué leer, **citando las frases por número**. Lo que diga por conocimiento previo de la obra aparece aparte, como «no verificado».
- **Rol de un personaje**: escribe un personaje del reparto. El modelo recibe sus fórmulas («el coronel»), su familia, los motivos de sus pasajes y frases numeradas donde aparece, y propone su **papel en la trama** y su **papel simbólico**, citando por número. La propuesta se agrega a su ficha en la lectura (pestaña Personajes), marcada como hipótesis.
- **Medir la precisión de un motivo**: clasifica 20 apariciones al azar (¿la palabra se usa en ese sentido?). Así detectas, por ejemplo, *rosa* el color frente a *rosa* la flor.

**Tu clave:** en el panel 🔑 **Secretos** (barra izquierda de Colab), crea `OPENAI_API_KEY`, pega la clave y activa el acceso para este cuaderno. Nunca la escribas en una celda.

In [ ]:
#@title 5 · (Opcional) Lectura asistida con IA
tarea   = "leer una conexión" #@param ["leer una conexión", "medir la precisión de un motivo", "rol de un personaje (literal y simbólico)"]
motivo_a = ""   #@param {type:"string"}
motivo_b = ""   #@param {type:"string"}
sentido  = "como motivo de la obra, no un uso casual o con otro significado" #@param {type:"string"}
personaje = ""  #@param {type:"string"}
modelo_ia = "gpt-6-luna" #@param {type:"string"}

if tarea.startswith("rol"):
    if not globals().get("PJ"): print("Primero arma el reparto (paso 3).")
    elif CLIENTE or conectar_ia(modelo_ia):
        if ia_rol_personaje(PJ, O, personaje, MOTIVOS, TITULO) is not None and "ARCHIVO" in globals():
            DATOS["personajes"] = datos_personajes(PJ, O); FRAGMENTO = generar_lectura(DATOS, PLANTILLA)
            with open(ARCHIVO, "w", encoding="utf-8") as f: f.write(documento(FRAGMENTO, TITULO))
            print(f"La propuesta se agregó a la ficha del personaje en {ARCHIVO} (pestaña Personajes).")
            if EN_COLAB: files.download(ARCHIVO)
else:
    print("Motivos de tu diccionario:", ", ".join(m["nombre"] for m in MOTIVOS))
    if CLIENTE or conectar_ia(modelo_ia):
        if tarea == "leer una conexión":
            ia_leer_conexion(O, MOTIVOS, motivo_a, motivo_b, TITULO)
        else:
            ia_precision(O, MOTIVOS, motivo_a, sentido, TITULO)

**Qué sale y qué no.**
- ✅ Las respuestas quedan ancladas en frases con lugar y página, y se avisa si el modelo cita una frase que no existe.
- ⚠️ El modelo económico por defecto (`gpt-6-luna`) puede cambiar de nombre; si falla, escribe otro en `modelo_ia`.
- ⚠️ Se envían a OpenAI fragmentos cortos de la obra (no el libro completo), y cada consulta tiene un costo pequeño.
- ❌ La IA no sustituye tu interpretación: sus lecturas son hipótesis, como las de la red.

---
### Lo que mide la herramienta y lo que interpretas tú
La herramienta **mide** dónde aparecen las palabras y cuáles coinciden. **Tú interpretas** qué significa: símbolo, contraste, transformación o simple casualidad. Antes de sostener un hallazgo en un trabajo: (1) lee las frases; (2) mira si la unión es sólida o frágil; (3) cambia un motivo en el Taller y comprueba si el hallazgo resiste.